# Detección y clasificación de tumores cerebrales en RM con CNN (trazabilidad real por paciente)

**TFG · Joaquín González Rodríguez · Tutor: Manuel Carranza García · Cotutor: José Cristóbal Riquelme Santos**

**Tarea principal de esta versión: clasificación del TIPO de tumor (glioma / meningioma / pituitario) a nivel de paciente, usando únicamente pacientes de Cheng et al. (2017).**
La tarea binaria tumor/sano de versiones anteriores se conserva como **experimento previo y diagnóstico**: fue la que reveló que, tal como estaba planteada, la clase y la fuente de los datos estaban confundidas (todo el `tumor` viene de una colección, todo el `notumor` de otra). Ese hallazgo motivó el cambio de eje hacia la clasificación de tipo de tumor, que se hace **enteramente dentro de una única fuente** (Cheng) y por tanto no puede beneficiarse de ese atajo.

## Qué resuelve esta versión frente a la anterior

1. **Sesgo de procedencia en la tarea binaria** (documentado, no eliminado — no existe hoy una colección pública con casos sanos y con tumor del mismo escáner/protocolo). Ver sección de auditoría de sesgo.
2. **Reparto por paciente, no por imagen**, con trazabilidad real (`patient_id` de la fuente primaria) para ambas tareas.
3. **Un único framework**, controlado por una variable de configuración explícita (`TASK`), en vez de dos pipelines independientes para binario y multiclase.
4. **El split multiclase (Cheng, 3 clases) queda CONGELADO**: es exactamente el que produjo el 79,7 % de balanced accuracy patient-level en la ejecución de referencia (SLURM job 70436). Este cuaderno lo reconstruye, verifica su huella y se DETIENE si no coincide — nunca genera un split nuevo en su lugar. Ver `baseline_multiclase_cheng.md` para el detalle completo de esa verificación.

## Qué se puede y qué no se puede afirmar con estos datos

Con asertos que se ejecutan de verdad, este cuaderno **sí** sostiene:

- Cada imagen tiene un `patient_id` real, trazable hasta el fichero de origen.
- Ningún paciente aparece en más de un subconjunto (train / val / test), en NINGUNA de las dos tareas.
- El split multiclase de Cheng es exactamente el histórico: verificado por huella criptográfica, no supuesto.
- El conjunto de test se usa como mucho una vez por modelo final, con todo lo demás (arquitectura, agregación, umbral cuando aplica) congelado de antemano — con una única excepción documentada explícitamente: la comprobación de regresión del baseline `cnn_scratch` multiclase (no existe checkpoint del modelo original, así que reproducirlo exige reentrenar bajo el split congelado y comparar contra el resultado histórico).
- Las métricas principales son a nivel de **paciente**, con balanced accuracy como métrica primaria en ambas tareas.

Este cuaderno **no** sostiene:

- Que la tarea binaria esté libre de sesgo de procedencia (sección de auditoría de sesgo, sigue siendo la limitación estructural principal de esa tarea).
- Que exista validación clínica de ningún tipo: es un clasificador experimental de investigación.
- Que VGG16/ResNet50/MobileNetV2/EfficientNetB0 tengan un resultado histórico de referencia en la tarea multiclase — no lo tienen; esta es la primera vez que se evalúan en ella, y así se etiquetan explícitamente en la tabla comparativa ("new multiclass benchmark", frente a "baseline regression check" de `cnn_scratch`).

## Nota sobre las cifras

Ningún porcentaje de este cuaderno está escrito a mano; todas las cifras de texto y tablas
se generan a partir de las variables de la ejecución en curso. La única excepción,
señalada explícitamente donde aparece, es la cifra histórica `BALANCED_ACCURACY_HISTORICA
= 0.7965811965811967`, que es una constante de referencia documentada (viene de
`control_3clases.json` de la ejecución SLURM 70436), no un resultado de esta ejecución.


## Índice

**Parte 0 — Entorno y reproducibilidad**
- 0. Rutas y entorno (Colab / HPC, sin dependencias de Drive ni de `/content`)
- 1. Reproducibilidad, modos de trabajo y configuración del dataset
  - **1.5 Modo de ejecución (`RUN_MODE`, `RUN_BINARY_SECONDARY`)**

**Parte I — Datos y trazabilidad**
- 2. Construcción del dataset con trazabilidad real por paciente
  - 2.1 Adquisición de las fuentes primarias
  - 2.2 Manifiesto de trazabilidad
  - 2.3 Duplicados exactos, casi-duplicados y conflictos de etiqueta
  - 2.3b Ejemplos visualizables de los clústeres detectados
  - 2.4 Reparto train / val / test a nivel de paciente (dataset completo)
  - **2.5 Verificación y congelación del split MULTICLASE (Cheng, 3 clases)**
  - 2.6 Auditoría de sesgo de procedencia (tarea binaria)
  - **2.7 Auditoría de sesgo — control multiclase dentro de Cheng**
  - 2.8 Informe automático de auditoría + *sanity check*
  - 2.9 Manifiesto exportable y huella digital del dataset y del reparto

**Parte II — Framework compartido (binario / multiclase por configuración)**
- 3. Configuración explícita de tarea (`TASK_CONFIG`)
- 4. Pipeline único de datos (`tf.data`)
- 5. Métricas y agregación slice → paciente (generalizadas por `task_config`)
- 6. Arquitecturas y entrenamiento (generalizadas por `task_config`)

**Parte III — Tarea MULTICLASE (eje principal)**
- 7. Baseline `cnn_scratch` — regresión del resultado histórico (79,7 %)
- 8. Benchmark de transfer learning multiclase (VGG16 / ResNet50 / MobileNetV2 / EfficientNetB0) — nuevo, solo validación
- 9. Comparación multiclase, gráficas y resumen reproducible

**Parte IV — Tarea BINARIA (experimento previo / diagnóstico, secundaria)**
- 10. Entrenamiento y comparación de los 5 modelos (protocolo histórico, sin cambios metodológicos)
- 11. Evaluación final sobre test binario (una sola vez)
- 12. Interpretabilidad (Grad-CAM del modelo binario)
- 13. Inferencia (imagen suelta / volumen NIfTI)
- 14. Conclusiones

**Apéndices**
- A. (integrado en el cuerpo principal — puntero)
- B. Modo *legacy* / proxy — *EXPLORATORY, NOT VALID FOR FINAL PATIENT-LEVEL RESULTS*
- C. Qué haría falta para incorporar las fuentes descartadas

---

### Cómo ejecutar este cuaderno

1. Antes de la primera ejecución, copia `split_multiclase_cheng.json` (y opcionalmente
   `split_multiclase_cheng.csv`) en `PROJECT_ROOT/frozen/`. El cuaderno se detiene con un
   error explícito si no los encuentra o si no coinciden con el manifiesto actual.
2. Fija `RUN_MODE` en la Sección 1.5 según lo que quieras ejecutar esta vez:
   `"baseline_only"` (por defecto: solo el baseline `cnn_scratch`),
   `"multiclass_benchmark"` (además, los cuatro backbones nuevos) o `"full"`
   (todo, incluida la tarea binaria si además `RUN_BINARY_SECONDARY = True`).
3. `Entorno de ejecución` → `Reiniciar` → `Ejecutar todas` (Colab), o vía `jupyter nbconvert
   --execute` en el cluster (ver el script SLURM adjunto). Las secciones que `RUN_MODE`
   no activa se ejecutan igualmente pero no hacen nada: imprimen un aviso de que han sido
   omitidas y siguen con la siguiente celda.
4. La sección 2.1 descarga los datos originales (~880 MB de Cheng y ~2 GB de IXI en
   streaming). No hay que subir ningún ZIP ni preparar carpetas a mano.

Si algo crítico falla — falta un `patient_id`, un paciente aparece en dos subconjuntos, el
split multiclase congelado no coincide con el manifiesto actual, o la regresión del
baseline se sale del margen aceptado — el cuaderno se detiene con un error explícito
(`RuntimeError`) antes de seguir entrenando nada.


In [ ]:
# ==============================================================================
# SECCIÓN 0 — ENTORNO Y RUTAS (única fuente de verdad para todas las rutas)
# ==============================================================================
# Pensado para funcionar igual en Google Colab (desarrollo) y en el cluster HPC
# de la Universidad de Sevilla vía SLURM (ejecución de referencia): no depende
# de Google Drive, de rutas /content/... hardcodeadas en otras celdas, ni de
# ningún widget interactivo de Colab. Todas las demás celdas leen rutas de las
# variables definidas aquí; ninguna vuelve a construir un Path a mano.
#
# Resolución de PROJECT_ROOT, en este orden:
#   1. Variable de entorno TFG_PROJECT_ROOT, si está definida (así es como el
#      script de SLURM fija la ruta sin tocar el notebook).
#   2. /content/tfg_tumores, si /content existe (Google Colab).
#   3. <directorio actual>/tfg_run, en cualquier otro caso (HPC, portátil...).
import os
from pathlib import Path

PROJECT_ROOT = Path(
    os.environ.get("TFG_PROJECT_ROOT")
    or (str(Path("/content") / "tfg_tumores") if Path("/content").exists()
        else str(Path.cwd() / "tfg_run"))
)

DATA_DIR    = PROJECT_ROOT / "data"       # descargas + PNG exportados
RESULTS_DIR = PROJECT_ROOT / "results"    # todo lo que genera el cuaderno
MODELS_DIR  = RESULTS_DIR / "models"
PLOTS_DIR   = RESULTS_DIR / "plots"
LOGS_DIR    = RESULTS_DIR / "logs"
FROZEN_DIR  = PROJECT_ROOT / "frozen"     # split(s) congelado(s), fuente de verdad externa

# Alias heredados de versiones anteriores del cuaderno: se mantienen para no
# tener que reescribir cada celda que ya los usa (ROOT, RAW_DIR, IMG_DIR,
# ART_DIR, FIG_DIR eran los nombres previos).
ROOT = PROJECT_ROOT
RAW_DIR = DATA_DIR / "raw"
IMG_DIR = DATA_DIR / "images"
ART_DIR = RESULTS_DIR / "artifacts"
FIG_DIR = PLOTS_DIR

for _d in (DATA_DIR, RAW_DIR, IMG_DIR, RESULTS_DIR, ART_DIR, MODELS_DIR,
           PLOTS_DIR, LOGS_DIR, FROZEN_DIR):
    _d.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = ART_DIR / "manifest_final.csv"
TEMPLATE_PATH = ART_DIR / "manifest_template.csv"
AUDIT_JSON    = ART_DIR / "dataset_audit.json"

# Split multiclase congelado (ver Sección 2.5). Debe copiarse aquí ANTES de
# ejecutar el cuaderno; nunca se genera automáticamente.
SPLIT_MULTICLASE_JSON = FROZEN_DIR / "split_multiclase_cheng.json"
SPLIT_MULTICLASE_CSV  = FROZEN_DIR / "split_multiclase_cheng.csv"


def guardar_figura(fig, nombre, formatos=("png", "pdf")):
    """Guarda la figura en PLOTS_DIR, en buena resolución y en los formatos
    indicados (por defecto PNG + PDF, aptos para incluir en la memoria)."""
    for ext in formatos:
        fig.savefig(PLOTS_DIR / f"{nombre}.{ext}", dpi=200, bbox_inches="tight")


print("PROJECT_ROOT:", PROJECT_ROOT)
print("  DATA_DIR   :", DATA_DIR)
print("  RESULTS_DIR:", RESULTS_DIR)
print("  MODELS_DIR :", MODELS_DIR)
print("  PLOTS_DIR  :", PLOTS_DIR)
print("  LOGS_DIR   :", LOGS_DIR)
print("  FROZEN_DIR :", FROZEN_DIR, " (split_multiclase_cheng.json/.csv deben copiarse aquí)")


In [ ]:
# ==============================================================================
# SECCIÓN 1 -- REPRODUCIBILIDAD, MODOS DE TRABAJO Y CONFIGURACIÓN DEL DATASET
# ==============================================================================
# Las rutas ya se han fijado en la Sección 0; esta celda no vuelve a definir
# ningún directorio.
# todo lo configurable vive en esta celda; no hay constantes sueltas repartidas por ahí
#
# los dos interruptores que de verdad importan:
#   STRICT_PATIENT_MODE = True  -> solo admite imágenes con patient_id real,
#                                  verificable contra la fuente. si falta alguno,
#                                  para antes de entrenar -- es el único modo válido
#                                  para los resultados finales.
#   LEGACY_PROXY_MODE   = False -> modo exploratorio del Apéndice B, con el ZIP
#                                  antiguo de JPEG y un proxy_group_id sacado del
#                                  nombre de fichero -- no da resultados patient-level.

import os, sys, json, random, hashlib, subprocess, shutil, time, textwrap
from pathlib import Path

# ---------------------------------------------------------------- MODOS -----
STRICT_PATIENT_MODE = True      # obligatorio para los resultados definitivos
LEGACY_PROXY_MODE   = False     # solo Apéndice B (diagnóstico / desarrollo)

# Modo de prueba: genera datos sintéticos con la MISMA estructura que las fuentes
# reales (ficheros .mat estilo cjdata y volúmenes .nii.gz estilo IXI) para poder
# validar el flujo completo sin descargar ~3 GB. NUNCA debe estar activo en la
# ejecución que se entrega.
USAR_DATOS_SINTETICOS = False

# --------------------------------------------------------------- SEMILLA ----
SEED = 42
DETERMINISTA = True             # fija las semillas de Python, NumPy, TF y Keras
# El determinismo TOTAL de TensorFlow (enable_op_determinism) obliga a que cada
# operación dé el mismo resultado bit a bit. En GPU, algunas capas de aumento y
# algunos kernels de cuDNN no lo soportan y lanzan UnimplementedError EN MITAD del
# entrenamiento. Se deja desactivado por defecto: las semillas y las huellas del
# dataset ya hacen la ejecución auditable. Actívalo si necesitas reproducibilidad
# bit a bit y estás dispuesto a ejecutar en CPU.
DETERMINISMO_ESTRICTO = False

# ------------------------------------------------------------ ADQUISICIÓN ---
# Cheng et al. (2017) — figshare 10.6084/m9.figshare.1512427 (CC BY 4.0).
# 3.064 cortes T1 con contraste de 233 pacientes. Cada .mat contiene cjdata con
# los campos: label (1=meningioma, 2=glioma, 3=pituitario), PID, image,
# tumorBorder y tumorMask. El PID es el identificador REAL de paciente.
FIGSHARE_ARTICLE_ID = 1512427
FIGSHARE_ZIPS = {   # nombre -> (file_id, md5 publicado por figshare)
    "brainTumorDataPublic_1-766.zip":    (3381290, "74b949ad33f042e6e103523091cd1428"),
    "brainTumorDataPublic_767-1532.zip": (3381296, "7e8a875500d2c8a346f270538e29890e"),
    "brainTumorDataPublic_1533-2298.zip":(3381293, "8227bf6080cb71f15a88be8d25c79ae7"),
    "brainTumorDataPublic_2299-3064.zip":(3381302, "b378a80d6174e5317d59eb28430c6652"),
}
FIGSHARE_DOWNLOAD = "https://ndownloader.figshare.com/files/{file_id}"

# IXI Dataset (CC BY-SA 3.0), Brain Development / Imperial College London.
# Solo se ofrece como un .tar grande, así que se lee EN STREAMING y se corta en
# cuanto se han obtenido N_SUJETOS_SANOS sujetos: no se descargan los 4,5 GB.
IXI_T1_TAR  = "https://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI/IXI-T1.tar"
IXI_XLS     = "https://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI/IXI.xls"

# Fallback reproducible cuando el servidor de Imperial bloquea IPs de Colab.
# Zenodo 7047668 redistribuye 507 volúmenes T1 procedentes de IXI y publica
# checksum MD5 individual para cada .nii.gz. La fuente científica sigue siendo IXI.
IXI_ZENODO_RECORD_ID = 7047668
IXI_ZENODO_API = f"https://zenodo.org/api/records/{IXI_ZENODO_RECORD_ID}"
IXI_ZENODO_URL = f"https://zenodo.org/records/{IXI_ZENODO_RECORD_ID}"

# Cuántos sujetos sanos y cuántos cortes por sujeto. El objetivo es igualar a
# Cheng TANTO en nº de pacientes (233) COMO en nº de cortes por paciente (~13),
# para que el desequilibrio no se cuele por la puerta de atrás (ver sección 1.7).
N_SUJETOS_SANOS   = 233
PLANOS_IXI        = ("axial", "coronal", "sagital")
CORTES_POR_PLANO  = 4     # 3 planos x 4 = 12-13 cortes por sujeto sano

# -------------------------------------------------------------- IMÁGENES ----
# Los cortes de AMBAS fuentes se exportan con el MISMO procedimiento:
#   percentil 1-99 -> [0,255] -> PNG de 8 bits en escala de grises a PNG_SIZE.
# Esto homogeneiza resolución y compresión, que son dos de las vías por las que
# se cuela el sesgo de procedencia. No elimina el sesgo, pero cierra dos puertas.
PNG_SIZE = (256, 256)

IMG_SIZE    = (224, 224)   # entrada de los modelos
CANALES_CNN = 1            # CNN propia en escala de grises (las RM lo son)
BATCH_SIZE  = 32

# ------------------------------------------------------------- ENTRENAMIENTO -
EPOCHS_CNN       = 40
EPOCHS_BACKBONE  = 12
LR               = 1e-4
PACIENCIA        = 8

# Selección del modelo ganador:
#   "groupcv" -> validación cruzada agrupada por paciente sobre train+val (robusto)
#   "holdout" -> un único conjunto de validación (rápido)
# En AMBOS casos la selección usa EXCLUSIVAMENTE validación: test queda bloqueado.
SELECCION_MODELO = "groupcv"   # "holdout" para una ejecución rápida
N_FOLDS_CV       = 3           # con 5 modelos, groupcv entrena 5 x N_FOLDS_CV veces

# Reparto objetivo (sobre PACIENTES, no sobre imágenes)
FRAC_TEST = 0.15
FRAC_VAL  = 0.15

# ------------------------------------------------------------- DUPLICADOS ---
PHASH_SIZE        = 16    # hash perceptual de 16x16 (256 bits): más fino que 8x8
HAMMING_NEAR      = 12    # umbral de "casi duplicado" sobre 256 bits
SSIM_CONFIRMACION = 0.92  # SSIM por encima del cual se confirma el casi-duplicado

# ==============================================================================
# Semillas y determinismo
# ==============================================================================
os.environ["PYTHONHASHSEED"] = str(SEED)
# IMPORTANTE: las variables de determinismo de operaciones de TensorFlow solo se
# activan cuando se pide determinismo ESTRICTO. Con DETERMINISMO_ESTRICTO=False
# mantenemos semillas reproducibles sin forzar kernels deterministas en GPU.
if DETERMINISMO_ESTRICTO:
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
else:
    os.environ.pop("TF_DETERMINISTIC_OPS", None)
    os.environ.pop("TF_CUDNN_DETERMINISTIC", None)

random.seed(SEED)
try:
    import numpy as np
    np.random.seed(SEED)
except ImportError:
    np = None


def _pip(*paquetes):
    """Instala en silencio solo lo que falte."""
    faltan = []
    alias = {"scikit-image": "skimage", "pillow": "PIL", "opencv-python-headless": "cv2"}
    for p in paquetes:
        mod = alias.get(p, p.replace("-", "_"))
        try:
            __import__(mod)
        except ImportError:
            faltan.append(p)
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=False)


_pip("h5py", "nibabel", "imagehash", "scikit-image", "pandas", "scikit-learn", "matplotlib")

import numpy as np
import pandas as pd

np.random.seed(SEED)

print("Configuración cargada.")
print(f"  STRICT_PATIENT_MODE   = {STRICT_PATIENT_MODE}")
print(f"  LEGACY_PROXY_MODE     = {LEGACY_PROXY_MODE}")
print(f"  USAR_DATOS_SINTETICOS = {USAR_DATOS_SINTETICOS}")
print(f"  SEED                  = {SEED}  (determinista={DETERMINISTA})")
print(f"  Entrada de los modelos= {IMG_SIZE} x {CANALES_CNN} canal(es)")
print(f"  Selección del ganador = {SELECCION_MODELO}")
print(f"  Directorio de trabajo = {PROJECT_ROOT}")
if USAR_DATOS_SINTETICOS:
    print("\n  *** ATENCIÓN: datos SINTÉTICOS. Los resultados NO son válidos. ***")

### 1.5 Modo de ejecución (`RUN_MODE`)

Controla explícitamente cuánto de este cuaderno se ejecuta, para poder lanzarlo por
etapas en el cluster sin tener que comentar/descomentar celdas a mano:

- **`"baseline_only"`** (valor por defecto): verifica dataset y split, entrena/reproduce
  el baseline `cnn_scratch` (Sección 7) y **termina ahí** — no entrena los cuatro
  backbones nuevos ni ejecuta la tarea binaria.
- **`"multiclass_benchmark"`**: además de lo anterior, reutiliza el baseline ya guardado
  (si es válido para el split/manifiesto actuales) y entrena/evalúa VGG16, ResNet50,
  MobileNetV2 y EfficientNetB0 (Sección 8, solo train/validation), y genera la tabla y
  las gráficas comparativas (Sección 9). Sigue sin ejecutar la tarea binaria.
- **`"full"`**: lo anterior, y además la tarea binaria (Secciones 10-14) se ejecuta **si y
  solo si** `RUN_BINARY_SECONDARY = True`. `full` por sí solo no basta para reactivar la
  tarea binaria: es una comprobación adicional, deliberada, para no reentrenar/reevaluar
  el experimento previo por descuido.

Ninguna de estas tres opciones afecta a las Secciones 0-6 (rutas, configuración,
construcción del dataset, verificación y congelación del split, framework compartido):
esas se ejecutan siempre, en los tres modos, porque son la base sobre la que se apoya
cualquiera de las fases posteriores.

In [ ]:
# ==============================================================================
# 1.5  MODO DE EJECUCIÓN -- decide qué fases de este cuaderno se ejecutan
# ==============================================================================
RUN_MODE = "baseline_only"          # "baseline_only" | "multiclass_benchmark" | "full"
RUN_BINARY_SECONDARY = False        # solo importa si RUN_MODE == "full"

_RUN_MODES_VALIDOS = ("baseline_only", "multiclass_benchmark", "full")
assert RUN_MODE in _RUN_MODES_VALIDOS, (
    f"RUN_MODE={RUN_MODE!r} no reconocido; debe ser uno de {_RUN_MODES_VALIDOS}")

# Secciones 0-6 (rutas, split congelado, framework): SIEMPRE se ejecutan.
# Sección 7 (baseline cnn_scratch, regresión/reutilización): SIEMPRE se ejecuta.
_RUN_BENCHMARK_MC = RUN_MODE in ("multiclass_benchmark", "full")   # Secciones 8-9
_RUN_BINARIA = (RUN_MODE == "full") and RUN_BINARY_SECONDARY        # Secciones 10-14

print(f"RUN_MODE              = {RUN_MODE!r}")
print(f"RUN_BINARY_SECONDARY  = {RUN_BINARY_SECONDARY}")
print("--------------------------------------------------------------")
print("Secciones 0-6  (dataset, split congelado, framework)  : SIEMPRE")
print("Sección  7     (baseline cnn_scratch, regresión)      : SIEMPRE")
print(f"Secciones 8-9  (benchmark multiclase + comparación)   : {'SÍ' if _RUN_BENCHMARK_MC else 'NO (omitidas)'}")
print(f"Secciones 10-14 (tarea binaria, experimento previo)    : {'SÍ' if _RUN_BINARIA else 'NO (omitidas)'}")


## 2. Construcción del dataset con trazabilidad real por paciente

### El error conceptual que se corrige

Las resonancias no llegan de una en una: un paciente produce un volumen y un volumen
produce muchos cortes que se parecen mucho entre sí. Si esos cortes se reparten como si
fueran muestras independientes, cortes casi idénticos del mismo paciente caen a la vez en
entrenamiento y en test, y el modelo obtiene una métrica alta reconociendo pacientes que
ya ha visto. Es fuga de datos por paciente.

La versión anterior de este trabajo intentaba evitarlo agrupando por un `group_id`
deducido del **nombre del fichero**. El Apéndice B demuestra por qué eso no funcionaba:
`G_588`, `M_588`, `P_588` y `N_588` son cuatro imágenes distintas —dos de ellas de clases
opuestas— y recibían el mismo identificador. Agrupar por ahí no es agrupar por paciente.

### La solución adoptada

Volver a las fuentes primarias y usar el identificador que ellas mismas publican:

| Clase | Fuente | Identificador | Cómo se obtiene |
|---|---|---|---|
| `tumor` | Cheng et al. 2017 (figshare, CC BY 4.0) | `cjdata.PID` | se lee de cada `.mat` (HDF5) |
| `notumor` | IXI (CC BY-SA 3.0) | ID de sujeto | del nombre del volumen NIfTI |

Con eso, cada imagen del manifiesto responde a: de qué fichero original salió, de qué
volumen, de qué paciente y de qué colección. La frase *"este paciente está en test, estos
son todos sus cortes, esta es su fuente y ninguno de sus datos aparece durante el
entrenamiento"* pasa a ser comprobable, y de hecho se comprueba con asertos en 1.5.

### Homogeneización deliberada del preprocesado

Los cortes de ambas fuentes se exportan con **exactamente el mismo procedimiento**:
recorte a la caja del cerebro, normalización por percentiles 1–99 y PNG de 8 bits al mismo
tamaño. No es cosmético: resolución nativa y artefactos de compresión JPEG son dos vías
conocidas por las que la procedencia se cuela en el modelo. Cerrarlas no elimina el sesgo
—siguen siendo escáneres y protocolos distintos— pero elimina dos de sus atajos más
groseros, y lo que quede se mide en 1.6.

### Qué NO se hace

- **No se inventa ningún `patient_id`.** Lo que no viene de la fuente se marca como
  desconocido y bloquea la ejecución en modo estricto.
- **No se borran imágenes por parecerse.** Dos cortes vecinos del mismo cerebro se parecen
  mucho y son información clínica legítima. Se agrupan, no se eliminan.
- **No se afirma que no hay fuga.** Se ejecutan comprobaciones que fallan si la hay.

In [ ]:
# 1.2 -- de dónde salen los datos y cómo compruebo que la descarga está bien
# El cuaderno NO pide subir ningún ZIP preparado a mano: descarga él mismo los
# datos originales, con URL explícita y verificación de integridad. Eso es lo que
# hace que el dataset sea auditable por un tercero.
#
# (A) Cheng et al. 2017 — figshare. 4 ZIP (~880 MB) verificados por MD5.
# Se extraen los 3.064 ficheros .mat, cada uno con su cjdata.PID real.
#
# (B) IXI — se intenta primero el TAR primario en streaming. Si Imperial bloquea
# la IP de Colab (HTTP 403), se usa el registro Zenodo 7047668, que contiene
# 507 T1 procedentes de IXI con nombre original y MD5 por fichero.
#
# Ambas descargas son idempotentes: si el fichero ya está y su MD5 coincide, no
# se vuelve a bajar.
#
# --- ROBUSTEZ FRENTE A FALLOS DE RED (añadido tras observar HTTP 403 en Imperial
# y HTTP 504 en Zenodo durante la ejecución real) --------------------------------
# Un único fallo transitorio (timeout, 502/503/504, conexión cortada a mitad de
# fichero) ya NO aborta el cuaderno entero ni obliga a repetir a mano lo que ya
# se había descargado:
#   - `_con_reintentos` reintenta con backoff exponencial + jitter cualquier
#     llamada de red, e informa de cada reintento en el log.
#   - La descarga de cada fichero individual de Zenodo es idempotente por MD5 y,
#     si falla incluso tras los reintentos, se DESCARTA ese sujeto concreto y se
#     continúa con el siguiente candidato de la lista en vez de abortar el lote
#     completo (antes, un solo sujeto fallido tiraba abajo los ~200 ya en curso).
#   - Se añade una pequeña pausa entre peticiones a Zenodo para no golpear su API
#     más rápido de lo razonable (posible causa del 504 observado).
#   - Tras una ejecución real en la que Zenodo devolvió 504/503 en la práctica
#     totalidad de las peticiones secuenciales durante varios minutos seguidos,
#     las descargas individuales de Zenodo se lanzan en paralelo (varias
#     peticiones HTTP a la vez, cada una con sus propios reintentos): la
#     latencia observada es del servidor, no del cliente, así que solapar
#     peticiones reduce mucho el tiempo de pared sin relajar ninguna
#     verificación (MD5 por fichero, idempotencia, nunca silenciar un fallo).
# ==============================================================================
import tarfile, zipfile, urllib.request, urllib.error, http.client, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

CHENG_DIR = RAW_DIR / "cheng_mat"
IXI_DIR = RAW_DIR / "ixi_nii"
CHENG_DIR.mkdir(parents=True, exist_ok=True)
IXI_DIR.mkdir(parents=True, exist_ok=True)

_ERRORES_RED_TRANSITORIOS = (
    urllib.error.HTTPError, urllib.error.URLError, TimeoutError,
    ConnectionResetError, ConnectionAbortedError, http.client.IncompleteRead,
    http.client.RemoteDisconnected, OSError,
)


def _con_reintentos(fn, intentos=5, espera_base=2.0, etiqueta="", espera_max=60.0):
    """Ejecuta fn() reintentando errores de red transitorios con backoff
    exponencial + jitter (capado a `espera_max` por intento para no crecer sin
    límite en llamadas con muchos intentos). No oculta el error: si se agotan
    los intentos, se relanza con contexto claro (causa raíz + nº de intentos)
    en vez de fallar en silencio o simular que funcionó."""
    ultimo_error = None
    for intento in range(1, intentos + 1):
        try:
            return fn()
        except _ERRORES_RED_TRANSITORIOS as e:
            ultimo_error = e
            if intento == intentos:
                break
            espera = min(espera_base * (2 ** (intento - 1)), espera_max) + random.uniform(0, espera_base)
            print(f"    [reintento {intento}/{intentos}] {etiqueta}: "
                  f"{type(e).__name__}: {e} -> esperando {espera:.1f}s")
            time.sleep(espera)
    raise RuntimeError(
        f"Fallo tras {intentos} intentos en «{etiqueta}»: {type(ultimo_error).__name__}: {ultimo_error}"
    ) from ultimo_error


def md5_de(path, bloque=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bloque), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_de(path, bloque=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bloque), b""):
            h.update(chunk)
    return h.hexdigest()


def _descargar(url, destino, md5_esperado=None, etiqueta=""):
    destino = Path(destino)
    if destino.exists() and (md5_esperado is None or md5_de(destino) == md5_esperado):
        print(f"    [ok] {destino.name} ya presente y verificado")
        return destino
    print(f"    descargando {etiqueta or destino.name} ...")
    tmp = destino.with_suffix(destino.suffix + ".part")

    def _intento_descarga():
        with urllib.request.urlopen(url, timeout=120) as r, open(tmp, "wb") as f:
            total, leidos, t0 = r.length or 0, 0, time.time()
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
                leidos += len(chunk)
                if total and leidos % (32 << 20) < (1 << 20):
                    print(f"      {leidos/1e6:7.0f} / {total/1e6:.0f} MB "
                          f"({leidos/max(time.time()-t0,1e-9)/1e6:.1f} MB/s)")

    _con_reintentos(_intento_descarga, intentos=5, etiqueta=etiqueta or destino.name)
    tmp.rename(destino)
    if md5_esperado:
        real = md5_de(destino)
        if real != md5_esperado:
            raise RuntimeError(f"MD5 incorrecto en {destino.name}: {real} != {md5_esperado}")
    print(f"    [ok] {destino.name} descargado y MD5 verificado")
    return destino


# ==============================================================================
# (A) Cheng et al. 2017 — figshare
# ==============================================================================
def obtener_cheng():
    n_previos = len(list(CHENG_DIR.glob("*.mat")))
    if n_previos >= 3064:
        print(f"  [ok] Cheng ya extraído: {n_previos} ficheros .mat")
        return
    print("(A) Cheng et al. 2017 — figshare 10.6084/m9.figshare.1512427 (CC BY 4.0)")
    for nombre, (fid, md5) in FIGSHARE_ZIPS.items():
        z = _descargar(FIGSHARE_DOWNLOAD.format(file_id=fid), RAW_DIR / nombre, md5, nombre)
        with zipfile.ZipFile(z) as zf:
            miembros = [m for m in zf.namelist() if m.lower().endswith(".mat")]
            for m in miembros:
                destino = CHENG_DIR / Path(m).name
                if not destino.exists():
                    with zf.open(m) as src, open(destino, "wb") as dst:
                        shutil.copyfileobj(src, dst)
        print(f"    extraídos {len(miembros)} .mat de {nombre}")
    print(f"  Total .mat disponibles: {len(list(CHENG_DIR.glob('*.mat')))}")


# ==============================================================================
# (B) IXI — fuente primaria + fallback reproducible a Zenodo
# ==============================================================================
# Imperial College puede devolver HTTP 403 a rangos de IP de Google Colab.
# Primero se intenta el TAR primario. Si el servidor lo bloquea, se usa el
# registro Zenodo 7047668, que contiene 507 T1 de IXI con nombre original y MD5.
# En ambos casos se guardan .nii.gz con el mismo esquema que espera el manifiesto.
# ==============================================================================
import urllib.parse


def _sujetos_ixi_locales():
    """IDs IXI únicos ya presentes en disco."""
    return {
        p.name.split("-", 1)[0]
        for p in IXI_DIR.glob("*.nii.gz")
        if p.name.startswith("IXI")
    }


def _headers_http():
    return {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept": "*/*",
        "Accept-Encoding": "identity",
        "Connection": "keep-alive",
    }


def _abrir_ixi_stream():
    """Abre el TAR primario de Imperial. Puede fallar con 403 en Colab."""
    headers = _headers_http()
    headers["Referer"] = "https://brain-development.org/ixi-dataset/"
    req = urllib.request.Request(IXI_T1_TAR, headers=headers)
    return urllib.request.urlopen(req, timeout=180)


def _obtener_ixi_desde_imperial(n_sujetos):
    obtenidos = _sujetos_ixi_locales()
    t0 = time.time()

    # Solo 2 intentos aquí: un 403 de bloqueo de IP no es transitorio y no tiene
    # sentido insistir mucho antes de pasar al fallback de Zenodo.
    resp = _con_reintentos(_abrir_ixi_stream, intentos=2, espera_base=3.0,
                            etiqueta="conexión a Imperial (IXI)")
    with resp:
        print(f"    [ok] conectado a la fuente primaria: {resp.geturl()}")
        with tarfile.open(fileobj=resp, mode="r|") as tf_:
            for miembro in tf_:
                if not miembro.isfile() or not miembro.name.endswith("-T1.nii.gz"):
                    continue

                nombre = Path(miembro.name).name
                sujeto = nombre.split("-", 1)[0]
                if sujeto in obtenidos:
                    continue

                destino = IXI_DIR / nombre
                src = tf_.extractfile(miembro)
                if src is None:
                    continue

                tmp = destino.with_suffix(destino.suffix + ".part")
                try:
                    with src, open(tmp, "wb") as dst:
                        shutil.copyfileobj(src, dst, length=1 << 20)
                    tmp.replace(destino)
                finally:
                    if tmp.exists():
                        tmp.unlink(missing_ok=True)

                obtenidos.add(sujeto)
                if len(obtenidos) % 25 == 0:
                    print(
                        f"    {len(obtenidos)}/{n_sujetos} sujetos "
                        f"({time.time() - t0:.0f}s)"
                    )
                if len(obtenidos) >= n_sujetos:
                    break

    return len(obtenidos)


def _normalizar_ficheros_zenodo(meta):
    """Admite tanto la representación clásica como la nueva de la API Zenodo."""
    files = meta.get("files", [])

    if isinstance(files, list):
        return files

    if isinstance(files, dict):
        entries = files.get("entries", files)
        if isinstance(entries, list):
            return entries
        if isinstance(entries, dict):
            salida = []
            for key, info in entries.items():
                d = dict(info) if isinstance(info, dict) else {}
                d.setdefault("key", key)
                salida.append(d)
            return salida

    raise RuntimeError("Formato inesperado en la respuesta de la API de Zenodo.")


def _listar_t1_zenodo():
    """Devuelve los T1 IXI del registro Zenodo, ordenados por ID de sujeto."""
    def _peticion():
        req = urllib.request.Request(IXI_ZENODO_API, headers=_headers_http())
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read().decode("utf-8"))

    # El listado es una única petición ligera (metadatos, no el fichero en sí),
    # así que merece la pena insistir con un presupuesto generoso: Zenodo ha
    # mostrado 504 sostenidos varios minutos seguidos durante la ejecución real.
    meta = _con_reintentos(
        _peticion, intentos=8, espera_base=3.0, espera_max=45.0,
        etiqueta="listado de la API de Zenodo",
    )

    candidatos = []
    for f in _normalizar_ficheros_zenodo(meta):
        nombre = f.get("key") or f.get("filename")
        if not nombre or not nombre.endswith("-T1.nii.gz"):
            continue

        links = f.get("links") or {}
        url = links.get("self") or links.get("content") or links.get("download")
        if not url:
            url = (
                f"https://zenodo.org/records/{IXI_ZENODO_RECORD_ID}/files/"
                f"{urllib.parse.quote(nombre)}?download=1"
            )

        checksum = str(f.get("checksum") or "")
        if checksum.startswith("md5:"):
            md5 = checksum.split(":", 1)[1].lower()
        elif len(checksum) == 32:
            md5 = checksum.lower()
        else:
            raise RuntimeError(f"Zenodo no proporciona un MD5 válido para {nombre}")

        sujeto = nombre.split("-", 1)[0]
        try:
            orden = int(sujeto.replace("IXI", ""))
        except ValueError:
            orden = 10**9

        candidatos.append({
            "nombre": nombre,
            "sujeto": sujeto,
            "url": url,
            "md5": md5,
            "orden": orden,
        })

    candidatos.sort(key=lambda x: (x["orden"], x["nombre"]))

    # Seguridad: un único T1 por sujeto.
    unicos = []
    vistos = set()
    for f in candidatos:
        if f["sujeto"] not in vistos:
            vistos.add(f["sujeto"])
            unicos.append(f)

    if not unicos:
        raise RuntimeError("La API de Zenodo no devolvió ningún volumen IXI T1.")

    print(f"    Zenodo: {len(unicos)} volúmenes T1 IXI disponibles")
    return unicos


def _descargar_ixi_zenodo(f, destino):
    """Descarga un NIfTI de Zenodo y exige que su MD5 coincida."""
    destino = Path(destino)

    if destino.exists():
        if md5_de(destino) == f["md5"]:
            return False
        print(f"    [aviso] MD5 incorrecto en {destino.name}; se vuelve a descargar")
        destino.unlink()

    tmp = destino.with_suffix(destino.suffix + ".part")
    tmp.unlink(missing_ok=True)

    req = urllib.request.Request(f["url"], headers=_headers_http())
    try:
        with urllib.request.urlopen(req, timeout=180) as resp, open(tmp, "wb") as dst:
            while True:
                chunk = resp.read(1 << 20)
                if not chunk:
                    break
                dst.write(chunk)

        real = md5_de(tmp)
        if real != f["md5"]:
            raise RuntimeError(
                f"MD5 incorrecto en {f['nombre']}: {real} != {f['md5']}"
            )

        tmp.replace(destino)
        return True
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


def _obtener_ixi_desde_zenodo(n_sujetos, max_workers=6):
    disponibles = _listar_t1_zenodo()
    obtenidos = _sujetos_ixi_locales()

    faltan = n_sujetos - len(obtenidos)
    candidatos = [f for f in disponibles if f["sujeto"] not in obtenidos]
    if len(candidatos) < faltan:
        raise RuntimeError(
            f"Zenodo solo ofrece {len(candidatos)} sujetos IXI nuevos y faltan {faltan}."
        )

    print(
        f"    fallback Zenodo: descargando {faltan} sujetos que faltan "
        f"(registro {IXI_ZENODO_RECORD_ID}, {max_workers} descargas en paralelo)"
    )
    t0 = time.time()
    fallidos = []
    conseguidos_aqui = 0
    lock = threading.Lock()

    def _descargar_uno(f):
        _con_reintentos(
            lambda f=f: _descargar_ixi_zenodo(f, IXI_DIR / f["nombre"]),
            intentos=5, etiqueta=f"IXI {f['sujeto']} (Zenodo)",
        )
        return f["sujeto"]

    # Se recorren TODOS los candidatos (no solo los primeros `faltan`): si uno
    # falla persistentemente tras los reintentos, se descarta y se sigue con el
    # siguiente en vez de abortar el lote completo. Antes, un único sujeto
    # problemático tiraba abajo los ~200 ya descargados en la misma llamada.
    #
    # Las descargas se lanzan en paralelo (varias peticiones HTTP a la vez):
    # los 502/503/504 observados en la ejecución real son de latencia del lado
    # de Zenodo, no de ancho de banda del cliente, así que solapar peticiones
    # reduce mucho el tiempo total de espera sin dejar de reintentar cada
    # fichero individualmente ni de verificar su MD5. Se procesa en lotes de
    # `max_workers` y se para en cuanto se completan los `faltan` sujetos que
    # hacían falta.
    idx = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        while idx < len(candidatos) and conseguidos_aqui < faltan:
            lote = candidatos[idx: idx + max_workers]
            idx += len(lote)
            futuros = {executor.submit(_descargar_uno, f): f for f in lote}
            for fut in as_completed(futuros):
                f = futuros[fut]
                try:
                    sujeto = fut.result()
                    with lock:
                        obtenidos.add(sujeto)
                        conseguidos_aqui += 1
                        n = conseguidos_aqui
                    if n == 1 or n % 10 == 0 or n == faltan:
                        print(f"    {len(obtenidos)}/{n_sujetos} sujetos ({time.time() - t0:.0f}s)")
                except Exception as e:
                    print(f"    [aviso] {f['sujeto']} descartado tras reintentos: {e}")
                    with lock:
                        fallidos.append(f["sujeto"])
            # Pausa breve y cortés entre lotes de peticiones a la API de
            # Zenodo, para no golpear su API más rápido de lo razonable.
            time.sleep(0.3)

    if fallidos:
        print(f"    [aviso] {len(fallidos)} sujetos IXI fallaron tras reintentos y se "
              f"omitieron: {fallidos[:10]}{' ...' if len(fallidos) > 10 else ''}")

    if conseguidos_aqui < faltan:
        raise RuntimeError(
            f"Solo se consiguieron {conseguidos_aqui}/{faltan} sujetos IXI nuevos de "
            f"Zenodo tras agotar los candidatos disponibles y los reintentos "
            f"({len(fallidos)} descartados: {fallidos})."
        )

    return len(obtenidos)


def obtener_ixi(n_sujetos):
    existentes = _sujetos_ixi_locales()
    if len(existentes) >= n_sujetos:
        print(f"  [ok] IXI ya presente: {len(existentes)} sujetos")
        return

    print("(B) IXI — Brain Development / Imperial College London (CC BY-SA 3.0)")
    print(f"  objetivo: {n_sujetos} sujetos T1 sanos")

    # 1) Fuente primaria. En Colab puede devolver 403 por bloqueo del rango IP.
    try:
        _obtener_ixi_desde_imperial(n_sujetos)
    except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError,
            tarfile.TarError, OSError, RuntimeError) as e:
        print(f"  [aviso] fuente primaria no accesible: {type(e).__name__}: {e}")

    # 2) Si todavía faltan sujetos, usar un mirror de investigación en Zenodo.
    if len(_sujetos_ixi_locales()) < n_sujetos:
        print("  usando fallback verificable de Zenodo...")
        _obtener_ixi_desde_zenodo(n_sujetos)

    finales = _sujetos_ixi_locales()
    if len(finales) < n_sujetos:
        raise RuntimeError(
            f"IXI incompleto: {len(finales)} sujetos disponibles; "
            f"se esperaban {n_sujetos}."
        )

    print(f"  [ok] IXI: {len(finales)} sujetos T1 disponibles")


# ==============================================================================
# Modo de prueba: fuentes SINTÉTICAS con la misma estructura que las reales
# ==============================================================================
def generar_sinteticos(n_pacientes_tumor=30, cortes_por_paciente=12, n_sanos=30):
    """Crea .mat estilo cjdata y .nii.gz estilo IXI para validar el flujo entero
    sin descargar nada. Solo sirve para probar el código, nunca para resultados."""
    import h5py, nibabel as nib
    rng = np.random.RandomState(SEED)
    print("  generando fuentes SINTÉTICAS (solo para probar el flujo)")

    idx = 1
    for p in range(n_pacientes_tumor):
        pid = f"{1000000 + p*7}"  # PID estilo Cheng (cadena numérica)
        etiqueta = 1 + (p % 3)  # 1 meningioma, 2 glioma, 3 pituitario
        cx, cy = rng.randint(120, 390, size=2)
        for s in range(cortes_por_paciente):
            img = rng.normal(300, 60, size=(512, 512)).astype(np.float64)
            yy, xx = np.mgrid[0:512, 0:512]
            cerebro = ((xx-256)**2/180**2 + (yy-256)**2/210**2) < 1
            img[cerebro] += 900
            img[~cerebro] = rng.normal(20, 5, size=(~cerebro).sum())
            mask = ((xx-cx)**2 + (yy-cy-s*3)**2) < 30**2
            img[mask & cerebro] += 1400
            with h5py.File(CHENG_DIR / f"{idx}.mat", "w") as f:
                g = f.create_group("cjdata")
                g.create_dataset("label", data=np.array([[float(etiqueta)]]))
                g.create_dataset("PID", data=np.array([[ord(c)] for c in pid], dtype=np.uint16))
                g.create_dataset("image", data=img.astype(np.uint16).T)
                g.create_dataset("tumorMask", data=(mask & cerebro).astype(np.uint8).T)
            idx += 1

    for s in range(n_sanos):
        sid = f"IXI{s+2:03d}"
        sitio = ("Guys", "HH", "IOP")[s % 3]
        vol = rng.normal(60, 12, size=(150, 180, 150)).astype(np.float32)
        zz, yy, xx = np.mgrid[0:150, 0:180, 0:150]
        cerebro = ((xx-75)**2/55**2 + (yy-90)**2/70**2 + (zz-75)**2/58**2) < 1
        vol[cerebro] += 420
        vol[~cerebro] *= 0.15
        nib.save(nib.Nifti1Image(vol, np.eye(4)), IXI_DIR / f"{sid}-{sitio}-{1000+s}-T1.nii.gz")

    print(f"  sintéticos: {idx-1} .mat ({n_pacientes_tumor} pacientes) "
          f"y {n_sanos} volúmenes IXI")


# ==============================================================================
if USAR_DATOS_SINTETICOS:
    if not list(CHENG_DIR.glob("*.mat")):
        generar_sinteticos()
    else:
        print("  sintéticos ya generados")
else:
    obtener_cheng()
    obtener_ixi(N_SUJETOS_SANOS)

n_mat = len(list(CHENG_DIR.glob("*.mat")))
n_nii = len(list(IXI_DIR.glob("*.nii.gz")))
print(f"\nFuentes disponibles -> Cheng: {n_mat} .mat | IXI: {n_nii} volúmenes")
assert n_mat > 0 and n_nii > 0, "No hay datos de origen: revisa la conexión o activa USAR_DATOS_SINTETICOS."


In [ ]:
# ==============================================================================
# 1.3  MANIFIESTO DE TRAZABILIDAD  (imagen -> corte -> volumen -> paciente -> fuente)
# ==============================================================================
# Aquí es donde esta versión se separa de la anterior. Se recorre cada fichero
# ORIGINAL y se extrae el identificador de paciente que la propia fuente publica:
#
#   Cheng (.mat) : cjdata.PID          -> patient_id REAL
#   IXI  (.nii.gz): 'IXI002-Guys-0828' -> patient_id REAL (IXI002) + sitio (Guys)
#
# Ningún patient_id se deduce del nombre de un JPEG. Si una imagen no tiene
# identificador verificable, se marca patient_id_source = "desconocido" y en
# STRICT_PATIENT_MODE el cuaderno se detiene antes de entrenar (celda 1.7).
#
# Además, todos los cortes de AMBAS fuentes se exportan con el mismo tratamiento
# —recorte a la caja del cerebro, normalización por percentiles 1-99 y PNG de
# 8 bits a PNG_SIZE— para cerrar dos vías conocidas de sesgo de procedencia
# (resolución nativa distinta y artefactos de compresión JPEG). Esto NO elimina
# el sesgo (siguen siendo escáneres y protocolos distintos), solo dos de sus vías.
# ==============================================================================
import h5py, nibabel as nib
from PIL import Image
import imagehash

CHENG_URL = f"https://doi.org/10.6084/m9.figshare.{FIGSHARE_ARTICLE_ID}"
IXI_URL   = "https://brain-development.org/ixi-dataset/"
CHENG_TIPO = {1: "meningioma", 2: "glioma", 3: "pituitario"}

COLUMNAS = [
    "local_file", "label", "patient_id", "patient_id_source", "volume_id",
    "slice_id", "view", "source_dataset", "source_url", "original_file",
    "site", "tumor_type", "modality", "sha256", "phash",
    "width", "height", "duplicate_cluster", "subset",
]


def _a_png(arr2d, destino):
    """Recorta a la caja del cerebro, normaliza por percentiles y guarda PNG 8-bit."""
    a = np.nan_to_num(np.asarray(arr2d, dtype=np.float32))
    if a.ndim != 2 or min(a.shape) < 8:
        return None
    # --- caja del cerebro: todo lo que supere el 10 % del rango robusto ---
    p1, p99 = np.percentile(a, (1, 99))
    if p99 - p1 < 1e-6:
        return None
    norm = np.clip((a - p1) / (p99 - p1), 0, 1)
    mask = norm > 0.10
    if mask.sum() < 64:
        return None
    ys, xs = np.where(mask)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    # caja cuadrada centrada con un 4 % de margen, para no deformar la anatomía
    cy, cx = (y0 + y1) / 2.0, (x0 + x1) / 2.0
    lado = max(y1 - y0, x1 - x0) * 1.04
    y0 = int(max(0, round(cy - lado / 2))); y1 = int(min(a.shape[0], round(cy + lado / 2)))
    x0 = int(max(0, round(cx - lado / 2))); x1 = int(min(a.shape[1], round(cx + lado / 2)))
    recorte = norm[y0:y1, x0:x1]
    if min(recorte.shape) < 8:
        return None
    im = Image.fromarray((recorte * 255).astype(np.uint8), mode="L")
    im = im.resize(PNG_SIZE, Image.LANCZOS)
    im.save(destino, format="PNG", optimize=True)
    return im


def _leer_cjdata(path):
    """Devuelve (PID, label, image) de un .mat de Cheng.
    Los .mat originales son MATLAB v7.3 (HDF5); se admite también v7 por si
    alguien redistribuye la colección en el formato antiguo."""
    try:
        with h5py.File(path, "r") as f:
            g = f["cjdata"]
            pid_raw = np.array(g["PID"]).flatten()
            pid = "".join(chr(int(c)) for c in pid_raw if int(c) > 0).strip()
            label = int(np.array(g["label"]).flatten()[0])
            img = np.array(g["image"]).T          # HDF5 devuelve la matriz transpuesta
            return pid, label, img
    except OSError:
        from scipy.io import loadmat
        m = loadmat(path, squeeze_me=True, struct_as_record=False)["cjdata"]
        pid = str(m.PID).strip()
        return pid, int(m.label), np.asarray(m.image)


# ------------------------------------------------------------------ CHENG ---
def manifiesto_cheng():
    filas, sin_pid = [], 0
    ficheros = sorted(CHENG_DIR.glob("*.mat"), key=lambda p: int(p.stem) if p.stem.isdigit() else 0)
    for p in ficheros:
        try:
            pid, label, img = _leer_cjdata(p)
        except Exception as e:
            print(f"    [aviso] no se pudo leer {p.name}: {e}")
            continue
        destino = IMG_DIR / f"cheng_{p.stem}.png"
        if _a_png(img, destino) is None:
            continue
        tiene_pid = bool(pid) and pid.lower() not in ("nan", "none")
        sin_pid += (not tiene_pid)
        filas.append({
            "local_file": str(destino),
            "label": "tumor",
            "patient_id": f"CHENG-{pid}" if tiene_pid else "",
            "patient_id_source": "real:cjdata.PID" if tiene_pid else "desconocido",
            # Cheng no distribuye la agrupación por volumen/serie: solo el paciente.
            # No se inventa: se marca como no disponible y el paciente es la unidad.
            "volume_id": "NO_DISPONIBLE",
            "slice_id": p.stem,
            "view": "NO_DISPONIBLE",
            "source_dataset": "figshare/Cheng2017",
            "source_url": CHENG_URL,
            "original_file": p.name,
            "site": "Nanfang/Tianjin (CN)",
            "tumor_type": CHENG_TIPO.get(label, f"desconocido({label})"),
            "modality": "T1-CE",
        })
    print(f"  Cheng  : {len(filas)} cortes exportados | sin PID: {sin_pid}")
    return filas


# -------------------------------------------------------------------- IXI ---
def _cortes_ixi(vol):
    """Devuelve [(plano, indice, corte2d)] muestreando la zona central del cerebro."""
    salida = []
    cuerpo = vol > np.percentile(vol, 60)
    for eje, plano in zip((0, 1, 2), ("sagital", "coronal", "axial")):
        proj = cuerpo.any(axis=tuple(a for a in (0, 1, 2) if a != eje))
        idxs = np.where(proj)[0]
        if len(idxs) < 8:
            continue
        lo, hi = idxs.min(), idxs.max()
        centro = np.linspace(lo + 0.30 * (hi - lo), lo + 0.70 * (hi - lo), CORTES_POR_PLANO)
        for c in centro:
            k = int(round(c))
            corte = np.take(vol, k, axis=eje)
            salida.append((plano, k, corte))
    return salida


def manifiesto_ixi():
    filas = []
    for p in sorted(IXI_DIR.glob("*.nii.gz")):
        partes = p.name.replace(".nii.gz", "").split("-")
        sujeto = partes[0]
        sitio = partes[1] if len(partes) > 1 else "desconocido"
        try:
            img = nib.as_closest_canonical(nib.load(str(p)))
            vol = np.asarray(img.dataobj, dtype=np.float32)
        except Exception as e:
            print(f"    [aviso] no se pudo leer {p.name}: {e}")
            continue
        if vol.ndim == 4:
            vol = vol[..., 0]
        if vol.ndim != 3:
            continue
        for plano, k, corte in _cortes_ixi(vol):
            destino = IMG_DIR / f"ixi_{sujeto}_{plano}_{k}.png"
            if _a_png(corte, destino) is None:
                continue
            filas.append({
                "local_file": str(destino),
                "label": "notumor",
                "patient_id": f"IXI-{sujeto}",
                "patient_id_source": "real:IXI subject ID",
                "volume_id": p.name.replace(".nii.gz", ""),
                "slice_id": f"{plano}:{k}",
                "view": plano,
                "source_dataset": "IXI",
                "source_url": IXI_URL,
                "original_file": p.name,
                "site": sitio,
                "tumor_type": "ninguno",
                "modality": "T1",
            })
    print(f"  IXI    : {len(filas)} cortes exportados")
    return filas


# ------------------------------------------------- construcción y hashes ----
print("Construyendo el manifiesto a partir de los ficheros ORIGINALES...")
filas = manifiesto_cheng() + manifiesto_ixi()
manifest = pd.DataFrame(filas)

print("Calculando SHA-256 y hash perceptual de cada imagen...")
sha, ph, ws, hs = [], [], [], []
for f in manifest["local_file"]:
    sha.append(sha256_de(f))
    with Image.open(f) as im:
        ph.append(str(imagehash.phash(im.convert("L"), hash_size=PHASH_SIZE)))
        ws.append(im.width); hs.append(im.height)
manifest["sha256"] = sha
manifest["phash"] = ph
manifest["width"] = ws
manifest["height"] = hs
manifest["duplicate_cluster"] = ""
manifest["subset"] = ""
manifest = manifest[COLUMNAS]

print(f"\nManifiesto inicial: {len(manifest)} imágenes")
print(manifest.groupby(["source_dataset", "label"]).agg(
    imagenes=("local_file", "size"), pacientes=("patient_id", "nunique")))

In [ ]:
# ==============================================================================
# 1.4  DUPLICADOS EXACTOS, CASI-DUPLICADOS Y CONFLICTOS DE ETIQUETA
# ==============================================================================
# Aquí no se borra nada en masa; los casi-duplicados se agrupan:
#
#   Duplicados EXACTOS  -> SHA-256. Se conserva una copia. Si dos ficheros
#                          idénticos tienen etiquetas distintas es un CONFLICTO
#                          DE ETIQUETA y se marca como fallo de auditoría.
#
#   CASI-duplicados     -> hash perceptual de 256 bits (16x16) y confirmación por
#                          SSIM. NO se borran: dos cortes contiguos del mismo
#                          cerebro se parecen mucho y son información clínica
#                          legítima. Lo que se hace es agruparlos en un
#                          duplicate_cluster.
#
# El pHash NO se usa como sustituto del paciente. Se usa para detectar resubidas,
# reescalados, recompresiones y solapes entre fuentes, y para forzar que un
# clúster de imágenes casi idénticas no se reparta entre subconjuntos.
#
# La unidad de reparto de la celda 1.5 será la COMPONENTE CONEXA de:
#       (mismo patient_id)  ∪  (mismo fichero exacto)  ∪  (casi-duplicados)
# Con eso, por construcción, no puede quedar un par casi-idéntico cruzando
# train/val/test: no hay que "conseguir un cero", el cero está garantizado.
# ==============================================================================
from skimage.metrics import structural_similarity as ssim_fn

# ------------------------------------------------------------- union-find ---
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n)); self.r = [0]*n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.r[ra] < self.r[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]:
            self.r[ra] += 1
        return True


_POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def _bits_de_phash(serie):
    """Convierte los pHash hexadecimales en una matriz (n, nbytes) de uint8."""
    mats = [imagehash.hex_to_hash(h).hash.flatten() for h in serie]
    bits = np.array(mats, dtype=bool)
    return np.packbits(bits, axis=1)


def _pares_cercanos(packed, umbral, bloque=512):
    """Pares (i, j) con distancia de Hamming <= umbral, por bloques."""
    n = packed.shape[0]
    pares = []
    for i0 in range(0, n, bloque):
        i1 = min(i0 + bloque, n)
        a = packed[i0:i1][:, None, :]
        for j0 in range(i0, n, bloque):
            j1 = min(j0 + bloque, n)
            d = _POPCOUNT[np.bitwise_xor(a, packed[None, j0:j1, :])].sum(axis=2)
            ii, jj = np.where(d <= umbral)
            for x, y in zip(ii, jj):
                gi, gj = i0 + int(x), j0 + int(y)
                if gi < gj:
                    pares.append((gi, gj, int(d[x, y])))
    return pares


def _ssim_par(f1, f2):
    with Image.open(f1) as a, Image.open(f2) as b:
        A = np.asarray(a.convert("L").resize((128, 128)), dtype=np.float32)
        B = np.asarray(b.convert("L").resize((128, 128)), dtype=np.float32)
    return float(ssim_fn(A, B, data_range=255.0))


# ------------------------------------------------- (A) duplicados exactos ---
print("(A) Duplicados EXACTOS (SHA-256)")
grupos_sha = manifest.groupby("sha256")
conflictos_etiqueta = []
for h, g in grupos_sha:
    if len(g) > 1 and g["label"].nunique() > 1:
        conflictos_etiqueta.append({
            "sha256": h, "n": len(g),
            "etiquetas": sorted(g["label"].unique()),
            "ficheros": list(g["original_file"]),
            "fuentes": sorted(g["source_dataset"].unique()),
        })

dups_exactos = manifest.duplicated(subset="sha256", keep="first")
n_dup_exactos = int(dups_exactos.sum())
dup_exactos_cruzan_fuente = int(sum(
    1 for _, g in grupos_sha if len(g) > 1 and g["source_dataset"].nunique() > 1))
print(f"    ficheros idénticos redundantes : {n_dup_exactos}")
print(f"    clústeres exactos entre fuentes: {dup_exactos_cruzan_fuente}")
print(f"    CONFLICTOS de etiqueta         : {len(conflictos_etiqueta)}")

manifest_dedup = manifest[~dups_exactos].reset_index(drop=True).copy()
print(f"    imágenes tras quitar exactos   : {len(manifest_dedup)}")

# ------------------------------------------------- (B) casi-duplicados -----
print("\n(B) CASI-duplicados (pHash 256 bits + confirmación por SSIM)")
packed = _bits_de_phash(manifest_dedup["phash"])
candidatos = _pares_cercanos(packed, HAMMING_NEAR)
print(f"    pares candidatos por pHash     : {len(candidatos)}")

confirmados, rechazados = [], 0
for i, j, d in candidatos:
    s = _ssim_par(manifest_dedup.at[i, "local_file"], manifest_dedup.at[j, "local_file"])
    if s >= SSIM_CONFIRMACION:
        confirmados.append((i, j, d, s))
    else:
        rechazados += 1
print(f"    confirmados por SSIM >= {SSIM_CONFIRMACION}   : {len(confirmados)}")
print(f"    descartados (parecidos pero no el mismo caso): {rechazados}")

# ------------------------------------------------- (C) componentes conexas --
uf = UnionFind(len(manifest_dedup))
for _, g in manifest_dedup.groupby("patient_id"):
    idxs = list(g.index)
    for k in idxs[1:]:
        uf.union(idxs[0], k)
for i, j, _, _ in confirmados:
    uf.union(i, j)

manifest_dedup["split_group"] = [f"G{uf.find(i):05d}" for i in range(len(manifest_dedup))]
manifest_dedup["duplicate_cluster"] = ""
for i, j, _, _ in confirmados:
    c = f"D{uf.find(i):05d}"
    manifest_dedup.at[i, "duplicate_cluster"] = c
    manifest_dedup.at[j, "duplicate_cluster"] = c

# --------------------------------------------- (D) diagnóstico de clústeres -
near_entre_fuentes = sum(
    1 for i, j, _, _ in confirmados
    if manifest_dedup.at[i, "source_dataset"] != manifest_dedup.at[j, "source_dataset"])
near_entre_clases = sum(
    1 for i, j, _, _ in confirmados
    if manifest_dedup.at[i, "label"] != manifest_dedup.at[j, "label"])
near_entre_pacientes = sum(
    1 for i, j, _, _ in confirmados
    if manifest_dedup.at[i, "patient_id"] != manifest_dedup.at[j, "patient_id"])

print(f"\n    casi-duplicados ENTRE FUENTES   : {near_entre_fuentes}")
print(f"    casi-duplicados ENTRE CLASES    : {near_entre_clases}  "
      f"(si > 0 hay que revisarlos uno a uno)")
print(f"    casi-duplicados ENTRE PACIENTES : {near_entre_pacientes}  "
      f"(fusionan pacientes en un mismo grupo de reparto)")

grupos = manifest_dedup["split_group"].nunique()
pacientes = manifest_dedup["patient_id"].nunique()
print(f"\n    pacientes distintos             : {pacientes}")
print(f"    grupos de reparto (paciente ∪ casi-duplicado): {grupos}")

dup_report = pd.DataFrame(
    [{"img_a": Path(manifest_dedup.at[i, "local_file"]).name,
      "img_b": Path(manifest_dedup.at[j, "local_file"]).name,
      "hamming": d, "ssim": round(s, 4),
      "paciente_a": manifest_dedup.at[i, "patient_id"],
      "paciente_b": manifest_dedup.at[j, "patient_id"],
      "clase_a": manifest_dedup.at[i, "label"],
      "clase_b": manifest_dedup.at[j, "label"],
      "fuente_a": manifest_dedup.at[i, "source_dataset"],
      "fuente_b": manifest_dedup.at[j, "source_dataset"]}
     for i, j, d, s in confirmados])
dup_report.to_csv(ART_DIR / "duplicados_report.csv", index=False)
if len(dup_report):
    print("\n    Muestra de casi-duplicados confirmados:")
    print(dup_report.head(10).to_string(index=False))

In [ ]:
# ------------------------------------------------------------------------------
# 1.4b  Ejemplos visualizables de los clústeres detectados
# ------------------------------------------------------------------------------
# Una tabla de números no permite juzgar si un "casi-duplicado" es realmente el
# mismo caso o dos cortes vecinos distintos. Aquí se muestran los pares más
# parecidos para poder revisarlos a ojo y justificar la decisión.
# ------------------------------------------------------------------------------
import matplotlib.pyplot as plt

N_EJEMPLOS_DUP = 5
if len(dup_report) == 0:
    print("No se han confirmado casi-duplicados: no hay nada que visualizar.")
else:
    top = dup_report.sort_values(["ssim"], ascending=False).head(N_EJEMPLOS_DUP)
    fig, ax = plt.subplots(len(top), 2, figsize=(6, 3 * len(top)))
    ax = np.atleast_2d(ax)
    for r, (_, fila) in enumerate(top.iterrows()):
        for c, lado in enumerate(("a", "b")):
            ruta = IMG_DIR / fila[f"img_{lado}"]
            ax[r, c].imshow(Image.open(ruta), cmap="gray")
            ax[r, c].set_title(
                f"{fila[f'img_{lado}']}\n{fila[f'paciente_{lado}']} · {fila[f'clase_{lado}']}",
                fontsize=8)
            ax[r, c].axis("off")
        ax[r, 0].set_ylabel(f"H={fila['hamming']}  SSIM={fila['ssim']:.3f}")
    fig.suptitle("Pares confirmados como casi-duplicados (se agrupan, NO se borran)",
                 fontsize=11)
    plt.tight_layout()
    guardar_figura(fig, "1_4b_casi_duplicados")
    plt.show()

    mismos = int((top["paciente_a"] == top["paciente_b"]).sum())
    print(f"De los {len(top)} pares mostrados, {mismos} son del MISMO paciente "
          f"(cortes vecinos del mismo volumen) y {len(top)-mismos} son de pacientes "
          f"distintos.\nLos que son de pacientes distintos quedan igualmente unidos "
          f"en un mismo grupo de reparto: es la decisión conservadora.")

In [ ]:
# ==============================================================================
# 1.5  REPARTO TRAIN / VAL / TEST A NIVEL DE PACIENTE  (antes de cualquier augmentation)
# ==============================================================================
# El reparto se hace sobre la TABLA DE GRUPOS, no sobre la tabla de imágenes.
# La unidad es el split_group calculado en 1.4:
#
#        split_group  =  componente conexa de (paciente) ∪ (casi-duplicados)
#
# Primero se reparten los grupos, después se expanden sus imágenes. Así:
#   * todos los cortes de un paciente caen SIEMPRE en el mismo subconjunto;
#   * ningún par casi-idéntico puede cruzar subconjuntos (por construcción).
#
# La estratificación es real: se estratifica por la pareja (clase, fuente) del
# grupo, no por una variable que el splitter luego ignore.
# ==============================================================================
from sklearn.model_selection import train_test_split

# --------------------------------------------------- tabla de GRUPOS --------
grupo_tab = (manifest_dedup
             .groupby("split_group")
             .agg(label=("label", lambda s: s.mode().iat[0]),
                  n_clases=("label", "nunique"),
                  source=("source_dataset", lambda s: s.mode().iat[0]),
                  n_fuentes=("source_dataset", "nunique"),
                  n_pacientes=("patient_id", "nunique"),
                  n_imagenes=("local_file", "size"))
             .reset_index())

mixtos = grupo_tab[grupo_tab["n_clases"] > 1]
if len(mixtos):
    print(f"[AVISO] {len(mixtos)} grupos contienen más de una clase. "
          f"Se marcan para revisión y se excluyen del entrenamiento.")
    excluidos = set(mixtos["split_group"])
    manifest_dedup = manifest_dedup[~manifest_dedup["split_group"].isin(excluidos)].copy()
    grupo_tab = grupo_tab[grupo_tab["n_clases"] == 1].reset_index(drop=True)

grupo_tab["estrato"] = grupo_tab["label"] + "|" + grupo_tab["source"]
print(f"Grupos de reparto: {len(grupo_tab)}  "
      f"(pacientes: {manifest_dedup['patient_id'].nunique()}, "
      f"imágenes: {len(manifest_dedup)})")
print(grupo_tab.groupby(["source", "label"]).agg(
    grupos=("split_group", "size"), imagenes=("n_imagenes", "sum")))

# --------------------------------------------------- reparto de grupos ------
estratos = grupo_tab["estrato"]
usar_estrato = estratos.value_counts().min() >= 3
if not usar_estrato:
    print("[AVISO] algún estrato tiene menos de 3 grupos: se reparte sin estratificar.")

g_trainval, g_test = train_test_split(
    grupo_tab, test_size=FRAC_TEST, random_state=SEED,
    stratify=estratos if usar_estrato else None)

frac_val_rel = FRAC_VAL / (1.0 - FRAC_TEST)
est_tv = g_trainval["estrato"]
usar_estrato_tv = est_tv.value_counts().min() >= 3
g_train, g_val = train_test_split(
    g_trainval, test_size=frac_val_rel, random_state=SEED,
    stratify=est_tv if usar_estrato_tv else None)

asignacion = {}
for nombre, tabla in (("train", g_train), ("val", g_val), ("test", g_test)):
    for g in tabla["split_group"]:
        asignacion[g] = nombre

manifest_dedup["subset"] = manifest_dedup["split_group"].map(asignacion)
data = manifest_dedup.dropna(subset=["subset"]).reset_index(drop=True)

# --------------------------------------------------- comprobaciones ---------
pac = {s: set(data.loc[data["subset"] == s, "patient_id"]) for s in ("train", "val", "test")}
grp = {s: set(data.loc[data["subset"] == s, "split_group"]) for s in ("train", "val", "test")}
sha = {s: set(data.loc[data["subset"] == s, "sha256"]) for s in ("train", "val", "test")}
clu = {s: set(x for x in data.loc[data["subset"] == s, "duplicate_cluster"] if x)
       for s in ("train", "val", "test")}

# Verificación DIRECTA (no por construcción) de que ningún par casi-idéntico
# cruza subconjuntos, y cuántos pares que el pHash marcó como sospechosos pero el
# SSIM descartó quedan repartidos. Estos últimos NO son fuga —se ha comprobado que
# son imágenes distintas— pero se cuentan y se dejan a la vista en lugar de
# afirmar simplemente que el problema no existe.
_sub = manifest_dedup["subset"]
def _cruza(i, j):
    return (i in _sub.index and j in _sub.index
            and pd.notna(_sub.at[i]) and pd.notna(_sub.at[j])
            and _sub.at[i] != _sub.at[j])

near_confirmados_cruzando = sum(1 for i, j, _, _ in confirmados if _cruza(i, j))
near_candidatos_cruzando = sum(1 for i, j, _ in candidatos if _cruza(i, j))
print(f"\nPares pHash sospechosos que cruzan subconjuntos: {near_candidatos_cruzando}")
print(f"   de ellos CONFIRMADOS como casi-duplicados (SSIM >= {SSIM_CONFIRMACION}): "
      f"{near_confirmados_cruzando}")
print("   (los confirmados deben ser 0: el reparto agrupa cada clúster entero;")
print("    los no confirmados son imágenes visualmente parecidas pero distintas,")
print("    revisables en duplicados_report.csv y en la celda 1.4b)")

print("\nComprobaciones de fuga (deben ser todas 0):")
overlaps = {}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    overlaps[f"pacientes {a}-{b}"] = len(pac[a] & pac[b])
    overlaps[f"grupos {a}-{b}"]    = len(grp[a] & grp[b])
    overlaps[f"sha256 {a}-{b}"]    = len(sha[a] & sha[b])
    overlaps[f"clúster {a}-{b}"]   = len(clu[a] & clu[b])
overlaps["casi-duplicados confirmados cruzando subsets"] = near_confirmados_cruzando
for k, v in overlaps.items():
    print(f"   {k:24s}: {v}")

assert len(pac["train"] & pac["val"]) == 0,  "FUGA: paciente compartido train-val"
assert len(pac["train"] & pac["test"]) == 0, "FUGA: paciente compartido train-test"
assert len(pac["val"] & pac["test"]) == 0,   "FUGA: paciente compartido val-test"
assert len(sha["train"] & sha["test"]) == 0, "FUGA: fichero idéntico train-test"
assert all(v == 0 for v in overlaps.values()), "FUGA detectada en el reparto"

print("\nReparto resultante:")
resumen_split = (data.groupby("subset")
                 .agg(imagenes=("local_file", "size"),
                      pacientes=("patient_id", "nunique"),
                      tumor=("label", lambda s: int((s == "tumor").sum())),
                      notumor=("label", lambda s: int((s == "notumor").sum())))
                 .reindex(["train", "val", "test"]))
resumen_split["pac_tumor"] = [
    data[(data.subset == s) & (data.label == "tumor")]["patient_id"].nunique()
    for s in resumen_split.index]
resumen_split["pac_notumor"] = [
    data[(data.subset == s) & (data.label == "notumor")]["patient_id"].nunique()
    for s in resumen_split.index]
print(resumen_split)

### 2.5 Verificación y congelación del split MULTICLASE (Cheng, 3 clases)

Este split se congeló **fuera** de este cuaderno, a partir del `manifest_final.csv` de
la ejecución SLURM 70436 (la que dio 79,7 % de balanced accuracy patient-level en la tarea
de control de 3 clases). El detalle completo de esa verificación está en
`baseline_multiclase_cheng.md`.

Esta celda **nunca genera un split nuevo**. Reconstruye la asignación paciente → subset
exactamente como la calcula el reparto de la sección 2.4, la reduce a los pacientes de
Cheng, recalcula su huella y la compara contra la huella congelada. Si no coinciden, el
cuaderno se **detiene** con un error explícito.

In [ ]:
# ==============================================================================
# 2.5  VERIFICACIÓN Y CONGELACIÓN DEL SPLIT MULTICLASE (Cheng, 3 clases)
# ==============================================================================
# NUNCA se genera un split nuevo aquí. Si el manifiesto actual no reproduce
# EXACTAMENTE el split congelado, el cuaderno se DETIENE con un error explícito.
import hashlib

SPLIT_HASH_MULTICLASE_CONGELADO = "fda7e2daeb9ec2de"
SPLIT_HASH_MULTICLASE_CONGELADO_FULL = (
    "fda7e2daeb9ec2de596e807728454329cd140398dd10f834599457eca86c0092")

if not SPLIT_MULTICLASE_JSON.exists():
    raise RuntimeError(
        f"No se encuentra el split multiclase congelado en {SPLIT_MULTICLASE_JSON}.\n"
        "Copia ahí split_multiclase_cheng.json (y opcionalmente el .csv) ANTES de "
        "continuar. Este cuaderno NO debe generar un split multiclase nuevo.")

_split_congelado = json.loads(SPLIT_MULTICLASE_JSON.read_text(encoding="utf-8"))


def _hash_texto(txt):
    return hashlib.sha256(txt.encode("utf-8")).hexdigest()[:16]


# ---- reconstrucción desde el manifiesto ACTUAL, igual que hacía Apéndice A ----
cheng = data[data.source_dataset == "figshare/Cheng2017"].reset_index(drop=True)
TIPOS = sorted(cheng["tumor_type"].unique())
MAPA_TIPO = {t: i for i, t in enumerate(TIPOS)}

pac_tab_multiclase = (
    cheng.groupby("patient_id")
    .agg(subset=("subset", "first"), tumor_type=("tumor_type", "first"),
         n_slices=("patient_id", "size"))
    .reset_index().sort_values("patient_id").reset_index(drop=True)
)

_split_txt_actual = pac_tab_multiclase[["patient_id", "subset", "tumor_type"]].to_csv(index=False)
SPLIT_HASH_MULTICLASE_ACTUAL = _hash_texto(_split_txt_actual)

print(f"SPLIT_HASH_MULTICLASE (congelado)   : {SPLIT_HASH_MULTICLASE_CONGELADO}")
print(f"SPLIT_HASH_MULTICLASE (recalculado) : {SPLIT_HASH_MULTICLASE_ACTUAL}")

if SPLIT_HASH_MULTICLASE_ACTUAL != SPLIT_HASH_MULTICLASE_CONGELADO:
    raise RuntimeError(
        "EL SPLIT MULTICLASE RECALCULADO NO COINCIDE CON EL SPLIT CONGELADO.\n"
        f"  Congelado   : {SPLIT_HASH_MULTICLASE_CONGELADO}\n"
        f"  Recalculado : {SPLIT_HASH_MULTICLASE_ACTUAL}\n"
        "El manifest_final.csv actual (u otra pieza del pipeline anterior a esta "
        "celda) ha cambiado respecto al que produjo el 79,7% de balanced accuracy. "
        "NO se genera un split nuevo automáticamente: detente y revisa qué ha "
        "cambiado (descarga, deduplicación, orden de filas...) antes de continuar.")

print("\nEl split multiclase reconstruido coincide EXACTAMENTE con el congelado. "
      "Se usará esta asignación para toda la tarea 'tumor_type'.\n")

# ---- comprobaciones adicionales de integridad ----
assert pac_tab_multiclase["patient_id"].duplicated().sum() == 0, "paciente duplicado"
_tr = set(pac_tab_multiclase.loc[pac_tab_multiclase.subset == "train", "patient_id"])
_va = set(pac_tab_multiclase.loc[pac_tab_multiclase.subset == "val", "patient_id"])
_te = set(pac_tab_multiclase.loc[pac_tab_multiclase.subset == "test", "patient_id"])
assert not (_tr & _va), "FUGA multiclase train-val"
assert not (_tr & _te), "FUGA multiclase train-test"
assert not (_va & _te), "FUGA multiclase val-test"
assert len(_tr) == 163 and len(_va) == 35 and len(_te) == 35, (
    f"Tamaños de subconjunto distintos de los congelados: "
    f"train={len(_tr)} val={len(_va)} test={len(_te)}")
for _nombre, _s in (("train", _tr), ("val", _va), ("test", _te)):
    _clases = set(pac_tab_multiclase.loc[pac_tab_multiclase.patient_id.isin(_s), "tumor_type"])
    assert _clases == {"glioma", "meningioma", "pituitario"}, f"falta alguna clase en {_nombre}"

ap_tr = cheng[cheng.subset == "train"].reset_index(drop=True)
ap_va = cheng[cheng.subset == "val"].reset_index(drop=True)
ap_te = cheng[cheng.subset == "test"].reset_index(drop=True)
assert not (set(ap_tr.patient_id) & set(ap_te.patient_id)), "fuga train-test (Cheng)"

for n, d in (("train", ap_tr), ("val", ap_va), ("test", ap_te)):
    print(f"  {n:5s}: {len(d):4d} cortes · {d['patient_id'].nunique():3d} pacientes · "
          f"{d['tumor_type'].value_counts().to_dict()}")

print(f"\nClases de la tarea multiclase: {MAPA_TIPO}")
print("Todos los asserts de integridad del split multiclase: OK")


In [ ]:
# ==============================================================================
# 1.6  AUDITORÍA DE SESGO DE PROCEDENCIA  (el riesgo principal de este dataset)
# ==============================================================================
# Al construir la clase `tumor` con Cheng y la clase `notumor` con IXI, la fuente
# predice la etiqueta de forma PERFECTA. Es una limitación estructural: no existe
# ninguna colección pública con tumor y control sano del mismo escáner y protocolo.
#
# Lo honesto no es esconderlo, es medirlo. Esta celda hace cuatro cosas:
#
#   (1) tablas fuente x clase y fuente x paciente;
#   (2) descriptores de bajo nivel por imagen (intensidad, contraste, borde negro,
#       energía de bordes, entropía, tamaño de fichero) y su separación por clase;
#   (3) un "techo de atajo": qué accuracy alcanza un modelo TRIVIAL que solo ve
#       esos descriptores, evaluado con validación cruzada AGRUPADA POR PACIENTE
#       (si se evalúa sin agrupar, este número también sale inflado);
#   (4) un control de dominio posible dentro de la clase sana: IXI tiene tres
#       sitios de adquisición (Guy's, HH, IOP) con escáneres distintos, así que se
#       puede comprobar si el modelo distingue sitios.
#
# Interpretación: cuanto más se acerque (3) a 1.0, menos significativo es el resultado
# binario. La cifra se arrastra hasta las conclusiones automáticas.
# ==============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedGroupKFold

print("(1) Distribución fuente x clase")
tab_fuente_clase = pd.crosstab(data["source_dataset"], data["label"])
print(tab_fuente_clase)
print("\n    fuente x clase, contando PACIENTES:")
tab_fuente_pac = (data.groupby(["source_dataset", "label"])["patient_id"]
                  .nunique().unstack(fill_value=0))
print(tab_fuente_pac)
print("\n    fuente x subconjunto:")
print(pd.crosstab(data["subset"], data["source_dataset"]))

fuente_predice_clase = float((tab_fuente_clase.max(axis=1) /
                              tab_fuente_clase.sum(axis=1)).mean())
print(f"\n    Pureza media de clase por fuente: {fuente_predice_clase:.3f} "
      f"(1.000 = la fuente determina por completo la etiqueta)")

# ------------------------------------------------------------------ (2) -----
print("\n(2) Descriptores de bajo nivel por imagen")


def descriptores(path):
    with Image.open(path) as im:
        g = np.asarray(im.convert("L"), dtype=np.float32)
    gn = g / 255.0
    hist = np.histogram(g, bins=32, range=(0, 255))[0].astype(np.float64)
    p = hist / max(hist.sum(), 1)
    entropia = float(-(p[p > 0] * np.log2(p[p > 0])).sum())
    bordes = float(np.abs(np.diff(gn, axis=0)).mean() + np.abs(np.diff(gn, axis=1)).mean())
    p5, p50, p95 = np.percentile(gn, (5, 50, 95))
    return [float(gn.mean()), float(gn.std()), float(p5), float(p50), float(p95),
            float(p95 - p5), float((g < 12).mean()), bordes, entropia,
            float(os.path.getsize(path)) / 1024.0]

NOMBRES_DESC = ["media", "std", "p5", "p50", "p95", "contraste",
                "frac_negro", "energia_bordes", "entropia", "kB_fichero"]

X = np.array([descriptores(f) for f in data["local_file"]], dtype=np.float32)
desc_df = pd.DataFrame(X, columns=NOMBRES_DESC)
desc_df["label"] = data["label"].values
desc_df["source"] = data["source_dataset"].values
print(desc_df.groupby("source")[NOMBRES_DESC].mean().round(3).to_string())

# ------------------------------------------------------------------ (3) -----
print("\n(3) Techo de atajo: clasificador TRIVIAL con solo estos descriptores")
y_bin = (data["label"] == "tumor").astype(int).values
grupos_pac = data["patient_id"].values
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
n_folds_bias = int(min(5, pd.Series(y_bin).value_counts().min(),
                       data.groupby("label")["patient_id"].nunique().min()))
if n_folds_bias >= 2:
    cv = StratifiedGroupKFold(n_splits=n_folds_bias, shuffle=True, random_state=SEED)
    shortcut_acc = float(cross_val_score(clf, X, y_bin, groups=grupos_pac,
                                         cv=cv, scoring="accuracy").mean())
    shortcut_auc = float(cross_val_score(clf, X, y_bin, groups=grupos_pac,
                                         cv=cv, scoring="roc_auc").mean())
else:
    shortcut_acc = shortcut_auc = float("nan")

print(f"    accuracy sin mirar el tumor : {shortcut_acc:.3f}")
print(f"    ROC-AUC  sin mirar el tumor : {shortcut_auc:.3f}")
if shortcut_acc >= 0.90:
    veredicto_sesgo = "CRITICO"
    print("    -> La procedencia basta para separar las clases. Cualquier métrica de la\n"
          "       tarea binaria mide, en gran parte, reconocimiento de fuente.")
elif shortcut_acc >= 0.70:
    veredicto_sesgo = "ALTO"
    print("    -> Sesgo de procedencia importante: los resultados binarios están\n"
          "       parcialmente explicados por la fuente.")
else:
    veredicto_sesgo = "MODERADO"
    print("    -> No se aprecia un atajo dominante con estos descriptores.")

# ------------------------------------------------------------------ (4) -----
print("\n(4) Control de dominio dentro de la clase sana (sitios de IXI)")
sanos = data[data["label"] == "notumor"]
if sanos["site"].nunique() > 1:
    print(sanos.groupby("site").agg(imagenes=("local_file", "size"),
                                    sujetos=("patient_id", "nunique")))
    Xs = X[(data["label"] == "notumor").values]
    ys = sanos["site"].astype("category").cat.codes.values
    gs = sanos["patient_id"].values
    n_f = int(min(3, pd.Series(ys).value_counts().min()))
    if n_f >= 2:
        cv_s = StratifiedGroupKFold(n_splits=n_f, shuffle=True, random_state=SEED)
        acc_sitio = float(cross_val_score(clf, Xs, ys, groups=gs, cv=cv_s,
                                          scoring="accuracy").mean())
        print(f"    accuracy identificando el SITIO con descriptores de bajo nivel: "
              f"{acc_sitio:.3f}")
        print("    (muy por encima del azar = incluso dentro de una misma colección hay\n"
              "     firma de escáner; refuerza que la comparación entre colecciones\n"
              "     distintas no puede tomarse como detección de tumor)")
    else:
        acc_sitio = float("nan")
else:
    acc_sitio = float("nan")
    print("    Una sola sede en la clase sana: no se puede hacer control de dominio.")

print("\n    LEAVE-ONE-SOURCE-OUT: NO es aplicable en la tarea binaria.")
print("    Solo hay dos fuentes y cada una aporta una única clase; dejar una fuera")
print("    deja el problema con una sola clase. El control sin confusión")
print("    fuente↔clase se hace en el Apéndice A (3 tipos de tumor dentro de Cheng).")

bias_report = {
    "pureza_clase_por_fuente": round(fuente_predice_clase, 4),
    "shortcut_accuracy": None if np.isnan(shortcut_acc) else round(shortcut_acc, 4),
    "shortcut_auc": None if np.isnan(shortcut_auc) else round(shortcut_auc, 4),
    "accuracy_identificando_sitio_ixi": None if np.isnan(acc_sitio) else round(acc_sitio, 4),
    "veredicto": veredicto_sesgo,
    "leave_one_source_out": "no aplicable (2 fuentes, 1 clase cada una)",
}

### 2.7 Auditoría de sesgo — control multiclase dentro de Cheng

Esta comprobación vivía, en versiones anteriores, dentro del Apéndice A. Se traslada aquí
porque es, conceptualmente, la misma auditoría de sesgo de la sección 2.6 aplicada a la
tarea de control: ¿basta la procedencia (aquí, siempre la misma) para separar las tres
clases? Al ser todo Cheng, un valor cercano al azar confirma que **no** hay atajo de
procedencia disponible para esta tarea.

In [ ]:
# ==============================================================================
# 2.7  AUDITORÍA DE SESGO -- CONTROL MULTICLASE DENTRO DE CHENG
# ==============================================================================
# Reutiliza el clasificador de descriptores de bajo nivel (`clf`) y la matriz de
# descriptores (`X`) ya calculados en 2.6, restringidos a los pacientes de Cheng.
# Como aquí TODAS las imágenes vienen de la misma fuente, un resultado cercano al
# azar confirma que no hay firma de procedencia que explique el resultado de la
# tarea multiclase.
if min(len(ap_tr), len(ap_va), len(ap_te)) == 0 or len(TIPOS) < 2:
    atajo_multiclase = float("nan")
    print("No hay suficientes datos de Cheng en los tres subconjuntos: se omite.")
else:
    Xc = X[(data.source_dataset == "figshare/Cheng2017").values]
    yc = cheng["tumor_type"].map(MAPA_TIPO).values
    patient_ids_cheng = cheng["patient_id"].values
    nf = int(min(3, pd.Series(yc).value_counts().min()))
    if nf >= 2:
        cvc = StratifiedGroupKFold(n_splits=nf, shuffle=True, random_state=SEED)
        atajo_multiclase = float(cross_val_score(clf, Xc, yc, groups=patient_ids_cheng,
                                                 cv=cvc, scoring="accuracy").mean())
        azar_multiclase = float(pd.Series(yc).value_counts(normalize=True).max())
        print(f"Atajo de bajo nivel en la tarea multiclase: {atajo_multiclase:.3f} "
              f"(clase mayoritaria = {azar_multiclase:.3f})")
        print("  Al ser todo la misma fuente, un valor cercano al azar confirma que aquí")
        print("  no hay firma de procedencia que explique el resultado.")
    else:
        atajo_multiclase = float("nan")

bias_report["atajo_bajo_nivel_multiclase"] = (
    None if np.isnan(atajo_multiclase) else round(atajo_multiclase, 4))


In [ ]:
# ==============================================================================
# 1.7  INFORME AUTOMÁTICO DE AUDITORÍA DEL DATASET  +  SANITY CHECK
# ==============================================================================
# Este bloque es el que responde, con una sola ejecución, a la pregunta
# "¿cómo has construido el dataset y cómo sé que no hay fuga?".
#
# Si algo crítico falla, LEVANTA UNA EXCEPCIÓN y el cuaderno se detiene AQUÍ,
# antes de gastar media hora entrenando. En STRICT_PATIENT_MODE, "crítico" incluye
# que falte un solo patient_id real.
# ==============================================================================

def _cuenta(df, col="local_file"):
    return int(len(df))


audit = {
    "modo": {
        "STRICT_PATIENT_MODE": STRICT_PATIENT_MODE,
        "LEGACY_PROXY_MODE": LEGACY_PROXY_MODE,
        "USAR_DATOS_SINTETICOS": USAR_DATOS_SINTETICOS,
        "seed": SEED,
        "img_size": list(IMG_SIZE),
        "canales_cnn": CANALES_CNN,
    },
    "fuentes": {
        s: {
            "url": data.loc[data.source_dataset == s, "source_url"].iat[0],
            "imagenes": int((data.source_dataset == s).sum()),
            "pacientes": int(data.loc[data.source_dataset == s, "patient_id"].nunique()),
            "modalidad": data.loc[data.source_dataset == s, "modality"].iat[0],
            "origen_patient_id": sorted(
                data.loc[data.source_dataset == s, "patient_id_source"].unique()),
        } for s in sorted(data["source_dataset"].unique())
    },
    "imagenes": {
        "iniciales": int(len(manifest)),
        "validas": int(len(data)),
        "duplicados_exactos_eliminados": n_dup_exactos,
        "casi_duplicados_confirmados": len(confirmados),
        "casi_duplicados_entre_fuentes": near_entre_fuentes,
        "casi_duplicados_entre_clases": near_entre_clases,
        "casi_duplicados_entre_pacientes": near_entre_pacientes,
        "conflictos_de_etiqueta": len(conflictos_etiqueta),
        "casi_dup_confirmados_cruzando_subsets": near_confirmados_cruzando,
        "pares_phash_sospechosos_cruzando_subsets": near_candidatos_cruzando,
    },
    "pacientes": {
        "reales": int(data.loc[data.patient_id_source.str.startswith("real"), "patient_id"].nunique()),
        "sin_identificar": int((data["patient_id_source"] == "desconocido").sum()),
        "grupos_de_reparto": int(data["split_group"].nunique()),
        "imagenes_por_paciente_min": int(data.groupby("patient_id").size().min()),
        "imagenes_por_paciente_mediana": float(data.groupby("patient_id").size().median()),
        "imagenes_por_paciente_max": int(data.groupby("patient_id").size().max()),
    },
    "balance": {
        "imagenes_por_clase": data["label"].value_counts().to_dict(),
        "pacientes_por_clase": data.groupby("label")["patient_id"].nunique().to_dict(),
    },
    "split": {
        s: {
            "imagenes": int((data.subset == s).sum()),
            "pacientes": int(data.loc[data.subset == s, "patient_id"].nunique()),
            "tumor": int(((data.subset == s) & (data.label == "tumor")).sum()),
            "notumor": int(((data.subset == s) & (data.label == "notumor")).sum()),
            "pac_tumor": int(data.loc[(data.subset == s) & (data.label == "tumor"), "patient_id"].nunique()),
            "pac_notumor": int(data.loc[(data.subset == s) & (data.label == "notumor"), "patient_id"].nunique()),
            "fuentes": data.loc[data.subset == s, "source_dataset"].value_counts().to_dict(),
        } for s in ("train", "val", "test")
    },
    "solapes": overlaps,
    "sesgo_de_fuente": bias_report,
}

# ------------------------------------------------------------ comprobaciones -
fallos, avisos = [], []

if STRICT_PATIENT_MODE and audit["pacientes"]["sin_identificar"] > 0:
    fallos.append(f"{audit['pacientes']['sin_identificar']} imágenes sin patient_id real "
                  f"(prohibido en STRICT_PATIENT_MODE)")
if any(v > 0 for v in overlaps.values()):
    fallos.append("solape entre subconjuntos: " +
                  ", ".join(f"{k}={v}" for k, v in overlaps.items() if v > 0))
if len(conflictos_etiqueta) > 0:
    fallos.append(f"{len(conflictos_etiqueta)} ficheros idénticos con etiquetas contradictorias")
for s in ("train", "val", "test"):
    if audit["split"][s]["imagenes"] == 0:
        fallos.append(f"el subconjunto {s} está vacío")
    for c in ("tumor", "notumor"):
        if audit["split"][s][c] == 0:
            fallos.append(f"la clase {c} no aparece en {s}")
faltan = [f for f in data["local_file"] if not os.path.exists(f)]
if faltan:
    fallos.append(f"{len(faltan)} ficheros del manifiesto no existen en disco")
if USAR_DATOS_SINTETICOS:
    avisos.append("datos SINTÉTICOS: ningún resultado de esta ejecución es válido")

r = audit["balance"]["imagenes_por_clase"]
desbalance = max(r.values()) / max(min(r.values()), 1)
if desbalance > 1.5:
    avisos.append(f"desbalance de imágenes por clase {desbalance:.2f}:1 "
                  f"(se compensará con class_weight y métricas balanceadas)")
rp = audit["balance"]["pacientes_por_clase"]
desbalance_pac = max(rp.values()) / max(min(rp.values()), 1)
if desbalance_pac > 1.5:
    avisos.append(f"desbalance de PACIENTES por clase {desbalance_pac:.2f}:1")
if bias_report["shortcut_accuracy"] and bias_report["shortcut_accuracy"] >= 0.90:
    avisos.append(f"sesgo de procedencia {bias_report['veredicto']}: un modelo trivial "
                  f"alcanza {bias_report['shortcut_accuracy']:.3f} de accuracy sin ver el tumor")

estado = "FAIL" if fallos else ("WARNING" if avisos else "PASS")
audit["estado"] = estado
audit["fallos"] = fallos
audit["avisos"] = avisos

# --------------------------------------------------------------- informe ----
L = []
A = L.append
A("=" * 78)
A("DATASET AUDIT")
A("=" * 78)
for s, info in audit["fuentes"].items():
    A(f"Source: {s}")
    A(f"    url               : {info['url']}")
    A(f"    modalidad         : {info['modalidad']}")
    A(f"    imágenes          : {info['imagenes']}")
    A(f"    pacientes         : {info['pacientes']}")
    A(f"    origen patient_id : {', '.join(info['origen_patient_id'])}")
A("")
A(f"Patients (reales)         : {audit['pacientes']['reales']}")
A(f"Unknown patient IDs       : {audit['pacientes']['sin_identificar']}")
A(f"Grupos de reparto         : {audit['pacientes']['grupos_de_reparto']}")
A(f"Imágenes/paciente         : min {audit['pacientes']['imagenes_por_paciente_min']} · "
  f"mediana {audit['pacientes']['imagenes_por_paciente_mediana']:.1f} · "
  f"max {audit['pacientes']['imagenes_por_paciente_max']}")
A("")
A(f"Images (iniciales)        : {audit['imagenes']['iniciales']}")
A(f"Images (válidas)          : {audit['imagenes']['validas']}")
A(f"Exact duplicates          : {audit['imagenes']['duplicados_exactos_eliminados']}")
A(f"Near-duplicate clusters   : {audit['imagenes']['casi_duplicados_confirmados']}")
A(f"Cross-source duplicates   : {audit['imagenes']['casi_duplicados_entre_fuentes']}")
A(f"Cross-class near-dups     : {audit['imagenes']['casi_duplicados_entre_clases']}")
A(f"Label conflicts           : {audit['imagenes']['conflictos_de_etiqueta']}")
A(f"Near-dups cruzando subsets: {audit['imagenes']['casi_dup_confirmados_cruzando_subsets']} "
  f"(confirmados) · {audit['imagenes']['pares_phash_sospechosos_cruzando_subsets']} "
  f"(pHash sospechosos, descartados por SSIM)")
A("")
A(f"Imágenes por clase        : {audit['balance']['imagenes_por_clase']}")
A(f"Pacientes por clase       : {audit['balance']['pacientes_por_clase']}")
A("")
for s in ("train", "val", "test"):
    d = audit["split"][s]
    A(f"{s.upper():5s}: pacientes={d['pacientes']:4d}  imágenes={d['imagenes']:5d}  "
      f"tumor={d['tumor']:5d}  notumor={d['notumor']:5d}  "
      f"(pac tumor={d['pac_tumor']}, pac notumor={d['pac_notumor']})")
    A(f"       fuentes: {d['fuentes']}")
A("")
A("Patient overlap:")
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    A(f"    {a}-{b} = {overlaps[f'pacientes {a}-{b}']}")
A("Near-duplicate cluster overlap:")
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    A(f"    {a}-{b} = {overlaps[f'clúster {a}-{b}']}")
A("")
A("Source bias diagnostic:")
A(f"    pureza de clase por fuente     : {bias_report['pureza_clase_por_fuente']}")
A(f"    accuracy de atajo (sin tumor)  : {bias_report['shortcut_accuracy']}")
A(f"    AUC de atajo (sin tumor)       : {bias_report['shortcut_auc']}")
A(f"    veredicto                      : {bias_report['veredicto']}")
A(f"    leave-one-source-out           : {bias_report['leave_one_source_out']}")
A("")
for f in fallos:
    A(f"    [FALLO] {f}")
for w in avisos:
    A(f"    [AVISO] {w}")
A("")
A(f"STATUS: {estado}")
A("=" * 78)
informe_auditoria = "\n".join(L)
print(informe_auditoria)
(ART_DIR / "dataset_audit.txt").write_text(informe_auditoria, encoding="utf-8")

if fallos:
    raise RuntimeError(
        "SANITY CHECK FALLIDO — no se entrena con este dataset:\n  - " +
        "\n  - ".join(fallos))

In [ ]:
# ==============================================================================
# 1.8  MANIFIESTO EXPORTABLE Y HUELLA DIGITAL DEL DATASET Y DEL REPARTO
# ==============================================================================
# Se exporta `manifest_final.csv` con TODA la trazabilidad disponible por imagen.
# Es el fichero que permite a un tercero (o al tribunal) responder a la pregunta
# "¿de dónde salió esta imagen y en qué subconjunto estaba?".
#
# Además se calculan tres huellas:
#   MANIFEST_HASH -> contenido del dataset (fichero + paciente + clase + sha256)
#   SPLIT_HASH    -> a qué subconjunto va cada paciente
#   CONFIG_HASH   -> configuración de entrenamiento (tamaño, canales, semilla...)
#
# Cada modelo entrenado se guarda junto a estas tres huellas. Si el dataset o el
# reparto cambian, los .keras antiguos DEJAN de considerarse reutilizables: si no
# fuera así, podría recargarse un modelo entrenado con otros datos y presentar
# sus métricas como si fueran actuales.
# =============================================================================

data_export = data[COLUMNAS[:-1] + ["split_group", "subset"]].copy()
data_export.to_csv(MANIFEST_PATH, index=False)
print(f"Manifiesto final escrito en: {MANIFEST_PATH}  ({len(data_export)} filas)")
print("Columnas:", list(data_export.columns))

# Plantilla vacía, para incorporar en el futuro fuentes que hoy no tienen PID.
pd.DataFrame(columns=COLUMNAS).to_csv(TEMPLATE_PATH, index=False)
print(f"Plantilla de manifiesto para fuentes nuevas: {TEMPLATE_PATH}")


def _hash_texto(txt):
    return hashlib.sha256(txt.encode("utf-8")).hexdigest()[:16]


MANIFEST_HASH = _hash_texto(
    data.sort_values("local_file")[["local_file", "patient_id", "label", "sha256"]]
        .to_csv(index=False))
SPLIT_HASH = _hash_texto(
    data.sort_values("patient_id")[["patient_id", "subset"]]
        .drop_duplicates().to_csv(index=False))
CONFIG_HASH = _hash_texto(json.dumps({
    "img_size": IMG_SIZE, "canales": CANALES_CNN, "batch": BATCH_SIZE,
    "seed": SEED, "png_size": PNG_SIZE, "lr": LR,
    "strict": STRICT_PATIENT_MODE, "sinteticos": USAR_DATOS_SINTETICOS,
}, sort_keys=True))

FINGERPRINT = {
    "manifest_hash": MANIFEST_HASH,
    "split_hash": SPLIT_HASH,
    "config_hash": CONFIG_HASH,
    "n_imagenes": int(len(data)),
    "n_pacientes": int(data["patient_id"].nunique()),
    "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
}
audit["fingerprint"] = FINGERPRINT
(ART_DIR / "dataset_audit.json").write_text(
    json.dumps(audit, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print("\nHuella digital de esta ejecución:")
for k, v in FINGERPRINT.items():
    print(f"   {k:14s}: {v}")
print(f"\nAuditoría completa en JSON: {ART_DIR/'dataset_audit.json'}")
print(f"Informe legible          : {ART_DIR/'dataset_audit.txt'}")
print(f"Duplicados               : {ART_DIR/'duplicados_report.csv'}")

# Descarga directa del manifiesto (Colab).
try:
    from google.colab import files as colab_files
    print("\nPara descargar el manifiesto ejecuta:  "
          "colab_files.download(str(MANIFEST_PATH))")
except ImportError:
    pass

## 3. Configuración explícita de tarea (`TASK_CONFIG`)

A partir de aquí, el comportamiento binario/multiclase de cada función se decide
**explícitamente** a través de `task_config`, nunca infiriéndolo de la forma de un array
ni de ninguna otra pista implícita. `activar_tarea(...)` fija la tarea activa y crea (o
reutiliza) su árbol de resultados en `results/<tarea>/{models,predictions,metrics,plots,
gradcam,logs}/`, para no mezclar artefactos de una tarea con los de la otra.

La tarea activa/principal de este cuaderno es **`tumor_type`**. La tarea `binary` se
reactiva explícitamente más adelante, en la sección secundaria del experimento previo.

In [ ]:
# ==============================================================================
# 3. TASK_CONFIG -- configuración explícita de tarea (binaria / multiclase)
# ==============================================================================
TASK_CONFIGS = {
    "binary": {
        "name": "binary",
        "label_col": "label",
        "classes": ["notumor", "tumor"],
        "class_map": {"notumor": 0, "tumor": 1},
        "n_classes": 2,
        "activation_final": "sigmoid",
        "loss": "binary_crossentropy",
        "monitor_metric": "val_auc",
        "monitor_mode": "max",
        "results_subdir": "binary",
    },
    "tumor_type": {
        "name": "tumor_type",
        "label_col": "tumor_type",
        "classes": ["glioma", "meningioma", "pituitario"],
        "class_map": {"glioma": 0, "meningioma": 1, "pituitario": 2},
        "n_classes": 3,
        "activation_final": "softmax",
        "loss": "sparse_categorical_crossentropy",
        "monitor_metric": "val_accuracy",
        "monitor_mode": "max",
        "results_subdir": "tumor_type",
    },
}


def activar_tarea(nombre_tarea):
    """Fija la tarea activa y (re)crea su árbol de resultados. Todas las
    funciones de métricas/arquitectura/checkpointing leen la tarea activa a
    través de TASK / TASK_CONFIG (variables globales que esta función
    actualiza) o reciben task_config explícitamente como argumento -- en
    ningún caso se infiere la tarea a partir de los datos."""
    global TASK, TASK_CONFIG, TASK_RESULTS_DIR, TASK_MODELS_DIR, TASK_PREDICTIONS_DIR
    global TASK_METRICS_DIR, TASK_PLOTS_DIR, TASK_GRADCAM_DIR, TASK_LOGS_DIR
    global EXPERIMENT_SUMMARY_CSV
    TASK = nombre_tarea
    TASK_CONFIG = TASK_CONFIGS[TASK]
    TASK_RESULTS_DIR = RESULTS_DIR / TASK_CONFIG["results_subdir"]
    TASK_MODELS_DIR = TASK_RESULTS_DIR / "models"
    TASK_PREDICTIONS_DIR = TASK_RESULTS_DIR / "predictions"
    TASK_METRICS_DIR = TASK_RESULTS_DIR / "metrics"
    TASK_PLOTS_DIR = TASK_RESULTS_DIR / "plots"
    TASK_GRADCAM_DIR = TASK_RESULTS_DIR / "gradcam"
    TASK_LOGS_DIR = TASK_RESULTS_DIR / "logs"
    for _d in (TASK_MODELS_DIR, TASK_PREDICTIONS_DIR, TASK_METRICS_DIR,
               TASK_PLOTS_DIR, TASK_GRADCAM_DIR, TASK_LOGS_DIR):
        _d.mkdir(parents=True, exist_ok=True)
    EXPERIMENT_SUMMARY_CSV = TASK_RESULTS_DIR / "experiment_summary.csv"
    print(f"Tarea activa: {TASK}  ({TASK_CONFIG['n_classes']} clases: {TASK_CONFIG['classes']})")
    print(f"  Resultados en: {TASK_RESULTS_DIR}")
    return TASK_CONFIG


activar_tarea("tumor_type")   # tarea principal de este cuaderno


## 4. Pipeline único de datos (`tf.data`)

Los generadores ya no leen carpetas: leen el **manifiesto**. Ese cambio es lo que permite
saber a qué paciente pertenece cada predicción y, por tanto, calcular métricas a nivel de
paciente en la sección 5.

Tres reglas que se respetan en todo el cuaderno:

1. **El *data augmentation* se aplica solo a entrenamiento.** Validación y test se leen tal
   cual. Tampoco se hace sobremuestreo en validación ni en test.
2. **El reparto ya está hecho** (sección 1.5) antes de cualquier aumento, así que ninguna
   versión aumentada de una imagen puede aparecer en otro subconjunto.
3. **Todos los modelos ven las mismas imágenes y las mismas etiquetas.** Lo único que
   cambia es la adaptación de entrada que cada arquitectura exige: 1 canal en `[0,1]` para
   la CNN propia, 3 canales replicados más el `preprocess_input` correspondiente para los
   preentrenados. La comparación es justa porque la información clínica de partida es la
   misma; lo que cambia es el formato que cada red necesita.

El desequilibrio de clases se compensa con `class_weight` durante el entrenamiento y se
reporta siempre con métricas balanceadas, nunca solo con *accuracy*.

In [ ]:
# ==============================================================================
# 2.1  PIPELINE ÚNICO DE DATOS (tf.data) PARA TODOS LOS MODELOS
# ==============================================================================
# Aquí no se copian las imágenes a carpetas train/val/test ni se usa flow_from_directory.
# directamente del MANIFIESTO, que es la única fuente de verdad.
#
# Motivos:
#   * flow_from_directory no permite saber a qué PACIENTE pertenece cada
#     predicción, y sin eso no hay métricas a nivel de paciente (sección 5);
#   * copiar ficheros obligaba a renombrar colisiones con sufijos aleatorios, lo
#     que rompía la trazabilidad;
#   * con el manifiesto, el orden de las predicciones coincide fila a fila con el
#     DataFrame, así que la agregación por paciente es exacta.
#
# TODOS los modelos reciben EXACTAMENTE las mismas imágenes y las mismas
# etiquetas. Lo único que cambia es la adaptación de entrada que cada
# arquitectura exige:
#     CNN propia    -> 1 canal en [0,1]                (las RM son escala de grises)
#     preentrenados -> 3 canales replicados + su preprocess_input específico
# La comparación sigue siendo justa: la información clínica de entrada es idéntica.
# ==============================================================================
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(SEED)
if DETERMINISMO_ESTRICTO:
    try:
        tf.config.experimental.enable_op_determinism()
        print("Determinismo estricto de TensorFlow ACTIVADO (puede fallar en GPU).")
    except Exception as e:
        print("No se pudo activar el determinismo estricto:", e)
else:
    print("Semillas fijadas (Python, NumPy, TF, Keras). Determinismo estricto desactivado:")
    print("  ver la nota de la celda 1.1 sobre por qué en GPU puede romper el entrenamiento.")

AUTOTUNE = tf.data.AUTOTUNE
CLASES = ["notumor", "tumor"]           # índice 0 = notumor, 1 = tumor
CLASS_INDEX = {c: i for i, c in enumerate(CLASES)}
print("Mapa de clases:", CLASS_INDEX)

from tensorflow.keras.applications import (vgg16 as _vgg16, resnet50 as _resnet50,
                                           mobilenet_v2 as _mnv2, efficientnet as _effnet)

# nombre -> (constructor, preprocess_input, nº de canales)
ADAPTADOR_ENTRADA = {
    "cnn_scratch":    (None,                    None,                        CANALES_CNN),
    "vgg16":          (keras.applications.VGG16,          _vgg16.preprocess_input,   3),
    "resnet50":       (keras.applications.ResNet50,       _resnet50.preprocess_input, 3),
    "mobilenetv2":    (keras.applications.MobileNetV2,    _mnv2.preprocess_input,    3),
    "efficientnetb0": (keras.applications.EfficientNetB0, _effnet.preprocess_input,  3),
}

augmentador = keras.Sequential([
    keras.layers.RandomFlip("horizontal", seed=SEED),
    keras.layers.RandomRotation(0.03, fill_mode="nearest", seed=SEED),
    keras.layers.RandomZoom(0.10, fill_mode="nearest", seed=SEED),
    keras.layers.RandomTranslation(0.05, 0.05, fill_mode="nearest", seed=SEED),
    keras.layers.RandomContrast(0.10, seed=SEED),
], name="augmentation")


def _cargar(ruta):
    """PNG -> tensor uint8 (H, W, 1) al tamaño de entrada."""
    img = tf.io.decode_png(tf.io.read_file(ruta), channels=1)
    img = tf.image.resize(img, IMG_SIZE, method="bilinear")
    return tf.cast(img, tf.uint8)


def hacer_ds(df, arquitectura, task_config, entrenamiento=False,
             batch=None, cache_key=None):
    """Dataset alineado FILA A FILA con `df` cuando entrenamiento=False.

    La columna de etiqueta y el mapa clase->índice vienen EXPLÍCITAMENTE de
    `task_config` (TASK_CONFIGS["binary"] o TASK_CONFIGS["tumor_type"]);
    nunca se infieren del contenido de `df`.

    El augmentation se aplica SOLO si entrenamiento=True: validación y test
    nunca se aumentan ni se sobremuestrean.
    """
    _, preprocess, canales = ADAPTADOR_ENTRADA[arquitectura]
    batch = batch or BATCH_SIZE
    columna = task_config["label_col"]
    mapa = task_config["class_map"]
    rutas = df["local_file"].astype(str).tolist()
    etiquetas = df[columna].map(mapa).astype("float32").tolist()

    @tf.autograph.experimental.do_not_convert
    def _leer(p, y):
        return _cargar(p), y

    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    ds = ds.map(_leer, num_parallel_calls=AUTOTUNE)
    if cache_key:
        ds = ds.cache(str(ART_DIR / f"cache_{cache_key}"))
    if entrenamiento:
        ds = ds.shuffle(min(len(df), 4096), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch)

    @tf.autograph.experimental.do_not_convert
    def _adaptar(x, y):
        x = tf.cast(x, tf.float32)                      # [0,255], 1 canal
        if entrenamiento:
            x = augmentador(x, training=True)
            x = tf.clip_by_value(x, 0.0, 255.0)
        if canales == 3:
            x = tf.repeat(x, 3, axis=-1)
            x = preprocess(x)                           # cada red, el suyo
        else:
            x = x / 255.0
        return x, y

    ds = ds.map(_adaptar, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


# -------------------------------------------------- particiones del manifiesto
df_train = data[data.subset == "train"].reset_index(drop=True)
df_val   = data[data.subset == "val"].reset_index(drop=True)
df_test  = data[data.subset == "test"].reset_index(drop=True)

for nombre, d in (("train", df_train), ("val", df_val), ("test", df_test)):
    print(f"{nombre:5s}: {len(d):5d} imágenes | {d['patient_id'].nunique():4d} pacientes | "
          f"tumor={int((d.label=='tumor').sum())} notumor={int((d.label=='notumor').sum())}")

# ------------------------------------------------------------- class weights -
n_pos = int((df_train.label == "tumor").sum())
n_neg = int((df_train.label == "notumor").sum())
CLASS_WEIGHT = {0: (n_pos + n_neg) / (2.0 * max(n_neg, 1)),
                1: (n_pos + n_neg) / (2.0 * max(n_pos, 1))}
print(f"\nclass_weight (solo en entrenamiento): {CLASS_WEIGHT}")
print("No se hace oversampling en ningún subconjunto; validación y test se dejan")
print("exactamente como están para que las métricas reflejen la distribución real.")

# comprobación de formas
_x, _y = next(iter(hacer_ds(df_val, "cnn_scratch", TASK_CONFIGS["binary"])))
print(f"\nLote CNN  -> x{tuple(_x.shape)} rango [{float(tf.reduce_min(_x)):.2f}, "
      f"{float(tf.reduce_max(_x)):.2f}]  y{tuple(_y.shape)}")
_x3, _ = next(iter(hacer_ds(df_val, "vgg16", TASK_CONFIGS["binary"])))
print(f"Lote VGG16-> x{tuple(_x3.shape)} rango [{float(tf.reduce_min(_x3)):.2f}, "
      f"{float(tf.reduce_max(_x3)):.2f}]")

# df_trainval (train+val de la tarea binaria) se usa más adelante en la
# validación cruzada agrupada de la sección secundaria (tarea binaria).
df_trainval = pd.concat([df_train, df_val], ignore_index=True)


In [ ]:
# ------------------------------------------------------------------------------
# 2.2  EFECTO DEL DATA AUGMENTATION  (solo sobre entrenamiento)
# ------------------------------------------------------------------------------
# Se muestra qué transformaciones ve realmente el modelo. Solo se usan
# transformaciones anatómicamente razonables: no hay volteo vertical (una RM
# cerebral boca abajo no es un caso plausible) y las rotaciones son pequeñas.
# ------------------------------------------------------------------------------
import matplotlib.pyplot as plt

fila = df_train.iloc[0]
with Image.open(fila["local_file"]) as im:
    base = np.asarray(im.convert("L").resize(IMG_SIZE[::-1]), dtype=np.float32)

fig, ax = plt.subplots(1, 6, figsize=(16, 3))
ax[0].imshow(base, cmap="gray"); ax[0].set_title("original", fontsize=9); ax[0].axis("off")
lote = tf.convert_to_tensor(np.repeat(base[None, ..., None], 5, axis=0))
aug = augmentador(lote, training=True).numpy()
for i in range(5):
    ax[i + 1].imshow(np.clip(aug[i, ..., 0], 0, 255), cmap="gray")
    ax[i + 1].set_title(f"aug {i+1}", fontsize=9); ax[i + 1].axis("off")
fig.suptitle(f"Augmentation — {fila['patient_id']} ({fila['label']}) · solo en train",
             fontsize=11)
plt.tight_layout(); plt.show()

## 5. Métricas y agregación slice → paciente

Estas funciones son compartidas por las dos tareas. El comportamiento binario/multiclase
se decide **explícitamente** con el parámetro `task_config` que recibe cada función —
nunca observando si `y_prob` es un vector o una matriz.

- **Binario**: sin cambios de comportamiento respecto a versiones anteriores (mismas
  fórmulas, mismas claves de salida).
- **Multiclase**: `balanced_accuracy_score` de scikit-learn (media del recall por clase —
  fórmula distinta de la binaria, aunque el nombre de la métrica sea el mismo), macro-F1,
  recall/precisión/F1 por clase, matriz de confusión. La agregación paciente implementa
  exactamente `P_paciente(c) = media de P_corte(c)` → `pred_paciente = argmax_c
  P_paciente(c)`; cualquier otra estrategia de agregación para multiclase debe decidirse y
  congelarse con validación antes de usarse en test (no hay ninguna otra implementada
  todavía).

In [ ]:
# ==============================================================================
# 5. MÉTRICAS Y PIPELINE ÚNICO DE EVALUACIÓN (binario + multiclase, explícito)
# ==============================================================================
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix,
                             balanced_accuracy_score, classification_report)

AGREGACIONES = {
    "media":      lambda p: float(np.mean(p)),
    "maximo":     lambda p: float(np.max(p)),
    "mediana":    lambda p: float(np.median(p)),
    "top3_media": lambda p: float(np.mean(np.sort(p)[-3:])),
}


def predecir(modelo, df, arquitectura, task_config):
    """Probabilidades alineadas fila a fila con df.
    Binario    -> vector 1D con P(clase positiva).
    Multiclase -> matriz (n_filas, n_clases) con el softmax completo.
    """
    ds = hacer_ds(df, arquitectura, task_config, entrenamiento=False)
    p = modelo.predict(ds, verbose=0)
    if task_config["n_classes"] == 2:
        p = p.reshape(-1)
    assert len(p) == len(df), f"desalineación: {len(p)} predicciones para {len(df)} filas"
    return p


def _seguro(num, den):
    return float(num) / float(den) if den else float("nan")


def metricas(y_true, y_prob, task_config, umbral=None):
    """Métricas de clasificación. `task_config` decide EXPLÍCITAMENTE si se
    calculan como binario o como multiclase."""
    y_true = np.asarray(y_true)
    if task_config["n_classes"] == 2:
        y_true = y_true.astype(int)
        y_prob = np.asarray(y_prob, dtype=float)
        umbral = 0.5 if umbral is None else umbral
        y_pred = (y_prob >= umbral).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        sens = _seguro(tp, tp + fn); spec = _seguro(tn, tn + fp)
        prec = _seguro(tp, tp + fp); npv = _seguro(tn, tn + fn)
        f1 = _seguro(2 * prec * sens, prec + sens) if (prec + sens) else float("nan")
        dos_clases = len(np.unique(y_true)) == 2
        return {
            "task": "binary", "n_classes": 2, "n": int(len(y_true)),
            "accuracy": _seguro(tp + tn, len(y_true)),
            "balanced_accuracy": float(np.nanmean([sens, spec])),
            "roc_auc": float(roc_auc_score(y_true, y_prob)) if dos_clases else float("nan"),
            "pr_auc": float(average_precision_score(y_true, y_prob)) if dos_clases else float("nan"),
            "sensitivity": sens, "specificity": spec,
            "precision": prec, "npv": npv, "f1": f1,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "umbral": float(umbral),
        }

    # ---- multiclase: y_prob es (n_filas, n_clases) ----
    y_prob = np.asarray(y_prob, dtype=float)
    n_classes = task_config["n_classes"]
    clases = task_config["classes"]
    y_pred = y_prob.argmax(axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    reporte = classification_report(y_true, y_pred, labels=list(range(n_classes)),
                                    target_names=clases, output_dict=True, zero_division=0)
    recall_por_clase = {c: _seguro(cm[i, i], cm[i, :].sum()) for i, c in enumerate(clases)}
    return {
        "task": task_config["name"], "n_classes": n_classes, "n": int(len(y_true)),
        "accuracy": _seguro(int((y_pred == y_true).sum()), len(y_true)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(reporte["macro avg"]["f1-score"]),
        "recall_por_clase": recall_por_clase,
        "precision_por_clase": {c: float(reporte[c]["precision"]) for c in clases},
        "f1_por_clase": {c: float(reporte[c]["f1-score"]) for c in clases},
        "matriz_confusion": cm.tolist(),
        "clases": clases,
        "umbral": None,
    }


def por_paciente(df, y_prob, task_config, agregacion="media"):
    """Agrega probabilidades de CORTE a PACIENTE. task_config decide
    EXPLÍCITAMENTE el comportamiento binario/multiclase.

    Binario    -> columnas: patient_id, y, source, n_cortes, prob (idéntico al
                  comportamiento histórico).
    Multiclase -> columnas: patient_id, p_<clase> (media de probabilidad por
                  clase), pred_idx/pred (argmax), y (índice real), n_cortes.
                  Implementa EXACTAMENTE: P_paciente(c) = media de P_corte(c);
                  pred_paciente = argmax_c P_paciente(c).
    """
    if task_config["n_classes"] == 2:
        fn = AGREGACIONES[agregacion]
        tmp = df[["patient_id", task_config["label_col"], "source_dataset"]].copy()
        tmp = tmp.rename(columns={task_config["label_col"]: "label"})
        tmp["prob"] = np.asarray(y_prob, dtype=float)
        tmp["y"] = tmp["label"].map(task_config["class_map"]).astype(int)
        chequeo = tmp.groupby("patient_id")["y"].nunique()
        assert (chequeo == 1).all(), "un paciente tiene cortes con etiquetas distintas"
        return (tmp.groupby("patient_id")
                   .agg(y=("y", "first"), source=("source_dataset", "first"),
                        n_cortes=("prob", "size"), prob=("prob", lambda s: fn(s.values)))
                   .reset_index())

    if agregacion != "media":
        raise NotImplementedError(
            f"Agregación '{agregacion}' no implementada para multiclase. Solo "
            "'media' (mean probability + argmax) está disponible; cualquier otra "
            "estrategia debe decidirse y congelarse con validación antes de "
            "usarse en test.")
    clases = task_config["classes"]
    y_prob = np.asarray(y_prob, dtype=float)
    tmp = df[["patient_id", task_config["label_col"]]].copy()
    tmp = tmp.rename(columns={task_config["label_col"]: "clase_real"})
    for i, c in enumerate(clases):
        tmp[f"p_{c}"] = y_prob[:, i]
    chequeo = tmp.groupby("patient_id")["clase_real"].nunique()
    assert (chequeo == 1).all(), "un paciente tiene cortes con etiquetas de clase distintas"
    agr = (tmp.groupby("patient_id")
              .agg(**{f"p_{c}": (f"p_{c}", "mean") for c in clases},
                   clase_real=("clase_real", "first"),
                   n_cortes=(f"p_{clases[0]}", "size"))
              .reset_index())
    prob_cols = [f"p_{c}" for c in clases]
    agr["pred_idx"] = agr[prob_cols].values.argmax(axis=1)
    agr["pred"] = agr["pred_idx"].map(dict(enumerate(clases)))
    agr["y"] = agr["clase_real"].map(task_config["class_map"]).astype(int)
    return agr


def umbral_desde_validacion(y_true, y_prob, criterio="youden"):
    """Umbral elegido SOLO con validación (tarea BINARIA únicamente; multiclase
    decide por argmax y no usa umbral). Nunca se mira test para esto.

    `roc_curve` devuelve `inf` como primer umbral (el punto que clasifica todo
    como negativo). Si se toma tal cual, el modelo predice siempre 'notumor' y
    la especificidad sale 1.0 con sensibilidad 0. Se descarta ese punto y el
    resultado se acota al rango de probabilidades observado.
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    if len(np.unique(y_true)) < 2:
        return 0.5
    if criterio == "f1":
        prec, rec, t2 = precision_recall_curve(y_true, y_prob)
        f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
        candidato = float(t2[int(np.argmax(f1[:-1]))]) if len(t2) else 0.5
    elif criterio == "youden":
        fpr, tpr, thr = roc_curve(y_true, y_prob)
        finitos = np.isfinite(thr)
        if not finitos.any():
            return 0.5
        fpr, tpr, thr = fpr[finitos], tpr[finitos], thr[finitos]
        candidato = float(thr[int(np.argmax(tpr - fpr))])
    else:
        candidato = 0.5
    if not np.isfinite(candidato):
        return 0.5
    return float(np.clip(candidato, float(y_prob.min()), float(y_prob.max())))


def bootstrap_ic(pac_df, task_config, metric_fn, n_boot=2000, alfa=0.05, seed=SEED):
    """IC del percentil por bootstrap DE PACIENTES (cluster bootstrap), agnóstico
    de métrica: `metric_fn(df_boot) -> dict` calcula lo que se quiera medir sobre
    una muestra bootstrap de pacientes. No remuestrea cortes sueltos, porque los
    cortes de un mismo paciente no son independientes."""
    rng = np.random.RandomState(seed)
    n = len(pac_df)
    acumulado = {}
    validos = 0
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        boot_df = pac_df.iloc[idx]
        if boot_df["y"].nunique() < 2 and task_config["n_classes"] == 2:
            continue
        m = metric_fn(boot_df)
        validos += 1
        for k, v in m.items():
            if isinstance(v, (int, float)) and not isinstance(v, bool) and not np.isnan(v):
                acumulado.setdefault(k, []).append(v)
    ic = {}
    for k, vals in acumulado.items():
        vals = np.asarray(vals, dtype=float)
        ic[k] = (float(np.percentile(vals, 100 * alfa / 2)),
                 float(np.percentile(vals, 100 * (1 - alfa / 2)))) if len(vals) else (float("nan"),) * 2
    ic["n_boot_validos"] = int(validos)
    return ic


def bloque_metricas(titulo, m, task_config, ic=None):
    print(f"\n{titulo}")
    print("-" * len(titulo))
    if task_config["n_classes"] == 2:
        orden = [("accuracy", "Accuracy"), ("balanced_accuracy", "Balanced accuracy"),
                 ("roc_auc", "ROC-AUC"), ("pr_auc", "PR-AUC"),
                 ("sensitivity", "Sensibilidad (recall)"), ("specificity", "Especificidad"),
                 ("precision", "Precisión (PPV)"), ("npv", "NPV"), ("f1", "F1")]
        for k, etiqueta in orden:
            linea = f"  {etiqueta:22s}: {m[k]:.4f}"
            if ic and k in ic:
                linea += f"   IC95% [{ic[k][0]:.4f}, {ic[k][1]:.4f}]"
            print(linea)
        print(f"  {'Umbral':22s}: {m['umbral']:.4f}   (n={m['n']})")
        print(f"  Matriz de confusión    : TN={m['tn']} FP={m['fp']} FN={m['fn']} TP={m['tp']}")
        return
    print(f"  Accuracy               : {m['accuracy']:.4f}")
    linea_ba = f"  Balanced accuracy      : {m['balanced_accuracy']:.4f}"
    if ic and "balanced_accuracy" in ic:
        linea_ba += f"   IC95% [{ic['balanced_accuracy'][0]:.4f}, {ic['balanced_accuracy'][1]:.4f}]"
    print(linea_ba)
    print(f"  Macro-F1               : {m['macro_f1']:.4f}   (n={m['n']})")
    for c in task_config["classes"]:
        print(f"  Recall {c:12s}     : {m['recall_por_clase'][c]:.4f}   "
              f"(F1={m['f1_por_clase'][c]:.4f})")
    print(f"  Matriz de confusión (filas=real, columnas=predicho, orden={task_config['classes']}):")
    for fila in m["matriz_confusion"]:
        print("   ", fila)


print("Funciones de evaluación cargadas (task_config explícito en todas):")
print("  predecir · metricas · por_paciente · umbral_desde_validacion (binaria) · bootstrap_ic")
print("Reglas de agregación por paciente (binaria) disponibles:", list(AGREGACIONES))


## 6. Arquitecturas y entrenamiento (binario / multiclase por configuración)

La entrada de la CNN propia es de **un solo canal**. Las resonancias son en escala de
grises: replicar el canal tres veces no aporta información, solo triplica un dato
redundante y multiplica los parámetros de la primera convolución. Los modelos
preentrenados sí necesitan tres canales porque fueron entrenados en ImageNet, y los
reciben con su `preprocess_input` específico.

Esto no rompe la equidad de la comparación: todas las arquitecturas parten del mismo corte,
con el mismo recorte y la misma normalización; lo único distinto es la representación de
entrada que cada una exige. Todo el resto del cuaderno —Grad-CAM, inferencia y
visualizaciones— soporta ya la CNN de un canal.

Dos detalles heredados que se corrigen aquí:

- **Los nombres de capa ya no se escriben a mano en ningún sitio.** Grad-CAM localiza la
  última convolución de forma dinámica, así que cambiar la arquitectura no rompe la sección 6.
- **Un modelo guardado ya no se reutiliza a ciegas.** Cada `.keras` lleva al lado la huella
  del manifiesto, del reparto y de la configuración. Si alguna cambia, el artefacto se
  descarta y se reentrena. La versión anterior recargaba cualquier `.h5` que encontrara,
  aunque perteneciera a otro dataset.

En esta sección **no aparece el conjunto de test**. Todo lo que se decide aquí se decide
mirando validación.

A partir de aquí, `construir_backbone`, `construir` y `entrenar_o_cargar` reciben
`task_config` explícitamente: la capa de salida (1 sigmoide o *k* softmax), la
función de pérdida y la métrica que vigila el *early stopping* dependen de esa
configuración, nunca de inspeccionar los datos.

In [ ]:
# la CNN propia entra en 1 canal (así son las RM); las preentrenadas piden 3, así que les replico el canal.
# Grad-CAM (sección 12) localiza la última conv por nombre, dinámicamente -- no hardcodeo ningún 'conv2d_3'.
from tensorflow.keras import layers, models

NOMBRES_MODELOS = ["cnn_scratch", "vgg16", "resnet50", "mobilenetv2", "efficientnetb0"]
ETIQUETA_MODELO = {
    "cnn_scratch": "CNN propia (desde cero, 1 canal)",
    "vgg16": "VGG16 (transfer, base congelada)",
    "resnet50": "ResNet50 (transfer, base congelada)",
    "mobilenetv2": "MobileNetV2 (transfer, base congelada)",
    "efficientnetb0": "EfficientNetB0 (transfer, base congelada)",
}


def construir_cnn_scratch(n_salidas=1, activacion="sigmoid", nombre="cnn_scratch"):
    """CNN propia. Con n_salidas=1 y sigmoide es el clasificador binario; con
    n_salidas=k y softmax, la misma arquitectura sirve para la tarea multiclase."""
    m = models.Sequential(name=nombre)
    m.add(layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], CANALES_CNN)))
    for i, filtros in enumerate([32, 64, 128, 128]):
        m.add(layers.Conv2D(filtros, 3, padding="same", use_bias=False,
                             name=f"bloque{i+1}_conv"))
        m.add(layers.BatchNormalization(name=f"bloque{i+1}_bn"))
        m.add(layers.Activation("relu", name=f"bloque{i+1}_relu"))
        m.add(layers.MaxPooling2D(2, name=f"bloque{i+1}_pool"))
    m.add(layers.Conv2D(256, 3, padding="same", activation="relu", name="conv_final"))
    m.add(layers.GlobalAveragePooling2D(name="gap"))
    m.add(layers.Dropout(0.4, name="dropout"))
    m.add(layers.Dense(128, activation="relu", name="densa"))
    m.add(layers.Dropout(0.3, name="dropout2"))
    m.add(layers.Dense(n_salidas, activation=activacion, name="salida"))
    return m


def construir_backbone(nombre, task_config):
    constructor, _, _ = ADAPTADOR_ENTRADA[nombre]

    def _descargar_pesos_imagenet():
        # esto se me quedaba colgado descargando los pesos de ImageNet sin dar
        # ningún error -- literal 29 minutos con VGG16, hasta que lo maté a mano
        # con Ctrl+C. Hacía falta ponerle un timeout explícito al socket para que
        # eso salte como excepción y así _con_reintentos pueda reintentarlo con
        # backoff, igual que ya se hace con las descargas de Cheng/IXI en 2.1.
        import socket as _socket
        _previo = _socket.getdefaulttimeout()
        _socket.setdefaulttimeout(90.0)
        try:
            return constructor(weights="imagenet", include_top=False,
                                input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
        finally:
            _socket.setdefaulttimeout(_previo)

    base = _con_reintentos(_descargar_pesos_imagenet, intentos=5, espera_base=5.0,
                            espera_max=60.0, etiqueta=f"pesos ImageNet de {nombre}")
    base.trainable = False
    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = layers.Dropout(0.4, name="dropout")(x)
    x = layers.Dense(128, activation="relu", name="densa")(x)
    x = layers.Dropout(0.3, name="dropout2")(x)
    n_salidas = 1 if task_config["n_classes"] == 2 else task_config["n_classes"]
    salida = layers.Dense(n_salidas, activation=task_config["activation_final"],
                          name="salida")(x)
    return models.Model(base.input, salida, name=f"{nombre}_{task_config['name']}")


def construir(nombre, task_config):
    if nombre == "cnn_scratch":
        n_salidas = 1 if task_config["n_classes"] == 2 else task_config["n_classes"]
        m = construir_cnn_scratch(n_salidas=n_salidas,
                                  activacion=task_config["activation_final"],
                                  nombre=f"cnn_scratch_{task_config['name']}")
    else:
        m = construir_backbone(nombre, task_config)

    if task_config["n_classes"] == 2:
        metricas_keras = [keras.metrics.AUC(name="auc"),
                          keras.metrics.BinaryAccuracy(name="accuracy")]
    else:
        metricas_keras = [keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

    m.compile(optimizer=keras.optimizers.Adam(LR),
              loss=task_config["loss"], metrics=metricas_keras)
    return m

# 3.2 -- solo reentreno si hace falta: si el .keras que hay coincide en manifest/split/config, lo reutilizo.
# Cada tarea (binaria/multiclase) valida su propio split_hash -- así no se puede
# reutilizar por error un modelo entrenado para una tarea al evaluar la otra.


def _rutas(nombre, task_config, sufijo=""):
    base = TASK_MODELS_DIR / f"{task_config['name']}_{nombre}{sufijo}"
    return base.with_suffix(".keras"), base.with_suffix(".json")


def _meta_valida(meta, task_config, usar_reduce_lr=True):
    split_hash_esperado = (SPLIT_HASH if task_config["n_classes"] == 2
                           else SPLIT_HASH_MULTICLASE_CONGELADO)
    return (meta.get("manifest_hash") == MANIFEST_HASH
            and meta.get("config_hash") == CONFIG_HASH
            and meta.get("split_hash") == split_hash_esperado
            and meta.get("usar_reduce_lr", True) == usar_reduce_lr)


def entrenar_o_cargar(nombre, df_tr, df_va, task_config, epochs, sufijo="",
                      forzar=False, verbose=1, usar_reduce_lr=True):
    """`usar_reduce_lr=False` reproduce EXACTAMENTE la configuración histórica del
    baseline `cnn_scratch` multiclase (Adam lr=1e-4, EarlyStopping sobre
    val_accuracy, patience=8, restore_best_weights=True, máx. EPOCHS_CNN épocas,
    SIN ReduceLROnPlateau). El resto de modelos (benchmarks nuevos, tarea binaria)
    siguen usando `usar_reduce_lr=True` por defecto -- este parámetro NO cambia su
    comportamiento. Se guarda en el meta del checkpoint y forma parte de la
    comprobación de reutilización: un checkpoint entrenado con un valor de
    `usar_reduce_lr` distinto nunca se reutiliza por error para el otro caso."""
    ruta_modelo, ruta_meta = _rutas(nombre, task_config, sufijo)
    if ruta_modelo.exists() and ruta_meta.exists() and not forzar:
        meta = json.loads(ruta_meta.read_text())
        if _meta_valida(meta, task_config, usar_reduce_lr):
            print(f"  [reutilizado] {task_config['name']}_{nombre}{sufijo}: "
                  "artefacto compatible con este split")
            return keras.models.load_model(ruta_modelo), meta
        print(f"  [descartado]  {task_config['name']}_{nombre}{sufijo}: el artefacto "
              "guardado corresponde a otro dataset/split/configuración de entrenamiento "
              "-> se reentrena")

    keras.utils.set_random_seed(SEED)
    modelo = construir(nombre, task_config)
    ds_tr = hacer_ds(df_tr, nombre, task_config, entrenamiento=True)
    ds_va = hacer_ds(df_va, nombre, task_config, entrenamiento=False)
    cw = CLASS_WEIGHT if task_config["n_classes"] == 2 else None
    cbs = [
        keras.callbacks.EarlyStopping(monitor=task_config["monitor_metric"],
                                      mode=task_config["monitor_mode"],
                                      patience=PACIENCIA, restore_best_weights=True,
                                      verbose=verbose),
    ]
    if usar_reduce_lr:
        cbs.append(keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3,
                                                      patience=3, min_lr=1e-7,
                                                      verbose=verbose))
    t0 = time.time()
    hist = modelo.fit(ds_tr, validation_data=ds_va, epochs=epochs,
                      class_weight=cw, callbacks=cbs, verbose=verbose)
    segundos = time.time() - t0

    modelo.save(ruta_modelo)
    split_hash_task = SPLIT_HASH if task_config["n_classes"] == 2 else SPLIT_HASH_MULTICLASE_CONGELADO
    meta = {
        "nombre": nombre, "tarea": task_config["name"], "etiqueta": ETIQUETA_MODELO[nombre],
        "sufijo": sufijo,
        "manifest_hash": MANIFEST_HASH, "split_hash": split_hash_task,
        "config_hash": CONFIG_HASH, "seed": SEED, "usar_reduce_lr": usar_reduce_lr,
        "img_size": list(IMG_SIZE),
        "canales": ADAPTADOR_ENTRADA[nombre][2],
        "preprocessing": "grises/255" if nombre == "cnn_scratch"
                         else f"repetir 3 canales + {nombre}.preprocess_input",
        "class_map": task_config["class_map"],
        "epocas_realizadas": len(hist.history.get("loss", [])),
        "segundos_entrenamiento": round(segundos, 1),
        "parametros": int(modelo.count_params()),
        "parametros_entrenables": int(sum(int(np.prod(w.shape)) for w in modelo.trainable_weights)),
        "tamano_MB": round(os.path.getsize(ruta_modelo) / 1024 ** 2, 2),
        "historial": {k: [float(x) for x in v] for k, v in hist.history.items()},
        "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    ruta_meta.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"  [entrenado]   {task_config['name']}_{nombre}{sufijo}: "
          f"{meta['epocas_realizadas']} épocas, {segundos:.0f}s, {meta['tamano_MB']} MB "
          f"(ReduceLROnPlateau={'sí' if usar_reduce_lr else 'NO -- config histórica'}) "
          f"-> {ruta_modelo.name}")
    return modelo, meta


modelo_demo = construir("cnn_scratch", TASK_CONFIG)
modelo_demo.summary()
print(f"\nEntrada de la CNN propia: {modelo_demo.input_shape} "
      f"({CANALES_CNN} canal = escala de grises); salida para tarea "
      f"'{TASK_CONFIG['name']}': {modelo_demo.output_shape}")
del modelo_demo


### Utilidades compartidas de comparación de modelos (test bloqueado)

Regla que este cuaderno respeta sin excepciones (salvo la comprobación de regresión del
baseline `cnn_scratch` multiclase, documentada explícitamente donde ocurre):

```
TRAIN  -> ajustar pesos
VAL    -> early stopping, hiperparámetros, elección del modelo, elección de
          agregación por paciente y de umbral (tarea binaria)
TEST   -> se usa como mucho UNA VEZ por modelo final, con todo lo anterior congelado
```

Estas funciones son **comunes** a las dos tareas; reciben `task_config` explícitamente y
guardan cada resultado bajo `results/<tarea>/...`, para no mezclar artefactos binarios con
multiclase.

In [ ]:
# ==============================================================================
# UTILIDADES COMPARTIDAS PARA LA COMPARACIÓN DE MODELOS (TEST BLOQUEADO)
# ==============================================================================
import gc
from sklearn.model_selection import StratifiedGroupKFold


def _latencia_ms(modelo, arquitectura, task_config, df_referencia, repeticiones=5):
    ds = hacer_ds(df_referencia.head(BATCH_SIZE), arquitectura, task_config)
    xb, _ = next(iter(ds))
    for _ in range(2):                      # calentamiento: evita medir el tracing
        modelo.predict(xb, verbose=0)
    t0 = time.time()
    for _ in range(repeticiones):
        modelo.predict(xb, verbose=0)
    return (time.time() - t0) * 1000.0 / (repeticiones * int(xb.shape[0]))


def probabilidades_de_validacion(nombre, task_config, df_tr, df_va, df_trainval_activo):
    """Devuelve (df_referencia, probabilidades, meta, modelo_o_None) usando SOLO
    validación (holdout) o predicciones out-of-fold (groupcv). df_tr/df_va/
    df_trainval_activo deben ser EXPLÍCITAMENTE los de la tarea activa -- nunca
    se leen los globales df_train/df_val a ciegas."""
    epochs = EPOCHS_CNN if nombre == "cnn_scratch" else EPOCHS_BACKBONE
    if SELECCION_MODELO == "holdout":
        modelo, meta = entrenar_o_cargar(nombre, df_tr, df_va, task_config, epochs, verbose=0)
        return df_va, predecir(modelo, df_va, nombre, task_config), meta, modelo

    # ---- validación cruzada agrupada por paciente sobre train+val ----
    y = df_trainval_activo[task_config["label_col"]].map(task_config["class_map"]).values
    grupos = df_trainval_activo["patient_id"].values
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS_CV, shuffle=True, random_state=SEED)
    if task_config["n_classes"] == 2:
        oof = np.full(len(df_trainval_activo), np.nan)
    else:
        oof = np.full((len(df_trainval_activo), task_config["n_classes"]), np.nan)
    epocas, metas = [], []
    for k, (itr, iva) in enumerate(sgkf.split(df_trainval_activo, y, groups=grupos), start=1):
        tr = df_trainval_activo.iloc[itr].reset_index(drop=True)
        va = df_trainval_activo.iloc[iva].reset_index(drop=True)
        assert not (set(tr.patient_id) & set(va.patient_id)), "fuga dentro de la CV"
        modelo, meta = entrenar_o_cargar(nombre, tr, va, task_config, epochs,
                                         sufijo=f"_cv{k}", verbose=0)
        oof[iva] = predecir(modelo, va, nombre, task_config)
        epocas.append(meta["epocas_realizadas"]); metas.append(meta)
        del modelo
        keras.backend.clear_session()
        gc.collect()
    meta = dict(metas[-1])
    meta["epocas_mediana_cv"] = int(np.median(epocas))
    assert not np.isnan(oof).any(), "quedaron cortes sin predicción out-of-fold"
    return df_trainval_activo, oof, meta, None


def _ruta_resultado_val(nombre, task_config):
    return TASK_METRICS_DIR / f"resultado_val_{task_config['name']}_{nombre}.json"


def _resultado_val_valido(fila, task_config):
    split_hash_esperado = (SPLIT_HASH if task_config["n_classes"] == 2
                           else SPLIT_HASH_MULTICLASE_CONGELADO)
    return (fila.get("manifest_hash") == MANIFEST_HASH
            and fila.get("split_hash") == split_hash_esperado
            and fila.get("config_hash") == CONFIG_HASH)


def evaluar_y_guardar_modelo(nombre, task_config, df_tr, df_va, df_trainval_activo,
                             tipo_experimento, forzar=False):
    """Entrena (o reutiliza) UN modelo bajo `task_config`, lo evalúa SOLO con
    validación, guarda el resultado en disco (bajo results/<tarea>/) y libera
    memoria/GPU. `tipo_experimento` es una etiqueta descriptiva ("baseline
    regression check", "new multiclass benchmark", "diagnostic prior
    experiment"...) que queda guardada junto al resultado."""
    ruta_resultado = _ruta_resultado_val(nombre, task_config)
    if ruta_resultado.exists() and not forzar:
        fila_previa = json.loads(ruta_resultado.read_text())
        if _resultado_val_valido(fila_previa, task_config):
            print(f"[reutilizado] {task_config['name']}/{ETIQUETA_MODELO[nombre]}: "
                  "resultado de validación ya calculado para este split -> no se "
                  "reentrena ni se reevalúa\n")
            return fila_previa
        print(f"[descartado]  {task_config['name']}/{ETIQUETA_MODELO[nombre]}: el "
              "resultado guardado corresponde a otro dataset/split -> se recalcula")

    print(f"--- {ETIQUETA_MODELO[nombre]}  [{task_config['name']} · {tipo_experimento}]")
    df_ref, probs, meta, modelo_vivo = probabilidades_de_validacion(
        nombre, task_config, df_tr, df_va, df_trainval_activo)

    if task_config["n_classes"] == 2:
        y_slice = df_ref[task_config["label_col"]].map(task_config["class_map"]).values
        m_slice = metricas(y_slice, probs, task_config, 0.5)

        mejor_agg, mejor_auc_pac = None, -1.0
        for agg in AGREGACIONES:
            pac = por_paciente(df_ref, probs, task_config, agg)
            auc_p = roc_auc_score(pac["y"], pac["prob"])
            if auc_p > mejor_auc_pac:
                mejor_agg, mejor_auc_pac = agg, float(auc_p)
        pac = por_paciente(df_ref, probs, task_config, mejor_agg)
        thr = umbral_desde_validacion(pac["y"], pac["prob"], "youden")
        m_pac = metricas(pac["y"], pac["prob"], task_config, thr)
        fila = {
            "modelo": ETIQUETA_MODELO[nombre], "clave": nombre,
            "tarea": task_config["name"], "tipo_experimento": tipo_experimento,
            "val_auc_corte": float(m_slice["roc_auc"]), "val_prauc_corte": float(m_slice["pr_auc"]),
            "val_auc_paciente": float(m_pac["roc_auc"]), "val_prauc_paciente": float(m_pac["pr_auc"]),
            "val_bal_acc_paciente": float(m_pac["balanced_accuracy"]),
            "val_sens_paciente": float(m_pac["sensitivity"]),
            "val_spec_paciente": float(m_pac["specificity"]),
            "agregacion": mejor_agg, "umbral": float(thr),
            "n_pacientes_val": int(len(pac)),
        }
    else:
        # Multiclase: agregación fija = media de probabilidad por clase + argmax
        # (única implementada). Selección por balanced accuracy, no por AUC/umbral.
        pac = por_paciente(df_ref, probs, task_config, "media")
        prob_cols = [f"p_{c}" for c in task_config["classes"]]
        m_pac = metricas(pac["y"], pac[prob_cols].values, task_config)
        fila = {
            "modelo": ETIQUETA_MODELO[nombre], "clave": nombre,
            "tarea": task_config["name"], "tipo_experimento": tipo_experimento,
            "val_bal_acc_paciente": float(m_pac["balanced_accuracy"]),
            "val_accuracy_paciente": float(m_pac["accuracy"]),
            "val_macro_f1_paciente": float(m_pac["macro_f1"]),
            "val_recall_por_clase": m_pac["recall_por_clase"],
            "agregacion": "media", "umbral": None,
            "n_pacientes_val": int(len(pac)),
        }

    if modelo_vivo is None:                 # en CV se mide sobre un modelo suelto
        ruta_cv1 = _rutas(nombre, task_config, "_cv1")[0]
        ruta_holdout = _rutas(nombre, task_config)[0]
        modelo_vivo = keras.models.load_model(
            ruta_cv1 if SELECCION_MODELO == "groupcv" else ruta_holdout)
    ms = _latencia_ms(modelo_vivo, nombre, task_config, df_va)
    params = int(modelo_vivo.count_params())
    params_entrenables = int(sum(int(np.prod(w.shape)) for w in modelo_vivo.trainable_weights))
    tam_mb = float(meta["tamano_MB"])
    del modelo_vivo
    keras.backend.clear_session()
    gc.collect()

    fila["parametros"] = params
    fila["parametros_entrenables"] = params_entrenables
    fila["tamano_MB"] = tam_mb
    fila["ms_por_img"] = float(ms)

    fila_guardada = dict(fila)
    fila_guardada["manifest_hash"] = MANIFEST_HASH
    fila_guardada["split_hash"] = (SPLIT_HASH if task_config["n_classes"] == 2
                                   else SPLIT_HASH_MULTICLASE_CONGELADO)
    fila_guardada["config_hash"] = CONFIG_HASH
    fila_guardada["seed"] = SEED
    fila_guardada["fecha"] = time.strftime("%Y-%m-%d %H:%M:%S")
    fila_guardada["meta"] = meta
    ruta_resultado.write_text(json.dumps(fila_guardada, indent=2, default=str), encoding="utf-8")

    if task_config["n_classes"] == 2:
        print(f"    AUC paciente={m_pac['roc_auc']:.4f} · bal.acc={m_pac['balanced_accuracy']:.4f} "
              f"· agregación={mejor_agg} · umbral={thr:.3f} · {ms:.1f} ms/img")
    else:
        print(f"    bal.acc paciente={m_pac['balanced_accuracy']:.4f} · "
              f"macro-F1={m_pac['macro_f1']:.4f} · agregación=media · {ms:.1f} ms/img")
    print(f"    [guardado] -> {ruta_resultado.name}\n")
    return fila_guardada


print("Utilidades de comparación listas (probabilidades_de_validacion, "
      "evaluar_y_guardar_modelo, _latencia_ms).")


## 7. Baseline `cnn_scratch` — regresión del resultado histórico (79,7 %)

No existe checkpoint ni predicciones guardadas del modelo que dio 79,7 % de balanced
accuracy patient-level (ver `baseline_multiclase_cheng.md`, sección 11): la única forma de
comprobar que este framework refactorizado reproduce ese resultado es reentrenar
`cnn_scratch` bajo el split/configuración exactamente congelados y comparar.

**Esta es la única celda de la tarea multiclase que toca TEST en esta fase**, y lo hace por
la razón anterior, documentada como *"baseline regression check"*: es una prueba de
regresión de código, **no** una selección de modelo ni una decisión metodológica. Su
resultado no se usa para elegir arquitectura, hiperparámetros ni estrategia de agregación
— eso sigue decidiéndose exclusivamente con validación, como en el resto del cuaderno.

**Reproduce exactamente la configuración histórica**, distinta de la que usan los
benchmarks nuevos de la Sección 8: Adam `lr=1e-4`, `EarlyStopping` sobre `val_accuracy`,
`patience=8`, `restore_best_weights=True`, máximo `EPOCHS_CNN` (40) épocas, **sin**
`ReduceLROnPlateau`, sin `class_weight` (ya es `None` para la tarea multiclase), mismo
augmentation, mismo split y misma agregación por media de probabilidades. Esto se
consigue con `entrenar_o_cargar(..., usar_reduce_lr=False)`, que **solo** afecta a esta
llamada — los benchmarks de la Sección 8 y la tarea binaria de la Sección 10 siguen
usando `ReduceLROnPlateau` como hasta ahora.

Si la balanced accuracy reproducida se aleja más de `UMBRAL_REGRESION_ACEPTABLE` (±0.03)
de la histórica, el cuaderno guarda el resultado (para poder diagnosticar la causa) y
**lanza `RuntimeError`**: la ejecución se detiene aquí y NO se entrenan los cuatro
backbones de la Sección 8. No se ajustan hiperparámetros para forzar artificialmente el
79,7 %; si esto ocurre, hay que revisar la causa (split, preprocesamiento, arquitectura)
antes de continuar.

In [ ]:
# ==============================================================================
# 7. BASELINE cnn_scratch (multiclase) -- REGRESIÓN DEL RESULTADO HISTÓRICO
# ==============================================================================
TIPO_EXPERIMENTO_BASELINE = "baseline regression check"
BALANCED_ACCURACY_HISTORICA = 0.7965811965811967   # control_3clases.json, run SLURM 70436
UMBRAL_REGRESION_ACEPTABLE = 0.03   # margen razonable por DETERMINISMO_ESTRICTO=False en GPU
USAR_REDUCE_LR_BASELINE = False     # el cnn_scratch histórico NO usaba ReduceLROnPlateau

activar_tarea("tumor_type")

print("Configuración de entrenamiento del baseline (reproduce EXACTAMENTE la histórica):")
print(f"  Optimizador       : Adam(lr={LR})")
print(f"  EarlyStopping     : monitor={TASK_CONFIG['monitor_metric']}, patience={PACIENCIA}, "
      "restore_best_weights=True")
print(f"  Épocas máximas    : {EPOCHS_CNN}")
print(f"  ReduceLROnPlateau : {'SÍ' if USAR_REDUCE_LR_BASELINE else 'NO (config histórica)'}")
print("  class_weight      : None (ya es así para la tarea multiclase)")

keras.utils.set_random_seed(SEED)
modelo_baseline, meta_baseline = entrenar_o_cargar(
    "cnn_scratch", ap_tr, ap_va, TASK_CONFIG, EPOCHS_CNN, sufijo="_baseline_regresion",
    usar_reduce_lr=USAR_REDUCE_LR_BASELINE)

prob_val_baseline = predecir(modelo_baseline, ap_va, "cnn_scratch", TASK_CONFIG)
pac_val_baseline = por_paciente(ap_va, prob_val_baseline, TASK_CONFIG, "media")
_prob_cols = [f"p_{c}" for c in TASK_CONFIG["classes"]]
m_val_baseline = metricas(pac_val_baseline["y"], pac_val_baseline[_prob_cols].values, TASK_CONFIG)
bloque_metricas("cnn_scratch baseline -- VALIDACIÓN (nivel de paciente)", m_val_baseline, TASK_CONFIG)

# ---- comprobación de regresión sobre TEST (excepción documentada) ----
prob_test_baseline = predecir(modelo_baseline, ap_te, "cnn_scratch", TASK_CONFIG)
pac_test_baseline = por_paciente(ap_te, prob_test_baseline, TASK_CONFIG, "media")
m_test_baseline = metricas(pac_test_baseline["y"], pac_test_baseline[_prob_cols].values, TASK_CONFIG)

diferencia_abs = m_test_baseline["balanced_accuracy"] - BALANCED_ACCURACY_HISTORICA
print("\n" + "=" * 78)
print("REGRESIÓN DEL BASELINE -- cnn_scratch, tarea multiclase (Cheng, 3 clases)")
print("=" * 78)
bloque_metricas("cnn_scratch baseline -- TEST (baseline regression check, NO selección de modelo)",
                m_test_baseline, TASK_CONFIG)
print(f"\nBalanced accuracy histórica (run SLURM 70436) : {BALANCED_ACCURACY_HISTORICA:.4f}")
print(f"Balanced accuracy reproducida ahora            : {m_test_baseline['balanced_accuracy']:.4f}")
print(f"Diferencia absoluta                            : {diferencia_abs:+.4f}")

_regresion_aceptada = abs(diferencia_abs) <= UMBRAL_REGRESION_ACEPTABLE

resultado_baseline = {
    "modelo": "cnn_scratch", "tarea": "tumor_type",
    "tipo_experimento": TIPO_EXPERIMENTO_BASELINE,
    "balanced_accuracy_historica": BALANCED_ACCURACY_HISTORICA,
    "balanced_accuracy_test": m_test_baseline["balanced_accuracy"],
    "diferencia_absoluta": diferencia_abs,
    "umbral_regresion_aceptable": UMBRAL_REGRESION_ACEPTABLE,
    "regresion_aceptada": _regresion_aceptada,
    "usar_reduce_lr": USAR_REDUCE_LR_BASELINE,
    "accuracy_test": m_test_baseline["accuracy"],
    "macro_f1_test": m_test_baseline["macro_f1"],
    "recall_por_clase_test": m_test_baseline["recall_por_clase"],
    "balanced_accuracy_val": m_val_baseline["balanced_accuracy"],
    "matriz_confusion_test": m_test_baseline["matriz_confusion"],
    "split_hash_multiclase": SPLIT_HASH_MULTICLASE_CONGELADO,
    "manifest_hash": MANIFEST_HASH, "seed": SEED,
    "parametros": int(modelo_baseline.count_params()),
    "parametros_entrenables": int(sum(int(np.prod(w.shape)) for w in modelo_baseline.trainable_weights)),
    "epocas_realizadas": meta_baseline["epocas_realizadas"],
    "segundos_entrenamiento": meta_baseline["segundos_entrenamiento"],
    "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
}
# Se guarda SIEMPRE, incluso si la regresión falla, para poder diagnosticar la causa
# sin tener que reentrenar.
(TASK_METRICS_DIR / "baseline_regression_cnn_scratch.json").write_text(
    json.dumps(resultado_baseline, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

pac_test_baseline.rename(columns={"clase_real": "true_class", "pred": "predicted_class"})[
    ["patient_id", "true_class", "predicted_class"] + _prob_cols + ["n_cortes"]
].to_csv(TASK_PREDICTIONS_DIR / "predicciones_paciente_test_baseline_cnn_scratch.csv", index=False)

print(f"\nGuardado: baseline_regression_cnn_scratch.json, "
      "predicciones_paciente_test_baseline_cnn_scratch.csv")

if not _regresion_aceptada:
    raise RuntimeError(
        f"REGRESIÓN DEL BASELINE FALLIDA: la diferencia absoluta ({diferencia_abs:+.4f}) "
        f"supera el margen aceptado (±{UMBRAL_REGRESION_ACEPTABLE}) frente al resultado "
        f"histórico ({BALANCED_ACCURACY_HISTORICA:.4f}). El resultado se ha guardado en "
        f"{TASK_METRICS_DIR / 'baseline_regression_cnn_scratch.json'} para diagnóstico, "
        "pero la EJECUCIÓN SE DETIENE aquí: no se entrenan VGG16/ResNet50/MobileNetV2/"
        "EfficientNetB0 hasta que se revise y resuelva la causa de esta diferencia (split, "
        "preprocesamiento, arquitectura, semilla...). No se ajustan hiperparámetros para "
        "forzar artificialmente el resultado histórico.")

print(f"\nDiferencia dentro de un margen razonable de variabilidad por "
      f"DETERMINISMO_ESTRICTO=False en GPU (±{UMBRAL_REGRESION_ACEPTABLE}). Se acepta "
      "como reproducción del baseline histórico. Continúa la ejecución.")


## 8. Benchmark de transfer learning multiclase (nuevo, solo validación)

VGG16, ResNet50, MobileNetV2 y EfficientNetB0 **no tienen ningún resultado histórico en la
tarea multiclase** — hasta ahora solo se habían evaluado en la tarea binaria. Por tanto no
se presentan como "reproducción" de nada: son el **primer benchmark multiclase** de estas
cuatro arquitecturas, etiquetado explícitamente como `"new multiclass benchmark"` en la
tabla comparativa.

Test permanece bloqueado para estos cuatro modelos en esta fase: la comparación se hace
íntegramente con `train`/`validation`. La evaluación final en test (para el modelo que se
elija como definitivo) queda para más adelante, cuando también se hayan incorporado
InceptionV3 y la fusión de características — así se puede atribuir cualquier cambio de
resultados a una modificación concreta.

In [ ]:
if _RUN_BENCHMARK_MC:
    # ==============================================================================
    # 8.0  Preparación común del benchmark multiclase
    # ==============================================================================
    activar_tarea("tumor_type")
    ap_trainval = pd.concat([ap_tr, ap_va], ignore_index=True)
    TIPO_EXPERIMENTO_BENCHMARK = "new multiclass benchmark"
    resultados_benchmark_mc = {}
    print("Listo para evaluar VGG16 / ResNet50 / MobileNetV2 / EfficientNetB0 en la tarea "
          "multiclase (train+val, protocolo "
          f"{SELECCION_MODELO.upper()}). Test permanece bloqueado.")
else:
    print("[RUN_MODE] Sección 8.0 omitida (RUN_MODE='baseline_only'): no se preparan los benchmarks.")


In [ ]:
if _RUN_BENCHMARK_MC:
    # ==============================================================================
    # 8.1  VGG16 -- benchmark multiclase (independiente del resto de 8.x)
    # ==============================================================================
    # Requiere haber ejecutado antes, en esta sesión: secciones 0-6 y la celda 8.0.
    # "new multiclass benchmark": SIN resultado histórico previo, solo train/validation.
    resultados_benchmark_mc["vgg16"] = evaluar_y_guardar_modelo(
        "vgg16", TASK_CONFIG, ap_tr, ap_va, ap_trainval, TIPO_EXPERIMENTO_BENCHMARK)
else:
    print("[RUN_MODE] Sección 8.1 omitida (RUN_MODE='baseline_only'): VGG16 no se entrena.")


In [ ]:
if _RUN_BENCHMARK_MC:
    # ==============================================================================
    # 8.2  ResNet50 -- benchmark multiclase (independiente del resto de 8.x)
    # ==============================================================================
    # Requiere haber ejecutado antes, en esta sesión: secciones 0-6 y la celda 8.0.
    # "new multiclass benchmark": SIN resultado histórico previo, solo train/validation.
    resultados_benchmark_mc["resnet50"] = evaluar_y_guardar_modelo(
        "resnet50", TASK_CONFIG, ap_tr, ap_va, ap_trainval, TIPO_EXPERIMENTO_BENCHMARK)
else:
    print("[RUN_MODE] Sección 8.2 omitida (RUN_MODE='baseline_only'): ResNet50 no se entrena.")


In [ ]:
if _RUN_BENCHMARK_MC:
    # ==============================================================================
    # 8.3  MobileNetV2 -- benchmark multiclase (independiente del resto de 8.x)
    # ==============================================================================
    # Requiere haber ejecutado antes, en esta sesión: secciones 0-6 y la celda 8.0.
    # "new multiclass benchmark": SIN resultado histórico previo, solo train/validation.
    resultados_benchmark_mc["mobilenetv2"] = evaluar_y_guardar_modelo(
        "mobilenetv2", TASK_CONFIG, ap_tr, ap_va, ap_trainval, TIPO_EXPERIMENTO_BENCHMARK)
else:
    print("[RUN_MODE] Sección 8.3 omitida (RUN_MODE='baseline_only'): MobileNetV2 no se entrena.")


In [ ]:
if _RUN_BENCHMARK_MC:
    # ==============================================================================
    # 8.4  EfficientNetB0 -- benchmark multiclase (independiente del resto de 8.x)
    # ==============================================================================
    # Requiere haber ejecutado antes, en esta sesión: secciones 0-6 y la celda 8.0.
    # "new multiclass benchmark": SIN resultado histórico previo, solo train/validation.
    resultados_benchmark_mc["efficientnetb0"] = evaluar_y_guardar_modelo(
        "efficientnetb0", TASK_CONFIG, ap_tr, ap_va, ap_trainval, TIPO_EXPERIMENTO_BENCHMARK)
else:
    print("[RUN_MODE] Sección 8.4 omitida (RUN_MODE='baseline_only'): EfficientNetB0 no se entrena.")


## 9. Comparación multiclase, gráficas y resumen reproducible

Tabla acumulativa con el baseline (`cnn_scratch`, *baseline regression check*, evaluado en
test por la excepción documentada) y los cuatro modelos del benchmark nuevo (*new
multiclass benchmark*, evaluados solo en validación). No se elige todavía un modelo
definitivo ni se toca test para los cuatro nuevos: eso llega en la siguiente iteración,
junto con InceptionV3 y la fusión de características.

In [ ]:
if _RUN_BENCHMARK_MC:
    # ==============================================================================
    # 9.1  TABLA COMPARATIVA MULTICLASE (baseline regression + new multiclass benchmark)
    # ==============================================================================
    _baseline_json = json.loads((TASK_METRICS_DIR / "baseline_regression_cnn_scratch.json").read_text())

    _filas_comparacion = [{
        "modelo": "cnn_scratch", "tipo_experimento": _baseline_json["tipo_experimento"],
        "balanced_accuracy_paciente": _baseline_json["balanced_accuracy_test"],
        "accuracy_paciente": _baseline_json["accuracy_test"],
        "macro_f1_paciente": _baseline_json["macro_f1_test"],
        "recall_glioma": _baseline_json["recall_por_clase_test"]["glioma"],
        "recall_meningioma": _baseline_json["recall_por_clase_test"]["meningioma"],
        "recall_pituitario": _baseline_json["recall_por_clase_test"]["pituitario"],
        "parametros": _baseline_json["parametros"],
        "parametros_entrenables": _baseline_json["parametros_entrenables"],
        "mejor_epoca": _baseline_json["epocas_realizadas"],
        "tiempo_entrenamiento_s": _baseline_json["segundos_entrenamiento"],
        "split_hash": _baseline_json["split_hash_multiclase"], "seed": _baseline_json["seed"],
        "conjunto_evaluado": "TEST (baseline regression check, excepción documentada)",
    }]

    for _nombre, _fila in resultados_benchmark_mc.items():
        _filas_comparacion.append({
            "modelo": _nombre, "tipo_experimento": _fila["tipo_experimento"],
            "balanced_accuracy_paciente": _fila["val_bal_acc_paciente"],
            "accuracy_paciente": _fila["val_accuracy_paciente"],
            "macro_f1_paciente": _fila["val_macro_f1_paciente"],
            "recall_glioma": _fila["val_recall_por_clase"]["glioma"],
            "recall_meningioma": _fila["val_recall_por_clase"]["meningioma"],
            "recall_pituitario": _fila["val_recall_por_clase"]["pituitario"],
            "parametros": _fila["parametros"],
            "parametros_entrenables": _fila["parametros_entrenables"],
            "mejor_epoca": _fila["meta"].get("epocas_mediana_cv", _fila["meta"].get("epocas_realizadas")),
            "tiempo_entrenamiento_s": _fila["meta"]["segundos_entrenamiento"],
            "split_hash": _fila["split_hash"], "seed": _fila["seed"],
            "conjunto_evaluado": "VALIDACIÓN (selección pendiente de InceptionV3/fusión)",
        })

    tabla_multiclase = pd.DataFrame(_filas_comparacion)
    pd.set_option("display.width", 220)
    pd.set_option("display.float_format", lambda v: f"{v:.4f}")
    print("COMPARACIÓN MULTICLASE -- cnn_scratch es 'baseline regression check' (usa TEST, "
          "por la razón documentada); el resto son 'new multiclass benchmark' (SOLO "
          "validación, test permanece bloqueado)\n")
    print(tabla_multiclase.to_string(index=False))

    tabla_multiclase.to_csv(EXPERIMENT_SUMMARY_CSV, index=False)
    print(f"\nGuardado: {EXPERIMENT_SUMMARY_CSV}")
else:
    print("[RUN_MODE] Sección 9.1 omitida (RUN_MODE='baseline_only'): no hay resultados de benchmark que comparar.")


In [ ]:
if _RUN_BENCHMARK_MC:
    # ==============================================================================
    # 9.2  GRÁFICAS COMPARATIVAS -- TAREA MULTICLASE
    # ==============================================================================
    fig, ax = plt.subplots(2, 2, figsize=(13, 10))
    ax = ax.ravel()
    orden = tabla_multiclase.sort_values("balanced_accuracy_paciente", ascending=False)

    ax[0].barh(orden["modelo"], orden["balanced_accuracy_paciente"], color="#4C72B0")
    ax[0].axvline(BALANCED_ACCURACY_HISTORICA, color="r", ls="--",
                  label=f"histórico={BALANCED_ACCURACY_HISTORICA:.3f}")
    ax[0].set_title("Balanced accuracy (paciente)"); ax[0].invert_yaxis(); ax[0].legend()

    ax[1].barh(orden["modelo"], orden["macro_f1_paciente"], color="#55A868")
    ax[1].set_title("Macro-F1 (paciente)"); ax[1].invert_yaxis()

    _x = np.arange(len(tabla_multiclase))
    _ancho = 0.25
    ax[2].bar(_x - _ancho, tabla_multiclase["recall_glioma"], _ancho, label="glioma")
    ax[2].bar(_x, tabla_multiclase["recall_meningioma"], _ancho, label="meningioma")
    ax[2].bar(_x + _ancho, tabla_multiclase["recall_pituitario"], _ancho, label="pituitario")
    ax[2].set_xticks(_x); ax[2].set_xticklabels(tabla_multiclase["modelo"], rotation=30, ha="right")
    ax[2].set_title("Recall por clase"); ax[2].legend(); ax[2].set_ylim(0, 1)

    ax[3].scatter(tabla_multiclase["tiempo_entrenamiento_s"], tabla_multiclase["balanced_accuracy_paciente"],
                  s=tabla_multiclase["parametros"] / 5e4 + 40, c="#C44E52", edgecolors="k", alpha=.8)
    for _, r in tabla_multiclase.iterrows():
        ax[3].annotate(r["modelo"], (r["tiempo_entrenamiento_s"], r["balanced_accuracy_paciente"]),
                       fontsize=8, xytext=(5, 5), textcoords="offset points")
    ax[3].set_xlabel("tiempo de entrenamiento (s)"); ax[3].set_ylabel("balanced accuracy (paciente)")
    ax[3].set_title("Calidad frente a coste (tamaño = nº parámetros)")

    plt.tight_layout()
    guardar_figura(fig, "comparacion_multiclase_resumen")
    plt.show()
    print(f"Guardado en: {TASK_PLOTS_DIR}/comparacion_multiclase_resumen.{{png,pdf}}")
else:
    print("[RUN_MODE] Sección 9.2 omitida (RUN_MODE='baseline_only'): no hay tabla comparativa que graficar.")


In [ ]:
if _RUN_BENCHMARK_MC:
    # ==============================================================================
    # 9.3  RESUMEN REPRODUCIBLE DEL EXPERIMENTO (multiclase)
    # ==============================================================================
    resumen_experimento_mc = {
        "tarea_activa": TASK, "clases": TASK_CONFIG["classes"],
        "split_hash_multiclase": SPLIT_HASH_MULTICLASE_CONGELADO,
        "manifest_hash": MANIFEST_HASH, "config_hash": CONFIG_HASH,
        "seed": SEED, "determinismo_estricto": DETERMINISMO_ESTRICTO,
        "img_size": list(IMG_SIZE), "batch_size": BATCH_SIZE,
        "seleccion_modelo": SELECCION_MODELO, "n_folds_cv": N_FOLDS_CV,
        "modelos_evaluados": list(tabla_multiclase["modelo"]),
        "balanced_accuracy_historica_cnn_scratch": BALANCED_ACCURACY_HISTORICA,
        "diferencia_absoluta_regresion": _baseline_json["diferencia_absoluta"],
        "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    (TASK_RESULTS_DIR / "resumen_reproducible.json").write_text(
        json.dumps(resumen_experimento_mc, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(resumen_experimento_mc, indent=2, ensure_ascii=False))
else:
    print("[RUN_MODE] Sección 9.3 omitida (RUN_MODE='baseline_only'): no hay resumen de benchmark que guardar.")


In [ ]:
if _RUN_BENCHMARK_MC:
    # ==============================================================================
    # 9.4  CONCLUSIONES PRELIMINARES -- TAREA MULTICLASE (generadas, no escritas a mano)
    # ==============================================================================
    from IPython.display import Markdown, display

    _orden_bench = tabla_multiclase[tabla_multiclase["tipo_experimento"] == "new multiclass benchmark"] \
        .sort_values("balanced_accuracy_paciente", ascending=False)
    _mejor_bench = _orden_bench.iloc[0] if len(_orden_bench) else None

    _texto = [
        f"### Conclusiones preliminares -- tarea multiclase (glioma/meningioma/pituitario)",
        "",
        f"- Reproducción del baseline histórico (`cnn_scratch`): balanced accuracy "
        f"{_baseline_json['balanced_accuracy_test']:.4f} en test frente al histórico "
        f"{BALANCED_ACCURACY_HISTORICA:.4f} (diferencia {_baseline_json['diferencia_absoluta']:+.4f}, "
        f"{'dentro' if abs(_baseline_json['diferencia_absoluta']) <= UMBRAL_REGRESION_ACEPTABLE else 'FUERA'} "
        f"del margen esperado por no determinismo).",
    ]
    if _mejor_bench is not None:
        _texto.append(
            f"- Mejor modelo del benchmark nuevo en VALIDACIÓN: **{_mejor_bench['modelo']}** "
            f"(balanced accuracy {_mejor_bench['balanced_accuracy_paciente']:.4f}, macro-F1 "
            f"{_mejor_bench['macro_f1_paciente']:.4f}). Esta cifra es de VALIDACIÓN, no de test: "
            "no se ha hecho todavía ninguna selección final ni se ha tocado test para estos "
            "cuatro modelos.")
    _texto.append(
        "- Esta fase se considera un punto de control intermedio, no el cierre de la "
        "experimentación: quedan pendientes InceptionV3, un fine-tuning limitado, una o dos "
        "estrategias de fusión de características y la evaluación final en test del modelo "
        "que finalmente se elija (ver `baseline_multiclase_cheng.md` y el plan de FASE 2).")
    display(Markdown("\n".join(_texto)))
else:
    print("[RUN_MODE] Sección 9.4 omitida (RUN_MODE='baseline_only'): sin benchmark, no hay conclusiones que generar todavía.")


## 10. Tarea binaria — experimento previo y diagnóstico (sección secundaria)

**Secciones 10 a 14 completas quedan condicionadas a `RUN_MODE == "full"` y
`RUN_BINARY_SECONDARY == True`** (ver 1.5). Con la configuración por defecto
(`RUN_MODE = "baseline_only"`) cada una de estas celdas se ejecuta pero no hace nada:
imprime un aviso de que ha sido omitida y continúa con la siguiente. El código sigue
íntegro y sin cambios metodológicos; solo se activa cuando se pide explícitamente.

A partir de aquí se reactiva explícitamente la tarea `binary` (`activar_tarea("binary")`):
todo lo que sigue usa el mismo framework generalizado de las secciones 3-6, con
`TASK_CONFIG = TASK_CONFIGS["binary"]`. **No hay ningún cambio metodológico, de
arquitectura, de split ni de hiperparámetros respecto a versiones anteriores** — es
exactamente el mismo protocolo histórico, simplemente ejecutado a través de las funciones
ya generalizadas en vez de una copia aparte.

Esta tarea se conserva porque fue la que reveló el problema de sesgo de procedencia (Sección
2.6) que motivó el cambio de eje hacia la clasificación de tipo de tumor, pero **ya no es el
resultado principal del TFG**.

In [ ]:
if _RUN_BINARIA:
    activar_tarea("binary")
else:
    print("[RUN_MODE] Sección 10 (tarea binaria) omitida: RUN_MODE != 'full' o RUN_BINARY_SECONDARY=False.")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 10.1  ENTRENAMIENTO DE LA CNN PROPIA -- TAREA BINARIA (evaluación SOLO en validación)
    # ==============================================================================
    # Aquí NO aparece el conjunto de test. Test permanece bloqueado hasta la sección 5.
    # Todo lo que se decide en esta sección —número de épocas efectivo, learning rate,
    # pesos restaurados por early stopping— se decide mirando validación.
    # ==============================================================================
    FORZAR_REENTRENAMIENTO = False

    model_scratch, meta_scratch = entrenar_o_cargar(
        "cnn_scratch", df_train, df_val, TASK_CONFIG, EPOCHS_CNN, forzar=FORZAR_REENTRENAMIENTO)

    prob_val_scratch = predecir(model_scratch, df_val, "cnn_scratch", TASK_CONFIG)
    m_val_05 = metricas((df_val.label == "tumor").astype(int), prob_val_scratch, TASK_CONFIG, 0.5)
    bloque_metricas("CNN propia -- VALIDACIÓN, nivel de corte, umbral 0.5", m_val_05, TASK_CONFIG)

    pac_val_scratch = por_paciente(df_val, prob_val_scratch, TASK_CONFIG, "media")
    m_val_pac = metricas(pac_val_scratch["y"], pac_val_scratch["prob"], TASK_CONFIG, 0.5)
    bloque_metricas("CNN propia -- VALIDACIÓN, nivel de PACIENTE (agregación: media), umbral 0.5",
                    m_val_pac, TASK_CONFIG)
    print(f"\nPacientes en validación: {len(pac_val_scratch)} "
          f"(tumor={int(pac_val_scratch.y.sum())}, notumor={int((1-pac_val_scratch.y).sum())})")
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ------------------------------------------------------------------------------
    # 3.4  CURVAS DE ENTRENAMIENTO Y VALIDACIÓN
    # ------------------------------------------------------------------------------
    # El historial se guarda dentro del .json del modelo, así que las curvas se
    # dibujan igual aunque el modelo se haya recargado de disco. En la versión
    # anterior, al recargar se creaba un objeto de historial vacío y las gráficas
    # salían en blanco sin avisar.
    # ------------------------------------------------------------------------------
    hist = meta_scratch.get("historial", {})
    if not hist:
        print("Este modelo se recargó de un artefacto sin historial: no hay curvas que dibujar.")
    else:
        claves = [("loss", "Pérdida"), ("auc", "ROC-AUC"), ("accuracy", "Accuracy")]
        fig, ax = plt.subplots(1, len(claves), figsize=(5 * len(claves), 4))
        for a, (k, titulo) in zip(np.atleast_1d(ax), claves):
            if k in hist:
                a.plot(hist[k], label="train")
            if f"val_{k}" in hist:
                a.plot(hist[f"val_{k}"], label="validación")
            a.set_title(f"{titulo} — CNN propia")
            a.set_xlabel("época"); a.grid(alpha=.3); a.legend()
        plt.tight_layout()
        guardar_figura(fig, "3_4_curvas_entrenamiento")
        plt.show()
        mejor = int(np.argmax(hist.get("val_auc", [0]))) + 1
        print(f"Mejor época según val_auc: {mejor} de {len(hist.get('loss', []))} "
              f"(early stopping restauró esos pesos)")
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


### 10.3 Entrenamiento y comparación de los 5 modelos (protocolo histórico)

In [ ]:
if _RUN_BINARIA:
    TIPO_EXPERIMENTO_BINARIO = "diagnostic prior experiment"
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 4.1.1  CNN PROPIA — entrena/evalúa/guarda (independiente del resto de 4.1.x)
    # ==============================================================================
    # Requiere haber ejecutado antes, en esta sesión: secciones 1-3 y la celda 4.1.0.
    # Si ya existe un resultado válido para este split, no reentrena ni reevalúa: solo
    # lo reutiliza. Libera memoria y GPU al terminar.
    # ==============================================================================
    _ = evaluar_y_guardar_modelo(
        "cnn_scratch", TASK_CONFIG, df_train, df_val, df_trainval, TIPO_EXPERIMENTO_BINARIO)
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 4.1.2  VGG16 (transfer, base congelada) — entrena/evalúa/guarda
    # ==============================================================================
    # Requiere haber ejecutado antes, en esta sesión: secciones 1-3 y la celda 4.1.0.
    # Independiente de las demás celdas 4.1.x: puedes ejecutar solo esta hoy y el
    # resto otro día. Si ya existe un resultado válido para este split, lo reutiliza.
    # ==============================================================================
    _ = evaluar_y_guardar_modelo(
        "vgg16", TASK_CONFIG, df_train, df_val, df_trainval, TIPO_EXPERIMENTO_BINARIO)
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 4.1.3  ResNet50 (transfer, base congelada) — entrena/evalúa/guarda
    # ==============================================================================
    # Requiere haber ejecutado antes, en esta sesión: secciones 1-3 y la celda 4.1.0.
    # Independiente de las demás celdas 4.1.x. Si ya existe un resultado válido para
    # este split, lo reutiliza sin reentrenar.
    # ==============================================================================
    _ = evaluar_y_guardar_modelo(
        "resnet50", TASK_CONFIG, df_train, df_val, df_trainval, TIPO_EXPERIMENTO_BINARIO)
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 4.1.4  MobileNetV2 (transfer, base congelada) — entrena/evalúa/guarda
    # ==============================================================================
    # Requiere haber ejecutado antes, en esta sesión: secciones 1-3 y la celda 4.1.0.
    # Independiente de las demás celdas 4.1.x. Si ya existe un resultado válido para
    # este split, lo reutiliza sin reentrenar.
    # ==============================================================================
    _ = evaluar_y_guardar_modelo(
        "mobilenetv2", TASK_CONFIG, df_train, df_val, df_trainval, TIPO_EXPERIMENTO_BINARIO)
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 4.1.5  EfficientNetB0 (transfer, base congelada) — entrena/evalúa/guarda
    # ==============================================================================
    # Requiere haber ejecutado antes, en esta sesión: secciones 1-3 y la celda 4.1.0.
    # Independiente de las demás celdas 4.1.x. Si ya existe un resultado válido para
    # este split, lo reutiliza sin reentrenar.
    # ==============================================================================
    _ = evaluar_y_guardar_modelo(
        "efficientnetb0", TASK_CONFIG, df_train, df_val, df_trainval, TIPO_EXPERIMENTO_BINARIO)
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 10.4  COMPARACIÓN FINAL (binaria) -- carga resultados ya guardados, SIN reentrenar
    # ==============================================================================
    # Lee, para cada arquitectura de NOMBRES_MODELOS, el fichero
    # ART_DIR / f"resultado_val_{nombre}.json" que haya dejado su celda 4.1.x
    # correspondiente. No entrena ni carga ningún modelo de Keras: solo reconstruye
    # results_val_df y val_meta, tal y como los consumían las secciones 4.2 y 4.3.
    # Si falta el resultado de algún modelo (o no corresponde a este split), se avisa
    # explícitamente en vez de fallar en silencio o inventar una fila.
    # ==============================================================================
    _COLS_TABLA = ["modelo", "clave", "val_auc_corte", "val_prauc_corte", "val_auc_paciente",
                   "val_prauc_paciente", "val_bal_acc_paciente", "val_sens_paciente",
                   "val_spec_paciente", "agregacion", "umbral", "parametros", "tamano_MB",
                   "ms_por_img", "n_pacientes_val"]

    filas_val, val_meta, faltantes = [], {}, []
    for nombre in NOMBRES_MODELOS:
        ruta_resultado = _ruta_resultado_val(nombre, TASK_CONFIG)
        if not ruta_resultado.exists():
            faltantes.append(nombre)
            continue
        fila_guardada = json.loads(ruta_resultado.read_text())
        if not _resultado_val_valido(fila_guardada, TASK_CONFIG):
            faltantes.append(nombre)
            continue
        filas_val.append({k: fila_guardada[k] for k in _COLS_TABLA})
        val_meta[nombre] = fila_guardada.get("meta", {})

    if faltantes:
        print("AVISO: todavía faltan resultados de validación (o no corresponden a este "
              f"split) para: {faltantes}")
        print("Ejecuta la celda 4.1.x correspondiente a cada uno de ellos antes de "
              "continuar. La tabla de abajo solo incluye los modelos ya evaluados.\n")

    assert filas_val, ("no hay ningún resultado de validación guardado todavía: ejecuta "
                        "al menos una celda 4.1.x")

    results_val_df = pd.DataFrame(filas_val)
    print(f"Comparación con {len(results_val_df)}/{len(NOMBRES_MODELOS)} modelos evaluados "
          f"(protocolo {SELECCION_MODELO.upper()}):\n")
    print(results_val_df[["modelo", "val_auc_paciente", "val_bal_acc_paciente",
                           "agregacion", "umbral", "ms_por_img"]].to_string(index=False))
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 10.5  TABLA DE VALIDACIÓN Y ELECCIÓN DEL GANADOR (binaria)
    # ==============================================================================
    # Aquí se separan dos preguntas que antes se mezclaban en una sola fórmula:
    #
    #   (a) ¿Qué modelo DISCRIMINA mejor?  -> ROC-AUC a nivel de PACIENTE en validación,
    #       con PR-AUC y balanced accuracy como desempate. Es el criterio que decide
    #       el modelo final.
    #
    #   (b) ¿Qué modelo es mejor para DESPLEGAR? -> `deployment_score`, que combina
    #       tamaño y latencia. Es una pregunta de ingeniería, NO una métrica médica,
    #       y por eso no elige el modelo diagnóstico.
    #
    # Mezclar MB y milisegundos con AUC en una única cifra y llamar al resultado
    # "mejor modelo diagnóstico" no es defendible; aquí se separan explícitamente.
    # ==============================================================================
    tabla = results_val_df.copy()
    tabla["deployment_score"] = (
        0.5 * (1 - tabla["tamano_MB"] / tabla["tamano_MB"].max())
        + 0.5 * (1 - tabla["ms_por_img"] / tabla["ms_por_img"].max()))

    orden_calidad = tabla.sort_values(
        ["val_auc_paciente", "val_prauc_paciente", "val_bal_acc_paciente"],
        ascending=False).reset_index(drop=True)

    cols = ["modelo", "val_auc_paciente", "val_bal_acc_paciente", "val_sens_paciente",
            "val_spec_paciente", "val_auc_corte", "agregacion", "umbral",
            "parametros", "tamano_MB", "ms_por_img", "deployment_score"]
    pd.set_option("display.width", 200)
    pd.set_option("display.float_format", lambda v: f"{v:.4f}")
    print("SELECCIÓN DE MODELO — solo VALIDACIÓN (test bloqueado)\n")
    print(orden_calidad[cols].to_string(index=False))

    GANADOR = orden_calidad.iloc[0]["clave"]
    GANADOR_ETIQUETA = orden_calidad.iloc[0]["modelo"]
    AGREGACION_FINAL = orden_calidad.iloc[0]["agregacion"]
    UMBRAL_FINAL = float(orden_calidad.iloc[0]["umbral"])

    mejor_despliegue = tabla.sort_values("deployment_score", ascending=False).iloc[0]

    print(f"\nGanador por capacidad discriminativa (AUC a nivel de paciente en validación):")
    print(f"   {GANADOR_ETIQUETA}")
    print(f"   agregación por paciente = {AGREGACION_FINAL}")
    print(f"   umbral congelado        = {UMBRAL_FINAL:.4f}")
    print(f"\nMejor para despliegue (deployment_score, criterio de ingeniería, NO médico):")
    print(f"   {mejor_despliegue['modelo']}  "
          f"({mejor_despliegue['tamano_MB']:.1f} MB, {mejor_despliegue['ms_por_img']:.1f} ms/img)")

    # ¿la diferencia entre los dos primeros es interpretable?
    if len(orden_calidad) > 1:
        d = orden_calidad.iloc[0]["val_auc_paciente"] - orden_calidad.iloc[1]["val_auc_paciente"]
        print(f"\nDistancia en AUC de paciente entre el 1º y el 2º: {d:.4f}")
        if d < 0.02:
            print("   Es una diferencia pequeña: con este número de pacientes no permite")
            print("   afirmar que un modelo sea claramente superior al otro.")

    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].barh(orden_calidad["modelo"], orden_calidad["val_auc_paciente"], color="#4C72B0")
    ax[0].set_title("ROC-AUC a nivel de paciente (VALIDACIÓN)"); ax[0].invert_yaxis()
    ax[0].set_xlim(0.5, 1.0)
    ax[1].scatter(tabla["ms_por_img"], tabla["val_auc_paciente"],
                  s=tabla["tamano_MB"] * 2 + 40, c="#55A868", edgecolors="k", alpha=.8)
    for _, r in tabla.iterrows():
        ax[1].annotate(r["clave"], (r["ms_por_img"], r["val_auc_paciente"]),
                       fontsize=8, xytext=(5, 5), textcoords="offset points")
    ax[1].set_xlabel("latencia (ms/imagen)"); ax[1].set_ylabel("AUC paciente (validación)")
    ax[1].set_title("Calidad frente a coste (tamaño de burbuja = MB)")
    plt.tight_layout(); plt.show()

    results_val_df = orden_calidad
    results_val_df.to_csv(TASK_METRICS_DIR / "resultados_validacion.csv", index=False)
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 10.6  CONGELAR EL MODELO FINAL (binaria) -- última celda antes de tocar TEST
    # ==============================================================================
    # Aquí se fija, en este orden y sin mirar test ni una vez:
    #
    #   1. el modelo ganador (elegido en 4.2 con el protocolo de validación),
    #   2. el modelo final entrenado (train, con early stopping sobre val),
    #   3. la regla de agregación por paciente, elegida con las predicciones de ESE
    #      modelo sobre validación,
    #   4. el umbral, elegido por Youden sobre esas mismas predicciones.
    #
    # Por qué el modelo final se entrena sobre TRAIN y no sobre train+val, incluso
    # cuando la selección se ha hecho por validación cruzada: si se reentrena con
    # train+val, la escala de probabilidades del modelo resultante ya no es la de las
    # predicciones out-of-fold con las que se calculó el umbral, y el umbral deja de
    # transferir. Se prefiere un umbral coherente con el modelo que realmente se va a
    # evaluar antes que exprimir un 15 % más de datos. Quien quiera lo contrario tiene
    # el interruptor de abajo, con la advertencia puesta.
    # ==============================================================================
    REENTRENAR_GANADOR_CON_TRAINVAL = False   # ver la nota de arriba antes de activarlo

    epocas_ganador = int(val_meta[GANADOR].get("epocas_mediana_cv",
                                               val_meta[GANADOR]["epocas_realizadas"]))

    if REENTRENAR_GANADOR_CON_TRAINVAL:
        print(f"[AVISO] Reentrenando {GANADOR} sobre train+val durante {epocas_ganador} épocas.")
        print("        El umbral se calculará sobre validación, que ahora forma parte del")
        print("        entrenamiento: será optimista. Anótalo como limitación.")
        keras.utils.set_random_seed(SEED)
        modelo_final = construir(GANADOR, TASK_CONFIG)
        modelo_final.fit(hacer_ds(df_trainval, GANADOR, TASK_CONFIG, entrenamiento=True),
                         epochs=epocas_ganador, class_weight=CLASS_WEIGHT, verbose=1)
        entrenado_sobre = "train+val"
    else:
        modelo_final, meta_final = entrenar_o_cargar(
            GANADOR, df_train, df_val, TASK_CONFIG,
            EPOCHS_CNN if GANADOR == "cnn_scratch" else EPOCHS_BACKBONE)
        epocas_ganador = int(meta_final["epocas_realizadas"])
        entrenado_sobre = "train"

    # ---- agregación y umbral, con las predicciones del MODELO FINAL sobre VAL ----
    prob_val_final = predecir(modelo_final, df_val, GANADOR, TASK_CONFIG)
    tabla_agg = []
    for agg in AGREGACIONES:
        pac_v = por_paciente(df_val, prob_val_final, TASK_CONFIG, agg)
        tabla_agg.append({"agregacion": agg,
                          "auc_paciente_val": float(roc_auc_score(pac_v["y"], pac_v["prob"])),
                          "n_pacientes": int(len(pac_v))})
    tabla_agg = pd.DataFrame(tabla_agg).sort_values("auc_paciente_val", ascending=False)
    print("\nElección de la regla de agregación por paciente (sobre VALIDACIÓN):")
    print(tabla_agg.to_string(index=False))

    AGREGACION_FINAL = tabla_agg.iloc[0]["agregacion"]
    pac_val_final = por_paciente(df_val, prob_val_final, TASK_CONFIG, AGREGACION_FINAL)
    UMBRAL_FINAL = umbral_desde_validacion(pac_val_final["y"], pac_val_final["prob"], "youden")
    m_val_final = metricas(pac_val_final["y"], pac_val_final["prob"], TASK_CONFIG, UMBRAL_FINAL)
    bloque_metricas(f"Modelo final en VALIDACIÓN (nivel de paciente, agregación "
                    f"{AGREGACION_FINAL})", m_val_final, TASK_CONFIG)

    RUTA_FINAL = TASK_MODELS_DIR / "modelo_final.keras"
    modelo_final.save(RUTA_FINAL)

    CONFIG_FINAL = {
        "modelo": GANADOR,
        "etiqueta": GANADOR_ETIQUETA,
        "arquitectura_entrada": {
            "img_size": list(IMG_SIZE),
            "canales": ADAPTADOR_ENTRADA[GANADOR][2],
            "preprocessing": "grises/255" if GANADOR == "cnn_scratch"
                             else f"repetir 3 canales + {GANADOR}.preprocess_input",
        },
        "class_index": CLASS_INDEX,
        "umbral": UMBRAL_FINAL,
        "agregacion_paciente": AGREGACION_FINAL,
        "criterio_umbral": "Youden sobre VALIDACIÓN con el modelo final (nunca sobre test)",
        "protocolo_seleccion": SELECCION_MODELO,
        "epocas": epocas_ganador,
        "entrenado_sobre": entrenado_sobre,
        "seed": SEED,
        "manifest_hash": MANIFEST_HASH,
        "split_hash": SPLIT_HASH,
        "config_hash": CONFIG_HASH,
        "parametros": int(modelo_final.count_params()),
        "tamano_MB": round(os.path.getsize(RUTA_FINAL) / 1024 ** 2, 2),
        "metricas_validacion_paciente": m_val_final,
        "seleccion_agregacion": tabla_agg.to_dict("records"),
        "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    (TASK_METRICS_DIR / "modelo_final.json").write_text(
        json.dumps(CONFIG_FINAL, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

    print("\nCONFIGURACIÓN CONGELADA")
    for k in ("modelo", "umbral", "agregacion_paciente", "entrenado_sobre", "epocas",
              "parametros", "tamano_MB", "manifest_hash", "split_hash", "config_hash"):
        print(f"   {k:22s}: {CONFIG_FINAL[k]}")
    print(f"\nArtefactos: {RUTA_FINAL.name} + modelo_final.json")
    print("A partir de aquí ya se puede abrir el conjunto de test, y solo una vez.")
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


## 11. Evaluación final sobre el conjunto de test binario (una sola vez)

Idéntica en metodología a versiones anteriores: única celda que toca test en la tarea
binaria, con modelo/umbral/agregación ya congelados en la sección 10.6.

In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 11.1  EVALUACIÓN FINAL SOBRE TEST BINARIO (una sola vez)
    # ==============================================================================
    # Esta es la ÚNICA celda del cuaderno que toca test. Todo lo que usa —modelo,
    # umbral y regla de agregación— viene congelado de la celda 4.3 y no se retoca a
    # la vista de estos resultados. Si algo de lo de abajo no gusta, la única salida
    # metodológicamente válida es cambiar el diseño y volver a separar un test nuevo.
    #
    # La métrica PRINCIPAL es la de nivel de PACIENTE: es la que responde a
    # "¿generaliza a pacientes nuevos?". La de nivel de corte se reporta como
    # análisis secundario.
    # ==============================================================================
    TEST_YA_USADO = globals().get("TEST_YA_USADO", False)
    if TEST_YA_USADO:
        print("[AVISO] El test ya se evaluó en esta sesión. Volver a ejecutar esta celda no\n"
              "        invalida nada por sí solo, pero si entre medias se ha cambiado el\n"
              "        modelo o el umbral, el resultado deja de ser una estimación honesta.")

    y_test_slice = (df_test.label == "tumor").astype(int).values
    prob_test = predecir(modelo_final, df_test, GANADOR, TASK_CONFIG)

    # ------------------------------------------------------- nivel de corte -----
    m_test_slice = metricas(y_test_slice, prob_test, TASK_CONFIG, UMBRAL_FINAL)
    m_test_slice_05 = metricas(y_test_slice, prob_test, TASK_CONFIG, 0.5)

    # ---------------------------------------------------- nivel de paciente -----
    pac_test = por_paciente(df_test, prob_test, TASK_CONFIG, AGREGACION_FINAL)
    m_test_pac = metricas(pac_test["y"], pac_test["prob"], TASK_CONFIG, UMBRAL_FINAL)
    m_test_pac_05 = metricas(pac_test["y"], pac_test["prob"], TASK_CONFIG, 0.5)
    ic_pac = bootstrap_ic(pac_test, TASK_CONFIG,
                          metric_fn=lambda d: metricas(d["y"].values, d["prob"].values,
                                                       TASK_CONFIG, UMBRAL_FINAL),
                          n_boot=2000)

    # IC a nivel de corte: también por bootstrap DE PACIENTES (cluster bootstrap)
    pac_ids = df_test["patient_id"].values
    _rng = np.random.RandomState(SEED)
    _unicos = pac_test["patient_id"].values
    _acum = {k: [] for k in ("accuracy", "balanced_accuracy", "roc_auc", "sensitivity", "specificity")}
    _indices = {p: np.where(pac_ids == p)[0] for p in _unicos}
    for _ in range(1000):
        elegidos = _rng.choice(_unicos, len(_unicos), replace=True)
        idx = np.concatenate([_indices[p] for p in elegidos])
        if len(np.unique(y_test_slice[idx])) < 2:
            continue
        mm = metricas(y_test_slice[idx], prob_test[idx], TASK_CONFIG, UMBRAL_FINAL)
        for k in _acum:
            _acum[k].append(mm[k])
    ic_slice = {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
                for k, v in _acum.items() if len(v)}

    print("=" * 78)
    print(f"RESULTADO FINAL — {GANADOR_ETIQUETA}")
    print(f"umbral congelado = {UMBRAL_FINAL:.4f} · agregación = {AGREGACION_FINAL}")
    print("=" * 78)
    print(f"\nPacientes -> train={audit['split']['train']['pacientes']}  "
          f"val={audit['split']['val']['pacientes']}  test={audit['split']['test']['pacientes']}")

    bloque_metricas("TEST -- nivel de PACIENTE (MÉTRICA PRINCIPAL)", m_test_pac, TASK_CONFIG, ic_pac)
    bloque_metricas("TEST -- nivel de corte (análisis secundario)", m_test_slice, TASK_CONFIG, ic_slice)
    print("\nReferencia con umbral 0.5 (sin optimizar), solo informativa:")
    print(f"  paciente : acc={m_test_pac_05['accuracy']:.4f}  "
          f"bal.acc={m_test_pac_05['balanced_accuracy']:.4f}  "
          f"sens={m_test_pac_05['sensitivity']:.4f}  spec={m_test_pac_05['specificity']:.4f}")
    print(f"  corte    : acc={m_test_slice_05['accuracy']:.4f}  "
          f"bal.acc={m_test_slice_05['balanced_accuracy']:.4f}")

    # --------------------------------------------------- desglose por fuente ----
    print("\nDesglose por fuente (test):")
    for s, g in pac_test.groupby("source"):
        if len(g):
            print(f"  {s:24s}: {len(g):3d} pacientes · "
                  f"prob. media={g['prob'].mean():.3f} · "
                  f"aciertos={(( g['prob']>=UMBRAL_FINAL).astype(int)==g['y']).mean():.3f}")
    print("  (con una fuente por clase, este desglose coincide con sensibilidad y")
    print("   especificidad: es la manifestación directa del sesgo de procedencia)")

    # ------------------------------------------------------------- gráficas -----
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    cm = np.array([[m_test_pac["tn"], m_test_pac["fp"]],
                   [m_test_pac["fn"], m_test_pac["tp"]]])
    ax[0].imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=16,
                       color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax[0].set_xticks([0, 1], ["pred. notumor", "pred. tumor"])
    ax[0].set_yticks([0, 1], ["real notumor", "real tumor"])
    ax[0].set_title("Matriz de confusión (pacientes)")

    fpr, tpr, _ = roc_curve(pac_test["y"], pac_test["prob"])
    ax[1].plot(fpr, tpr, lw=2, label=f"AUC={m_test_pac['roc_auc']:.4f}")
    ax[1].plot([0, 1], [0, 1], "k--", lw=1)
    ax[1].set_xlabel("1 - especificidad"); ax[1].set_ylabel("sensibilidad")
    ax[1].set_title("ROC (pacientes, test)"); ax[1].legend(); ax[1].grid(alpha=.3)

    prec, rec, _ = precision_recall_curve(pac_test["y"], pac_test["prob"])
    ax[2].plot(rec, prec, lw=2, label=f"PR-AUC={m_test_pac['pr_auc']:.4f}")
    ax[2].set_xlabel("sensibilidad"); ax[2].set_ylabel("precisión")
    ax[2].set_title("Precisión-Sensibilidad (pacientes, test)"); ax[2].legend(); ax[2].grid(alpha=.3)
    plt.tight_layout()
    guardar_figura(fig, "5_1_test_roc_pr_confusion")
    plt.show()
    test_report = {
        "modelo": GANADOR, "etiqueta": GANADOR_ETIQUETA,
        "umbral": UMBRAL_FINAL, "agregacion": AGREGACION_FINAL,
        "n_pacientes": {s: audit["split"][s]["pacientes"] for s in ("train", "val", "test")},
        "n_imagenes": {s: audit["split"][s]["imagenes"] for s in ("train", "val", "test")},
        "paciente": m_test_pac, "paciente_ic95": ic_pac,
        "corte": m_test_slice, "corte_ic95": ic_slice,
        "paciente_umbral_0.5": m_test_pac_05,
        "sesgo_de_fuente": bias_report,
        "fingerprint": FINGERPRINT,
    }
    (TASK_METRICS_DIR / "test_report.json").write_text(
        json.dumps(test_report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    pac_test.to_csv(TASK_PREDICTIONS_DIR / "predicciones_paciente_test.csv", index=False)
    df_test.assign(prob=prob_test).to_csv(TASK_PREDICTIONS_DIR / "predicciones_corte_test.csv", index=False)
    TEST_YA_USADO = True
    print(f"\nGuardado: test_report.json, predicciones_paciente_test.csv, "
          f"predicciones_corte_test.csv")
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


## 12. Interpretabilidad (Grad-CAM del modelo binario)

Grad-CAM muestra qué regiones de la imagen contribuyen a la predicción. Es una herramienta
de **interpretación visual**, no una prueba de validez clínica: que el mapa caiga sobre una
lesión no demuestra que el modelo sea fiable, aunque que caiga sistemáticamente fuera sí es
una señal de alarma aprovechable.

En este dataset tiene además un uso concreto: si en la clase `notumor` el calor se concentra
en bordes, fondo o cráneo, es evidencia visual del sesgo de procedencia que la sección 1.6
cuantifica numéricamente.

La localización de la última capa convolucional es **dinámica**: no hay ningún nombre de
capa escrito a mano que se rompa al cambiar de arquitectura, y funciona igual con la CNN de
un canal que con los preentrenados de tres.

In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 6.1  GRAD-CAM DEL MODELO FINAL
    # ==============================================================================
    # Qué es y qué NO es este mapa. Grad-CAM muestra qué regiones de la imagen
    # contribuyen a la predicción del modelo. Es una herramienta de inspección, no
    # una prueba de validez clínica: que el mapa caiga sobre una lesión no demuestra
    # que el modelo sea fiable, y que caiga fuera sí es una señal de alarma útil.
    # Por eso aquí se habla de "interpretación visual de las regiones que contribuyen
    # a la predicción" y nunca de fiabilidad clínica.
    #
    # En este dataset Grad-CAM tiene además un uso concreto: si los mapas de la clase
    # `notumor` se concentran en bordes, fondo o cráneo, es evidencia visual del
    # sesgo de procedencia medido en 1.6.
    #
    # Detalles técnicos a tener en cuenta aquí:
    #     a mano, que se rompía al cambiar de arquitectura);
    #   * funciona con la CNN de 1 canal y con los preentrenados de 3;
    #   * la imagen de fondo se dibuja siempre desde el PNG original, no desde el
    #     tensor preprocesado (que puede estar centrado en cero y salir en negativo).
    # ==============================================================================
    import matplotlib.cm as cm_mpl


    def ultima_conv(modelo):
        for capa in reversed(modelo.layers):
            salida = getattr(capa, "output", None)
            if salida is not None and len(getattr(salida, "shape", ())) == 4 \
               and isinstance(capa, keras.layers.Conv2D):
                return capa.name
        for capa in reversed(modelo.layers):
            if len(getattr(getattr(capa, "output", None), "shape", ())) == 4:
                return capa.name
        raise ValueError("No se encontró ninguna capa convolucional 4D en el modelo")


    def preparar_entrada(ruta, arquitectura):
        """PNG -> tensor listo para el modelo (1, H, W, C) + imagen para dibujar."""
        _, preprocess, canales = ADAPTADOR_ENTRADA[arquitectura]
        with Image.open(ruta) as im:
            gris = np.asarray(im.convert("L").resize(IMG_SIZE[::-1]), dtype=np.float32)
        x = gris[None, ..., None]
        if canales == 3:
            x = np.repeat(x, 3, axis=-1)
            x = preprocess(x.copy())
        else:
            x = x / 255.0
        return x.astype("float32"), gris / 255.0


    def gradcam(modelo, x, nombre_capa=None):
        nombre_capa = nombre_capa or ultima_conv(modelo)
        entrada = modelo.inputs[0] if isinstance(modelo.inputs, (list, tuple)) else modelo.inputs
        salida = modelo.outputs[0] if isinstance(modelo.outputs, (list, tuple)) else modelo.outputs
        grad_model = keras.Model(entrada, [modelo.get_layer(nombre_capa).output, salida])
        with tf.GradientTape() as cinta:
            conv, pred = grad_model(x, training=False)
            objetivo = pred[:, 0]
        grads = cinta.gradient(objetivo, conv)
        if grads is None:
            raise RuntimeError("No hay gradiente hacia la capa elegida")
        pesos = tf.reduce_mean(grads, axis=(1, 2))
        mapa = tf.einsum("bhwc,bc->bhw", conv, pesos)
        mapa = tf.nn.relu(mapa)[0].numpy()
        if mapa.max() > 0:
            mapa /= mapa.max()
        mapa = np.asarray(Image.fromarray((mapa * 255).astype(np.uint8))
                          .resize(IMG_SIZE[::-1], Image.BILINEAR)) / 255.0
        return mapa, float(pred.numpy().reshape(-1)[0])


    def superponer(gris, mapa, alfa=0.4):
        calor = cm_mpl.jet(mapa)[..., :3]
        base = np.stack([gris] * 3, axis=-1)
        return np.clip((1 - alfa) * base + alfa * calor, 0, 1)


    CAPA_CAM = ultima_conv(modelo_final)
    print(f"Modelo final: {GANADOR_ETIQUETA}")
    print(f"Última capa convolucional localizada dinámicamente: '{CAPA_CAM}'")

    # --- ejemplos de test: los más seguros de cada clase y un par de errores ------
    tmp = df_test.assign(prob=prob_test)
    tmp["pred"] = (tmp["prob"] >= UMBRAL_FINAL).astype(int)
    tmp["y"] = (tmp["label"] == "tumor").astype(int)
    ejemplos = pd.concat([
        tmp[tmp.y == 1].nlargest(2, "prob"),
        tmp[tmp.y == 0].nsmallest(2, "prob"),
        tmp[tmp.y != tmp.pred].head(2),
    ])

    if len(ejemplos):
        fig, ax = plt.subplots(2, len(ejemplos), figsize=(3 * len(ejemplos), 6.5))
        ax = np.atleast_2d(ax)
        for c, (_, fila) in enumerate(ejemplos.iterrows()):
            x, gris = preparar_entrada(fila["local_file"], GANADOR)
            try:
                mapa, p = gradcam(modelo_final, x, CAPA_CAM)
            except Exception as e:
                mapa, p = np.zeros(IMG_SIZE), float(fila["prob"])
                print("  Grad-CAM no disponible para este caso:", e)
            acierto = "OK" if int(p >= UMBRAL_FINAL) == fila["y"] else "ERROR"
            ax[0, c].imshow(gris, cmap="gray"); ax[0, c].axis("off")
            ax[0, c].set_title(f"{fila['label']} · p={p:.2f}\n{fila['patient_id']} [{acierto}]",
                               fontsize=8)
            ax[1, c].imshow(superponer(gris, mapa)); ax[1, c].axis("off")
            ax[1, c].set_title("Grad-CAM", fontsize=8)
        fig.suptitle("Interpretación visual de las regiones que contribuyen a la predicción",
                     fontsize=12)
        plt.tight_layout()
        guardar_figura(fig, "6_1_gradcam_ejemplos_test")
        plt.show()

    print("\nLectura prudente: el mapa indica dónde se apoya la predicción de ESTE modelo")
    print("sobre ESTAS imágenes. No permite afirmar que el modelo sea clínicamente fiable.")
    print("Si en la clase notumor el calor se concentra en el borde del encéfalo o en el")
    print("fondo, es coherente con el sesgo de procedencia cuantificado en la sección 1.6.")
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


## 13. Inferencia

### Qué es exactamente la inferencia sobre un volumen NIfTI

Conviene decirlo sin ambigüedad, porque es fácil describirlo mal:

```
volumen NIfTI  ->  cortes 2D  ->  CNN 2D corte a corte  ->  agregación
```

**Esto no es una clasificación volumétrica 3D.** El modelo nunca ve el volumen completo ni
la relación entre cortes contiguos. Lo que hay es una red 2D aplicada de forma
independiente a cada corte y una regla explícita que combina las probabilidades.

La coherencia con la sección 5 es deliberada: se usa **la misma regla de agregación** que se
eligió en validación y con la que se calcularon las métricas a nivel de paciente. Así, el
número que aparece al final de un volumen es del mismo tipo que el que se ha validado, y no
una regla nueva inventada en la última celda.

Los cortes se preparan igual que en 1.3 (recorte a la caja del cerebro, percentiles 1–99,
8 bits). Si se preparasen de otra forma, el modelo vería una distribución distinta de la de
entrenamiento y la predicción no sería comparable.

Una CNN 3D, un enfoque 2.5D o aprendizaje de múltiples instancias (MIL) queda como línea
futura.

In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 7.1  INFERENCIA SOBRE UNA IMAGEN SUELTA
    # ==============================================================================
    # Se reutiliza el modelo congelado en 4.3 con SU umbral y SU preprocesado. Si se
    # sube una imagen, se usa esa; si no, se toma un corte de test como demostración.
    # ==============================================================================
    ruta_demo = None
    try:
        from google.colab import files as colab_files
        print("Sube una imagen (.png/.jpg) o cancela para usar un ejemplo del test.")
        subida = colab_files.upload()
        if subida:
            ruta_demo = list(subida.keys())[0]
    except Exception:
        pass

    if ruta_demo is None:
        ruta_demo = df_test.sort_values("local_file").iloc[0]["local_file"]
        print("Sin subida: se usa un corte del conjunto de test ->", Path(ruta_demo).name)

    x, gris = preparar_entrada(ruta_demo, GANADOR)
    p = float(modelo_final.predict(x, verbose=0).reshape(-1)[0])
    etiqueta = "tumor" if p >= UMBRAL_FINAL else "notumor"
    mapa, _ = gradcam(modelo_final, x, CAPA_CAM)

    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(gris, cmap="gray"); ax[0].axis("off")
    ax[0].set_title(f"{etiqueta}  (p_tumor={p:.3f}, umbral={UMBRAL_FINAL:.3f})")
    ax[1].imshow(superponer(gris, mapa)); ax[1].axis("off"); ax[1].set_title("Grad-CAM")
    plt.tight_layout()
    guardar_figura(fig, "7_1_inferencia_imagen_suelta")
    plt.show()

    print("Recordatorio: se trata de un clasificador EXPERIMENTAL de investigación.")
    print("No es un dispositivo médico ni está validado para uso diagnóstico.")
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 7.2  INFERENCIA SOBRE UN VOLUMEN NIfTI  (qué es exactamente lo que se hace)
    # ==============================================================================
    # ESTO NO ES UN MODELO 3D. Lo que ocurre es:
    #
    #     volumen NIfTI  ->  cortes 2D  ->  CNN 2D corte a corte  ->  agregación
    #
    # Es decir, exactamente el mismo procedimiento que la métrica a nivel de paciente
    # de la sección 5: se predice cada corte y las probabilidades se combinan con la
    # MISMA regla congelada en 4.3 (`AGREGACION_FINAL`). Así la cifra que sale aquí es
    # comparable con la que se ha validado, y no una regla nueva inventada al final.
    #
    # Los cortes se preparan igual que en 1.3 (recorte a la caja del cerebro,
    # percentiles 1-99, 8 bits): si se preparan de otra forma, el modelo ve una
    # distribución distinta de la de entrenamiento y la predicción no es comparable.
    #
    # Una CNN 3D, 2.5D o un esquema de múltiples instancias (MIL) queda como línea
    # futura: aprovecharían la continuidad entre cortes, que este enfoque ignora.
    # ==============================================================================
    ruta_nii = None
    try:
        from google.colab import files as colab_files
        print("Sube un volumen .nii o .nii.gz (o cancela para usar uno de IXI).")
        subida = colab_files.upload()
        if subida:
            ruta_nii = list(subida.keys())[0]
    except Exception:
        pass

    if ruta_nii is None:
        candidatos = sorted(IXI_DIR.glob("*.nii.gz"))
        sujetos_test = set(df_test.loc[df_test.label == "notumor", "patient_id"])
        for c in candidatos:
            if f"IXI-{c.name.split('-')[0]}" in sujetos_test:
                ruta_nii = str(c); break
        ruta_nii = ruta_nii or (str(candidatos[0]) if candidatos else None)
        print("Sin subida: se usa", Path(ruta_nii).name if ruta_nii else "(no hay volúmenes)")

    if ruta_nii is None:
        print("No hay ningún volumen disponible para la demostración.")
    else:
        vol = np.asarray(nib.as_closest_canonical(nib.load(str(ruta_nii))).dataobj,
                         dtype=np.float32)
        if vol.ndim == 4:
            vol = vol[..., 0]
        print(f"Volumen {Path(ruta_nii).name}: {vol.shape}")

        tmp_dir = ROOT / "tmp_nifti"; shutil.rmtree(tmp_dir, ignore_errors=True); tmp_dir.mkdir()
        cortes, rutas_cortes = [], []
        for plano, k, corte in _cortes_ixi(vol):
            destino = tmp_dir / f"{plano}_{k}.png"
            if _a_png(corte, destino) is not None:
                rutas_cortes.append(destino); cortes.append((plano, k))

        probs = []
        for r in rutas_cortes:
            xx, _ = preparar_entrada(r, GANADOR)
            probs.append(float(modelo_final.predict(xx, verbose=0).reshape(-1)[0]))
        probs = np.array(probs)

        p_volumen = AGREGACIONES[AGREGACION_FINAL](probs)
        veredicto = "tumor" if p_volumen >= UMBRAL_FINAL else "notumor"
        print(f"\nCortes evaluados      : {len(probs)}  (mismos planos y misma zona central")
        print(f"                        que en el entrenamiento)")
        print(f"Regla de agregación   : {AGREGACION_FINAL}  (la congelada en 4.3)")
        print(f"Probabilidad de volumen: {p_volumen:.4f}")
        print(f"Umbral congelado       : {UMBRAL_FINAL:.4f}")
        print(f"VEREDICTO DEL VOLUMEN  : {veredicto}")
        print("\nRecuerda: es una agregación de predicciones 2D, no una clasificación")
        print("volumétrica 3D. El modelo nunca ve el volumen completo.")

        orden = np.argsort(-probs)[:4]
        fig, ax = plt.subplots(2, len(orden), figsize=(3.2 * len(orden), 6.5))
        ax = np.atleast_2d(ax)
        for c, i in enumerate(orden):
            xx, gris = preparar_entrada(rutas_cortes[i], GANADOR)
            mapa, _ = gradcam(modelo_final, xx, CAPA_CAM)
            ax[0, c].imshow(gris, cmap="gray"); ax[0, c].axis("off")
            ax[0, c].set_title(f"{cortes[i][0]} #{cortes[i][1]}\np={probs[i]:.3f}", fontsize=9)
            ax[1, c].imshow(superponer(gris, mapa)); ax[1, c].axis("off")
        fig.suptitle(f"Cortes con mayor probabilidad — {Path(ruta_nii).name}", fontsize=12)
        plt.tight_layout()
        guardar_figura(fig, "7_2_nifti_top_slices")
        plt.show()

        plt.figure(figsize=(9, 3))
        plt.plot(probs, marker="o", lw=1)
        plt.axhline(UMBRAL_FINAL, color="r", ls="--", label=f"umbral {UMBRAL_FINAL:.3f}")
        plt.xlabel("corte evaluado"); plt.ylabel("p(tumor)")
        plt.title("Perfil de probabilidad corte a corte"); plt.legend(); plt.grid(alpha=.3)
        plt.tight_layout()
        guardar_figura(plt.gcf(), "7_2_nifti_perfil_probabilidad")
        plt.show()
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


## 14. Conclusiones (experimento previo, tarea binaria)

Todo lo que sigue se genera a partir de las variables de esta ejecución (`audit`,
`results_val_df`, `test_report`). No hay ninguna cifra escrita a mano, de modo que el
cuaderno no puede contradecir sus propias salidas.

In [ ]:
if _RUN_BINARIA:
    # ==============================================================================
    # 14.1  CONCLUSIONES (TAREA BINARIA) GENERADAS A PARTIR DE LOS RESULTADOS DE ESTA EJECUCIÓN
    # ==============================================================================
    # Ni una sola cifra de este bloque está escrita a mano: todas salen de `audit`,
    # `results_val_df` y `test_report`. Si el dataset o el reparto cambian, el texto
    # cambia con ellos y el cuaderno no puede contradecir sus propias salidas.
    # ==============================================================================
    from IPython.display import Markdown, display

    mp, ic = test_report["paciente"], test_report["paciente_ic95"]
    ms_ = test_report["corte"]
    primero, segundo = results_val_df.iloc[0], (results_val_df.iloc[1]
                                                if len(results_val_df) > 1 else None)
    delta_auc = (float(primero["val_auc_paciente"] - segundo["val_auc_paciente"])
                 if segundo is not None else float("nan"))

    if segundo is None:
        frase_ganador = f"Solo se ha evaluado **{primero['modelo']}**."
    elif delta_auc < 0.02:
        frase_ganador = (
            f"**{primero['modelo']}** obtiene el mejor AUC a nivel de paciente en validación "
            f"({primero['val_auc_paciente']:.4f}), pero la distancia con "
            f"**{segundo['modelo']}** ({segundo['val_auc_paciente']:.4f}) es de solo "
            f"{delta_auc:.4f}. Con {int(primero['n_pacientes_val'])} pacientes de validación "
            f"esa diferencia **no permite declarar un ganador claro**: se elige uno para poder "
            f"cerrar el protocolo, no porque se haya demostrado que sea superior.")
    else:
        frase_ganador = (
            f"**{primero['modelo']}** es el mejor modelo por capacidad discriminativa en "
            f"validación (AUC de paciente {primero['val_auc_paciente']:.4f} frente a "
            f"{segundo['val_auc_paciente']:.4f} del siguiente, diferencia {delta_auc:.4f}).")

    sesgo = test_report["sesgo_de_fuente"]
    if sesgo["shortcut_accuracy"] and sesgo["shortcut_accuracy"] >= 0.90:
        frase_sesgo = (
            f"Un clasificador **trivial**, que solo ve descriptores de bajo nivel "
            f"(intensidad, contraste, borde negro, energía de bordes) y **no ve el tumor**, "
            f"alcanza {sesgo['shortcut_accuracy']:.3f} de accuracy y {sesgo['shortcut_auc']:.3f} "
            f"de AUC con validación cruzada agrupada por paciente. Es decir: en esta tarea "
            f"binaria **la procedencia basta para separar las clases**. Toda métrica binaria "
            f"de este cuaderno, incluida la de test, debe leerse con esa limitación delante: "
            f"mide en gran parte la capacidad de distinguir Cheng de IXI, no la de detectar "
            f"un tumor. La evidencia sin esa confusión está en el Apéndice A.")
    else:
        frase_sesgo = (
            f"El clasificador trivial sobre descriptores de bajo nivel alcanza "
            f"{sesgo['shortcut_accuracy']:.3f} de accuracy, por debajo del umbral que "
            f"consideraríamos crítico, aunque el diseño sigue teniendo una fuente por clase.")

    md = f"""
### Conclusiones (generadas automáticamente el {time.strftime('%d/%m/%Y %H:%M')})

**Dataset realmente utilizado.**
{audit['imagenes']['validas']} imágenes de {audit['pacientes']['reales']} pacientes con
identificador **real** ({audit['pacientes']['sin_identificar']} imágenes sin identificar).
Fuentes: {', '.join(f"{k} ({v['imagenes']} img / {v['pacientes']} pac.)" for k, v in audit['fuentes'].items())}.
Duplicados exactos eliminados: {audit['imagenes']['duplicados_exactos_eliminados']}.
Pares casi-duplicados confirmados y agrupados (no borrados): {audit['imagenes']['casi_duplicados_confirmados']}.
Conflictos de etiqueta: {audit['imagenes']['conflictos_de_etiqueta']}.

| Subconjunto | Pacientes | Imágenes | tumor | notumor |
|---|---|---|---|---|
| train | {audit['split']['train']['pacientes']} | {audit['split']['train']['imagenes']} | {audit['split']['train']['tumor']} | {audit['split']['train']['notumor']} |
| val | {audit['split']['val']['pacientes']} | {audit['split']['val']['imagenes']} | {audit['split']['val']['tumor']} | {audit['split']['val']['notumor']} |
| test | {audit['split']['test']['pacientes']} | {audit['split']['test']['imagenes']} | {audit['split']['test']['tumor']} | {audit['split']['test']['notumor']} |

**Ausencia de fuga.** Solape de pacientes train-val = {audit['solapes']['pacientes train-val']},
train-test = {audit['solapes']['pacientes train-test']},
val-test = {audit['solapes']['pacientes val-test']}.
Solape de clústeres de casi-duplicados entre subconjuntos:
{audit['solapes']['clúster train-val']} / {audit['solapes']['clúster train-test']} / {audit['solapes']['clúster val-test']}.
No es una afirmación: es el resultado de los asertos de la celda 1.5, que detienen
el cuaderno si alguno no se cumple. Estado de la auditoría: **{audit['estado']}**.

**Elección del modelo.** {frase_ganador}
La elección se hizo con el protocolo `{SELECCION_MODELO}` **sin mirar el test**.
Regla de agregación por paciente elegida en validación: `{AGREGACION_FINAL}`.
Umbral congelado en validación (Youden): `{UMBRAL_FINAL:.4f}`.

**Resultado final sobre test (una sola evaluación), nivel de PACIENTE.**
{test_report['n_pacientes']['test']} pacientes de test.

| Métrica | Valor | IC 95 % |
|---|---|---|
| ROC-AUC | {mp['roc_auc']:.4f} | [{ic['roc_auc'][0]:.4f}, {ic['roc_auc'][1]:.4f}] |
| PR-AUC | {mp['pr_auc']:.4f} | [{ic['pr_auc'][0]:.4f}, {ic['pr_auc'][1]:.4f}] |
| Balanced accuracy | {mp['balanced_accuracy']:.4f} | [{ic['balanced_accuracy'][0]:.4f}, {ic['balanced_accuracy'][1]:.4f}] |
| Sensibilidad | {mp['sensitivity']:.4f} | [{ic['sensitivity'][0]:.4f}, {ic['sensitivity'][1]:.4f}] |
| Especificidad | {mp['specificity']:.4f} | [{ic['specificity'][0]:.4f}, {ic['specificity'][1]:.4f}] |
| Accuracy | {mp['accuracy']:.4f} | [{ic['accuracy'][0]:.4f}, {ic['accuracy'][1]:.4f}] |

A nivel de corte (secundario): AUC {ms_['roc_auc']:.4f}, balanced accuracy {ms_['balanced_accuracy']:.4f}.

**Limitación principal.** {frase_sesgo}

**Qué se puede afirmar.** Que el reparto es estrictamente por paciente, que ningún
paciente ni ningún clúster de imágenes casi idénticas cruza subconjuntos, que el test
se usó una sola vez con el umbral y la agregación congelados de antemano, y que cada
imagen es trazable hasta su fichero de origen a través de `manifest_final.csv`.

**Qué NO se puede afirmar.** Que este modelo detecte tumores. El diseño no lo permite:
la fuente predice la etiqueta. Es un **clasificador experimental de investigación** con
**validación interna** y **validación externa pendiente**. No es un dispositivo médico.

**Líneas futuras.**
1. Cohorte con controles sanos del **mismo centro y protocolo** que los casos con tumor,
   único modo de convertir esta tarea binaria en una evaluación de detección real.
2. Recuperar `volume_id` y la vista (axial/coronal/sagital) de Cheng, que la colección
   publicada no distribuye, para pasar de paciente a volumen como unidad.
3. CNN 3D, 2.5D o aprendizaje de múltiples instancias (MIL) sobre el volumen completo,
   en lugar de agregar predicciones 2D independientes.
4. Calibración de probabilidades y cuantificación de incertidumbre.
5. Validación externa en un centro distinto.

*Huella de esta ejecución — manifest `{FINGERPRINT['manifest_hash']}`,
split `{FINGERPRINT['split_hash']}`, config `{FINGERPRINT['config_hash']}`.*
"""
    display(Markdown(md))
    (ART_DIR / "conclusiones.md").write_text(md, encoding="utf-8")
    print("Conclusiones guardadas en", ART_DIR / "conclusiones.md")
else:
    print("[RUN_MODE] Sección omitida: requiere RUN_MODE == 'full' y RUN_BINARY_SECONDARY == True (tarea binaria desactivada en este modo).")


## Apéndices

Lo que no forma parte del flujo principal vive aquí, para que el cuerpo del cuaderno se
pueda leer de principio a fin sin desvíos.

- **A — Tarea de control sin confusión fuente↔clase.** Repite el protocolo completo sobre
  la clasificación de los tres tipos de tumor usando solo pacientes de Cheng. Es la
  evidencia de si el modelo extrae información del tejido o solo la firma del escáner.
- **B — Modo *legacy* / proxy.** Demuestra con el dataset antiguo por qué el `group_id`
  deducido del nombre de fichero no identificaba pacientes. Marcado como
  *EXPLORATORY / NOT VALID FOR FINAL PATIENT-LEVEL RESULTS*.
- **C — Fuentes descartadas.** Qué fichero o metadato concreto haría falta para incorporar
  cada una de las diez colecciones de Kaggle conservando la trazabilidad, y la plantilla
  `manifest_template.csv` con el esquema a rellenar.

### Apéndice A — integrado en el cuerpo principal

La tarea de control de 3 clases que antes vivía por completo en este apéndice ya forma
parte del cuerpo principal del cuaderno:

- La verificación y congelación del split se hace en la **Sección 2.5**.
- El control de sesgo específico de Cheng (atajo de bajo nivel) se hace en la **Sección 2.7**.
- El entrenamiento y la evaluación (incluida la comprobación de regresión del 79,7 %) se
  hacen en la **Sección 7**.
- El benchmark de los otros cuatro modelos se hace en la **Sección 8**.

Esto evita mantener dos implementaciones independientes de la misma evaluación multiclase
(una "de control" aquí y otra en el cuerpo principal): ahora hay una sola, generalizada por
`task_config`, y este apéndice queda como referencia de dónde encontrar cada pieza.

In [ ]:
# ==============================================================================
# APÉNDICE B — MODO LEGACY / PROXY
#              *** EXPLORATORY / NOT VALID FOR FINAL PATIENT-LEVEL RESULTS ***
# ==============================================================================
# Este apéndice NO produce resultados para el TFG. Sirve para dos cosas:
#
#   1. Demostrar, con el ZIP antiguo de JPEG delante, POR QUÉ el `group_id`
#      deducido del nombre de fichero no era un identificador de paciente.
#   2. Dejar disponible el modo proxy para desarrollo, correctamente etiquetado.
#
# Reproduce literalmente la función `inferir_caso()` legacy y
# mide cuántos "pacientes" resultantes mezclan en realidad varias clases o varios
# tipos de tumor. Si un mismo "paciente" contiene a la vez imágenes `tumor` y
# `notumor`, el identificador no puede ser un paciente.
#
# Para ejecutarlo hay que subir el ZIP antiguo con las carpetas tumor/ y notumor/.
# Si no se sube nada, el apéndice se salta sin error.
# ==============================================================================
import re

EJECUTAR_APENDICE_B = False   # ponlo a True y sube el ZIP antiguo para la demostración

_SUFIJOS_AUG_LEGACY = ("_da_", "_hf_", "_vf_", "_aug", "_rot", "_flip")


def _stem_base_legacy(nombre):
    base = os.path.splitext(nombre)[0].lower()
    for suf in _SUFIJOS_AUG_LEGACY:
        base = base.replace(suf, "")
    return base.strip("_ -")


def inferir_fuente_legacy(nombre):
    n = _stem_base_legacy(nombre)
    if "_br_" in n or n.startswith("br"): return "Br35H"
    if "_sp_" in n: return "SARTAJ"
    if n.startswith(("g_", "m_", "p_", "n_")): return "figshare/Cheng"
    if n.startswith(("gl", "me", "pi", "no")): return "Nickparvar"
    return "otra"


def inferir_caso_legacy(nombre):
    """Función legacy, reproducida tal cual."""
    fuente = inferir_fuente_legacy(nombre)
    base = _stem_base_legacy(nombre)
    m = re.match(r"^[a-z]_(\d+)", base)
    if m:
        return "%s#%s" % (fuente, m.group(1))
    m = re.match(r"^(gl|me|pi|no)[-_]?(\d+)", base)
    if m:
        return "%s#%s" % (fuente, m.group(2))
    return "%s#%s" % (fuente, base)


if not EJECUTAR_APENDICE_B:
    print("Apéndice B desactivado (EJECUTAR_APENDICE_B = False).")
    print("\nDemostración mínima con nombres de fichero reales del dataset anterior:")
    for n in ["G_588_HF_.jpg", "M_588.jpg", "P_588_VF_.jpg", "N_588_DA_.jpg",
              "G_810.jpg", "N_153_SP_.jpg"]:
        print(f"   {n:18s} -> fuente={inferir_fuente_legacy(n):16s} "
              f"group_id={inferir_caso_legacy(n)}")
    print("\nG_588 (glioma), M_588 (meningioma), P_588 (pituitario) y N_588 (sin tumor)")
    print("reciben EL MISMO group_id. No son un paciente: son cuatro imágenes distintas")
    print("que comparten número de fichero, y dos de ellas son de CLASES OPUESTAS.")
    print("Además la fuente 'figshare/Cheng' está mal atribuida: figshare/Cheng no")
    print("contiene ninguna imagen sin tumor, así que ningún fichero N_* puede venir")
    print("de ahí (el patrón corresponde a colecciones derivadas de SARTAJ ya aumentadas).")
else:
    from google.colab import files as colab_files
    print("Sube el ZIP antiguo (carpetas tumor/ y notumor/):")
    subida = colab_files.upload()
    legacy_dir = ROOT / "legacy"; shutil.rmtree(legacy_dir, ignore_errors=True)
    legacy_dir.mkdir(parents=True)
    with zipfile.ZipFile(list(subida.keys())[0]) as z:
        z.extractall(legacy_dir)

    filas = []
    for raiz, _, ficheros in os.walk(legacy_dir):
        clase = Path(raiz).name.lower()
        if clase not in ("tumor", "notumor"):
            continue
        for f in ficheros:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                filas.append({"fichero": f, "clase": clase,
                              "fuente_proxy": inferir_fuente_legacy(f),
                              "proxy_group_id": inferir_caso_legacy(f)})
    leg = pd.DataFrame(filas)
    print(f"\nImágenes del ZIP antiguo: {len(leg)}")
    print(leg.groupby(["fuente_proxy", "clase"]).size().unstack(fill_value=0))

    g = leg.groupby("proxy_group_id")["clase"].nunique()
    mezclados = int((g > 1).sum())
    print(f"\n'Pacientes' (proxy_group_id) totales      : {len(g)}")
    print(f"'Pacientes' que contienen AMBAS clases    : {mezclados} "
          f"({100*mezclados/max(len(g),1):.1f} %)")
    print(f"Imágenes afectadas                        : "
          f"{int(leg[leg.proxy_group_id.isin(g[g>1].index)].shape[0])}")
    if mezclados:
        ejemplo = g[g > 1].index[0]
        print(f"\nEjemplo — {ejemplo}:")
        print(leg[leg.proxy_group_id == ejemplo][["fichero", "clase"]].head(12).to_string(index=False))
    print("\n*** EXPLORATORY / NOT VALID FOR FINAL PATIENT-LEVEL RESULTS ***")
    print("Un identificador que agrupa imágenes de clases opuestas no identifica pacientes,")
    print("y por tanto un reparto basado en él no es un reparto a nivel de paciente.")

In [ ]:
# ==============================================================================
# APÉNDICE C — QUÉ HARÍA FALTA PARA INCORPORAR LAS FUENTES DESCARTADAS
# ==============================================================================
# Las diez colecciones de Kaggle que se barajaron al principio no se usan aquí porque
# ninguna distribuye el identificador de paciente. No están "mal": simplemente no
# permiten un reparto a nivel de paciente, que es el requisito de este trabajo.
#
# Esta celda deja escrito, fuente por fuente, qué fichero o metadato concreto
# habría que conseguir para poder incorporarlas manteniendo la trazabilidad, y
# genera `manifest_template.csv` con el esquema exacto que habría que rellenar.
# ==============================================================================
PENDIENTES = [
    {"fuente": "SARTAJ — brain-tumor-classification-mri",
     "url": "https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri",
     "clase": "tumor",
     "identificador_disponible": "NO",
     "que_haria_falta": "La colección original de Chakrabarty / la fuente hospitalaria "
                        "con el ID de estudio. En Kaggle solo llegan JPEG renombrados."},
    {"fuente": "Br35H — brain-tumor-detection",
     "url": "https://www.kaggle.com/datasets/ahmedhamada0/brain-tumor-detection",
     "clase": "notumor (principal) y tumor",
     "identificador_disponible": "NO",
     "que_haria_falta": "El autor no publica procedencia por caso. Habría que "
                        "contactar con él o localizar la colección de la que derivan "
                        "las 1.500 imágenes 'no'."},
    {"fuente": "Masoud Nickparvar — brain-tumor-mri-dataset",
     "url": "https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset",
     "clase": "ambas",
     "identificador_disponible": "NO (es recombinación de figshare + SARTAJ + Br35H)",
     "que_haria_falta": "Nada nuevo: su parte de figshare ya se recupera aquí desde el "
                        "origen con PID real. Su parte de Br35H/SARTAJ hereda el problema."},
    {"fuente": "Crystal Clean (mohammadhossein77) — brain-tumors-dataset",
     "url": "https://www.kaggle.com/datasets/mohammadhossein77/brain-tumors-dataset",
     "clase": "ambas",
     "identificador_disponible": "NO, y además viene YA AUMENTADO",
     "que_haria_falta": "Es la colección de la que salían los ficheros G_/M_/P_/N_ con "
                        "sufijos _DA_/_HF_/_VF_/_RO_/_SP_. Al estar aumentada, una misma "
                        "imagen aparece varias veces con nombres distintos: usarla sin "
                        "trazabilidad garantiza fuga."},
    {"fuente": "tombackert, ishans24, ahmedsorour1, shreyag1103, indk214",
     "url": "varias",
     "clase": "ambas",
     "identificador_disponible": "NO (todas son recombinaciones de las anteriores)",
     "que_haria_falta": "Mapa fichero->colección de origen->caso. Sin él no se puede "
                        "descartar que una imagen ya esté presente por otra vía."},
    {"fuente": "darren2020 — ct-to-mri-cgan",
     "url": "https://www.kaggle.com/datasets/darren2020/ct-to-mri-cgan",
     "clase": "—",
     "identificador_disponible": "NO",
     "que_haria_falta": "No debería usarse en ningún caso: contiene TC y salidas de una "
                        "GAN, no RM reales. Es la fuente del problema de dominio que "
                        "detectó el tutor."},
]

pendientes_df = pd.DataFrame(PENDIENTES)
pendientes_df.to_csv(ART_DIR / "fuentes_pendientes.csv", index=False)
for p in PENDIENTES:
    print(f"\n· {p['fuente']}")
    print(f"    clase            : {p['clase']}")
    print(f"    ID de paciente   : {p['identificador_disponible']}")
    print(f"    qué haría falta  : {p['que_haria_falta']}")

print("\n" + "=" * 78)
print("CÓMO INCORPORAR UNA FUENTE NUEVA MÁS ADELANTE")
print("=" * 78)
print(f"""
1. Rellenar una fila por imagen en {TEMPLATE_PATH.name} con estas columnas:
   {', '.join(COLUMNAS)}

2. `patient_id` debe venir de la FUENTE, no del nombre del fichero, y
   `patient_id_source` debe describir de dónde sale (por ejemplo
   'real:DICOM PatientID' o 'real:cjdata.PID'). Si no se puede, dejar
   `patient_id` vacío y `patient_id_source='desconocido'`: en
   STRICT_PATIENT_MODE el cuaderno se detendrá, que es el comportamiento
   correcto.

3. Concatenar ese CSV con `manifest` antes de la celda 1.4. El resto del
   pipeline (duplicados, agrupación, reparto, auditoría) no cambia.

4. Al añadir una fuente nueva, `leave-one-source-out` pasa a ser posible: si
   hay al menos dos fuentes que aporten AMBAS clases, se puede entrenar
   dejando una fuera y medir la caída, que es la prueba fuerte contra el
   sesgo de procedencia.
""")
print(f"Plantilla vacía disponible en: {TEMPLATE_PATH}")
print(f"Lista de pendientes en       : {ART_DIR/'fuentes_pendientes.csv'}")